In [ ]:
%pip install -q ultralytics roboflow huggingface_hub

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()

zip_name = next(
    (name for name in uploaded if name.endswith(".zip")),
    None
)

assert zip_name is not None, "Upload the model ZIP file"

MODEL_DIR = Path("/content/football_models/ball_detector_v1")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name) as z:
    for name in z.namelist():
        if Path(name).name in {"best.pt", "last.pt"}:
            destination = MODEL_DIR / Path(name).name
            destination.write_bytes(z.read(name))

BEST_MODEL = MODEL_DIR / "best.pt"

assert BEST_MODEL.exists(), "best.pt was not found in the ZIP"

print("Model restored:", BEST_MODEL)
print("Size:", round(BEST_MODEL.stat().st_size / 1e6, 2), "MB")

In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/football_ball_model_v1.zip")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("Files inside ZIP:")
    for name in z.namelist():
        print(name)

In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/football_ball_model_v1.zip")
MODEL_DIR = Path("/content/football_models/ball_detector_v1")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    checkpoints = [
        name for name in z.namelist()
        if name.lower().endswith(".pt")
        and not name.startswith("__MACOSX/")
    ]

    print("Checkpoints found:", checkpoints)

    assert checkpoints, "No .pt files found inside ZIP!"

    # Prefer best.pt, otherwise inspect the available checkpoints
    best_candidates = [
        name for name in checkpoints
        if Path(name).name.lower() == "best.pt"
        or Path(name).name.lower().endswith("_best.pt")
    ]

    selected = best_candidates[0] if best_candidates else None

    if selected is None:
        raise ValueError(
            f"No recognizable best checkpoint. Available: {checkpoints}"
        )

    BEST_MODEL = MODEL_DIR / "best.pt"

    with z.open(selected) as source:
        with open(BEST_MODEL, "wb") as destination:
            import shutil
            shutil.copyfileobj(source, destination)

print("Extracted checkpoint:", BEST_MODEL)
print("Size:", round(BEST_MODEL.stat().st_size / 1e6, 2), "MB")

In [ ]:
from ultralytics import YOLO

ball_model = YOLO(str(BEST_MODEL))

print("Model loaded successfully!")
print("Classes:", ball_model.names)

In [ ]:
from roboflow import Roboflow
from getpass import getpass
from pathlib import Path

rf = Roboflow(api_key=getpass("Roboflow API key: "))

dataset = (
    rf.workspace("roboflow-jvuqo")
      .project("football-ball-detection-rejhg")
      .version(4)
      .download("yolov8")
)

DATASET_PATH = Path(dataset.location).resolve()

print("Dataset:", DATASET_PATH)

In [ ]:
import yaml

DATA_YAML = DATASET_PATH / "data.yaml"

config = {
    "path": str(DATASET_PATH),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": {0: "ball"}
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

test_images = [
    p for p in (DATASET_PATH / "test/images").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("Test images:", len(test_images))
assert len(test_images) == 500

In [ ]:
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"

metrics = ball_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=1280,
    batch=4 if DEVICE != "cpu" else 1,
    device=DEVICE,
    conf=0.001,
    plots=True,
    project="/content/football_outputs/ball_evaluation",
    name="roboflow_test_v1"
)

print("\n========== TEST RESULTS ==========")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print("==================================")

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

VIDEO_PATH = Path(
    "/content/football_dataset/mot/clips/117093.mp4"
)

VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)

DRIVE_VIDEO = Path(
    "/content/drive/MyDrive/Football_CV/videos/117093.mp4"
)

if VIDEO_PATH.exists():
    print("Video already restored.")

elif DRIVE_VIDEO.exists():
    print("Restoring video from Google Drive...")
    shutil.copy2(DRIVE_VIDEO, VIDEO_PATH)

else:
    print("Drive backup not found. Downloading from Hugging Face...")

    from huggingface_hub import hf_hub_download

    downloaded = hf_hub_download(
        repo_id="atomscott/soccertrack-v2",
        repo_type="dataset",
        filename="mot/clips/117093.mp4",
        local_dir="/content/football_dataset"
    )

    VIDEO_PATH = Path(downloaded)

print("Video available:", VIDEO_PATH.exists())
print("Video path:", VIDEO_PATH)

In [ ]:
import cv2

cap = cv2.VideoCapture(str(VIDEO_PATH))

assert cap.isOpened(), "Unable to open video"

fps = cap.get(cv2.CAP_PROP_FPS)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("Resolution:", width, "x", height)
print("FPS:", fps)
print("Total frames:", frames)
print("Duration:", round(frames / fps, 2), "seconds")

cap.release()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

VIDEO_PATH = Path(
    "/content/football_dataset/mot/clips/117093.mp4"
)

assert VIDEO_PATH.exists(), (
    "SoccerTrack video missing. Restore 117093.mp4 first."
)

cap = cv2.VideoCapture(str(VIDEO_PATH))

FRAME_NUMBER = 4000
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_NUMBER)

success, frame = cap.read()
cap.release()

assert success, "Unable to read the requested frame"

H, W = frame.shape[:2]

print("Frame number:", FRAME_NUMBER)
print("Resolution:", W, "x", H)

plt.figure(figsize=(18, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
full_result = ball_model.predict(
    source=frame,
    imgsz=1536,
    conf=0.05,
    device=0,
    verbose=False
)[0]

print("Full-frame ball candidates:", len(full_result.boxes))

for box in full_result.boxes:
    print(
        "Confidence:", round(float(box.conf[0]), 3),
        "Coordinates:", box.xyxy[0].cpu().numpy().round(1)
    )

In [ ]:
import torch
from torchvision.ops import nms

TILE_WIDTH = 1280
OVERLAP = 256
CONF = 0.05

starts = list(
    range(0, max(1, W - TILE_WIDTH + 1), TILE_WIDTH - OVERLAP)
)

last_start = max(0, W - TILE_WIDTH)

if last_start not in starts:
    starts.append(last_start)

starts = sorted(set(starts))

all_boxes = []
all_scores = []

for start_x in starts:

    tile = frame[:, start_x:min(start_x + TILE_WIDTH, W)]

    result = ball_model.predict(
        source=tile,
        imgsz=1280,
        conf=CONF,
        device=0,
        verbose=False
    )[0]

    for detection in result.boxes:

        box = detection.xyxy[0].cpu().numpy().copy()
        box[[0, 2]] += start_x

        all_boxes.append(box)
        all_scores.append(float(detection.conf[0]))

if all_boxes:

    boxes = torch.tensor(
        np.array(all_boxes), dtype=torch.float32
    )

    scores = torch.tensor(
        all_scores, dtype=torch.float32
    )

    keep = nms(boxes, scores, iou_threshold=0.5)

    final_boxes = boxes[keep].numpy()
    final_scores = scores[keep].numpy()

else:
    final_boxes = np.empty((0, 4))
    final_scores = np.empty((0,))

print("Tiles:", len(starts))
print("Raw detections:", len(all_boxes))
print("Detections after NMS:", len(final_boxes))

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/content/ball_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

annotated = frame.copy()

for i, (box, score) in enumerate(
    zip(final_boxes, final_scores)
):
    x1, y1, x2, y2 = box.astype(int)

    cv2.rectangle(
        annotated, (x1, y1), (x2, y2),
        (0, 0, 255), 2
    )

    cv2.putText(
        annotated,
        f"Ball? {score:.2f}",
        (x1, max(20, y1 - 8)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6, (0, 0, 255), 2
    )

output_path = OUTPUT_DIR / "soccertrack_frame4000_tiled.jpg"
cv2.imwrite(str(output_path), annotated)

plt.figure(figsize=(18, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

# Display up to 12 of the highest-confidence candidate crops
order = np.argsort(final_scores)[::-1][:12]

if len(order) == 0:
    print("No ball candidates found.")
else:
    fig, axes = plt.subplots(
        1, len(order),
        figsize=(min(20, 3 * len(order)), 4),
        squeeze=False
    )

    for ax, idx in zip(axes.flat, order):
        x1, y1, x2, y2 = final_boxes[idx]

        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        radius = 50

        crop = frame[
            max(0, cy-radius):min(H, cy+radius),
            max(0, cx-radius):min(W, cx+radius)
        ]

        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Conf: {final_scores[idx]:.2f}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

print("Saved:", output_path)

In [ ]:

import cv2
import numpy as np
import pandas as pd
import torch
from torchvision.ops import nms

FRAME_NUMBERS = list(range(4000, 4300, 30))

TILE_WIDTH = 1280
OVERLAP = 256
CONF_THRESHOLD = 0.01

def detect_ball_tiled(frame):
    H, W = frame.shape[:2]

    starts = list(
        range(
            0,
            max(1, W - TILE_WIDTH + 1),
            TILE_WIDTH - OVERLAP
        )
    )

    last_start = max(0, W - TILE_WIDTH)

    if last_start not in starts:
        starts.append(last_start)

    starts = sorted(set(starts))

    boxes = []
    scores = []

    for start_x in starts:
        tile = frame[:, start_x:start_x + TILE_WIDTH]

        result = ball_model.predict(
            source=tile,
            imgsz=1280,
            conf=CONF_THRESHOLD,
            device=DEVICE,
            verbose=False
        )[0]

        for detection in result.boxes:
            box = detection.xyxy[0].cpu().numpy().copy()
            box[[0, 2]] += start_x

            boxes.append(box)
            scores.append(float(detection.conf[0]))

    if not boxes:
        return np.empty((0, 4)), np.empty((0,))

    boxes = torch.tensor(np.asarray(boxes), dtype=torch.float32)
    scores = torch.tensor(scores, dtype=torch.float32)

    keep = nms(boxes, scores, iou_threshold=0.5)

    return boxes[keep].numpy(), scores[keep].numpy()


cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

frame_results = []

for frame_idx in FRAME_NUMBERS:

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    success, current_frame = cap.read()

    if not success:
        continue

    boxes, scores = detect_ball_tiled(current_frame)

    frame_results.append({
        "frame": frame_idx,
        "detections": len(boxes),
        "max_confidence": (
            round(float(scores.max()), 4)
            if len(scores) else 0
        )
    })

    print(
        f"Frame {frame_idx}: "
        f"{len(boxes)} candidates"
    )

cap.release()

results_df = pd.DataFrame(frame_results)
display(results_df)


In [ ]:

import matplotlib.pyplot as plt

candidate_frames = results_df[
    results_df["detections"] > 0
].sort_values("max_confidence", ascending=False)

if candidate_frames.empty:
    print("No ball candidates found in the sampled frames.")

else:
    best_frame_idx = int(candidate_frames.iloc[0]["frame"])

    cap = cv2.VideoCapture(str(VIDEO_PATH))
    cap.set(cv2.CAP_PROP_POS_FRAMES, best_frame_idx)
    success, best_frame = cap.read()
    cap.release()

    assert success

    boxes, scores = detect_ball_tiled(best_frame)
    H, W = best_frame.shape[:2]

    order = np.argsort(scores)[::-1][:8]

    fig, axes = plt.subplots(
        1, len(order),
        figsize=(min(20, len(order) * 3), 4),
        squeeze=False
    )

    for ax, idx in zip(axes.flat, order):

        x1, y1, x2, y2 = boxes[idx]

        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        radius = 65

        crop = best_frame[
            max(0, cy-radius):min(H, cy+radius),
            max(0, cx-radius):min(W, cx+radius)
        ]

        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Confidence {scores[idx]:.3f}")
        ax.axis("off")

    plt.suptitle(f"Ball candidates — Frame {best_frame_idx}")
    plt.tight_layout()
    plt.show()


In [ ]:

def detect_ball_square_tiles(frame, tile_size=640, overlap=160):
    H, W = frame.shape[:2]

    stride = tile_size - overlap

    xs = list(range(0, max(1, W - tile_size + 1), stride))
    ys = list(range(0, max(1, H - tile_size + 1), stride))

    xs.append(max(0, W - tile_size))
    ys.append(max(0, H - tile_size))

    xs = sorted(set(xs))
    ys = sorted(set(ys))

    all_boxes = []
    all_scores = []

    for y in ys:
        for x in xs:

            tile = frame[
                y:y + tile_size,
                x:x + tile_size
            ]

            result = ball_model.predict(
                source=tile,
                imgsz=1280,
                conf=0.01,
                device=DEVICE,
                verbose=False
            )[0]

            for det in result.boxes:
                box = det.xyxy[0].cpu().numpy().copy()

                box[[0, 2]] += x
                box[[1, 3]] += y

                all_boxes.append(box)
                all_scores.append(float(det.conf[0]))

    if not all_boxes:
        return np.empty((0, 4)), np.empty((0,))

    boxes = torch.tensor(np.asarray(all_boxes), dtype=torch.float32)
    scores = torch.tensor(all_scores, dtype=torch.float32)

    keep = nms(boxes, scores, iou_threshold=0.5)

    return boxes[keep].numpy(), scores[keep].numpy()


# Test frame 4000
cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, 4000)
success, frame = cap.read()
cap.release()

assert success

boxes, scores = detect_ball_square_tiles(frame)

print("Square-tile candidates:", len(boxes))

for box, score in zip(boxes, scores):
    print(
        f"Confidence: {score:.3f}",
        f"Box: {box.round(1)}"
    )


In [ ]:

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

FRAME_RANGE = range(4100, 4141, 2)

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

ball_records = []

for frame_idx in FRAME_RANGE:

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    success, frame = cap.read()

    if not success:
        continue

    boxes, scores = detect_ball_tiled(frame)

    for box, score in zip(boxes, scores):
        x1, y1, x2, y2 = map(float, box)

        ball_records.append({
            "frame": frame_idx,
            "timestamp_sec": frame_idx / 25.0,
            "center_x": (x1 + x2) / 2,
            "center_y": (y1 + y2) / 2,
            "width": x2 - x1,
            "height": y2 - y1,
            "confidence": float(score)
        })

    print(
        f"Frame {frame_idx}: "
        f"{len(boxes)} candidates"
    )

cap.release()

ball_df = pd.DataFrame(
    ball_records,
    columns=[
        "frame", "timestamp_sec",
        "center_x", "center_y",
        "width", "height", "confidence"
    ]
)

print("\nTotal candidates:", len(ball_df))
display(ball_df.head(10))


In [ ]:

import matplotlib.pyplot as plt

if ball_df.empty:
    print("No ball candidates available.")
else:
    fig, ax = plt.subplots(figsize=(15, 6))

    scatter = ax.scatter(
        ball_df["center_x"],
        ball_df["center_y"],
        c=ball_df["frame"],
        s=30 + 250 * ball_df["confidence"],
        cmap="viridis",
        alpha=0.8
    )

    ax.set_xlim(0, 4096)
    ax.set_ylim(1080, 0)

    ax.set_xlabel("Image X coordinate")
    ax.set_ylabel("Image Y coordinate")
    ax.set_title(
        "Ball candidate positions — Frames 4100–4140"
    )

    plt.colorbar(
        scatter,
        ax=ax,
        label="Frame number"
    )

    plt.show()


In [ ]:

import matplotlib.pyplot as plt

top_candidates = (
    ball_df.sort_values(
        "confidence",
        ascending=False
    )
    .head(12)
)

cap = cv2.VideoCapture(str(VIDEO_PATH))

fig, axes = plt.subplots(3, 4, figsize=(15, 12))

for ax in axes.flat:
    ax.axis("off")

for ax, row in zip(
    axes.flat,
    top_candidates.itertuples(index=False)
):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.frame))
    success, frame = cap.read()

    if not success:
        continue

    H, W = frame.shape[:2]

    cx = int(row.center_x)
    cy = int(row.center_y)

    radius = 90

    x1 = max(0, cx-radius)
    x2 = min(W, cx+radius)
    y1 = max(0, cy-radius)
    y2 = min(H, cy+radius)

    crop = frame[y1:y2, x1:x2].copy()

    cv2.circle(
        crop,
        (cx-x1, cy-y1),
        12,
        (0, 0, 255),
        2
    )

    ax.imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    )

    ax.set_title(
        f"F{row.frame} | Conf {row.confidence:.3f}",
        fontsize=10
    )

plt.tight_layout()
plt.show()

cap.release()


In [ ]:

from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v1"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "ball_candidates_4100_4140.csv"

ball_df.to_csv(csv_path, index=False)

print("Saved:", csv_path)


In [ ]:
import numpy as np
import pandas as pd

assert not ball_df.empty, "Run ball detection first"

ANCHOR_FRAME = 4116
MAX_SPEED_PX_PER_FRAME = 45
MAX_GAP_FRAMES = 6

detections = ball_df.copy()
detections = detections.sort_values(["frame", "confidence"])

anchor_candidates = detections[
    detections.frame == ANCHOR_FRAME
]

assert not anchor_candidates.empty, "Anchor frame has no detections"

anchor = anchor_candidates.loc[
    anchor_candidates.confidence.idxmax()
].copy()

print("Anchor:")
print(anchor[["frame", "center_x", "center_y", "confidence"]])


def track_direction(anchor, direction=1):
    """Greedy temporal association from an anchor detection."""
    selected = []
    previous = anchor.copy()
    velocity = np.array([0.0, 0.0])

    available_frames = sorted(
        detections.frame.unique(),
        reverse=(direction == -1)
    )

    for current_frame in available_frames:

        if direction * (current_frame - anchor.frame) <= 0:
            continue

        dt = abs(current_frame - previous.frame)

        if dt > MAX_GAP_FRAMES:
            break

        candidates = detections[
            detections.frame == current_frame
        ].copy()

        if candidates.empty:
            continue

        previous_position = np.array([
            previous.center_x, previous.center_y
        ])

        # Predict location from recent observed motion.
        predicted = previous_position + velocity * dt * direction

        positions = candidates[
            ["center_x", "center_y"]
        ].to_numpy()

        distances = np.linalg.norm(
            positions - predicted,
            axis=1
        )

        max_distance = MAX_SPEED_PX_PER_FRAME * dt

        candidates["distance"] = distances
        candidates = candidates[
            candidates.distance <= max_distance
        ].copy()

        if candidates.empty:
            continue

        # Favor a nearby candidate with good detector confidence.
        candidates["association_cost"] = (
            candidates.distance / max_distance
            - 0.25 * candidates.confidence
        )

        best = candidates.loc[
            candidates.association_cost.idxmin()
        ].copy()

        current_position = np.array([
            best.center_x, best.center_y
        ])

        observed_velocity = (
            (current_position - previous_position) / dt
        ) * direction

        velocity = (
            0.4 * velocity +
            0.6 * observed_velocity
        )

        selected.append(best)
        previous = best

    return selected


backward = track_direction(anchor, direction=-1)
forward = track_direction(anchor, direction=1)

trajectory = pd.DataFrame(
    backward + [anchor] + forward
)

trajectory = (
    trajectory.sort_values("frame")
    .drop_duplicates("frame")
    .reset_index(drop=True)
)

trajectory["timestamp_sec"] = trajectory.frame / 25.0

print("Tracked frames:", len(trajectory))
print("First frame:", trajectory.frame.min())
print("Last frame:", trajectory.frame.max())

display(
    trajectory[
        ["frame", "center_x", "center_y", "confidence"]
    ]
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(15, 6))

# All detections in gray
ax.scatter(
    ball_df.center_x,
    ball_df.center_y,
    s=15,
    alpha=0.25,
    color="gray",
    label="All candidates"
)

# Selected trajectory
ax.plot(
    trajectory.center_x,
    trajectory.center_y,
    "-o",
    linewidth=2,
    markersize=4,
    color="red",
    label="Selected ball trajectory"
)

ax.scatter(
    [anchor.center_x],
    [anchor.center_y],
    s=110,
    marker="*",
    color="orange",
    label="Anchor frame 4116",
    zorder=5
)

ax.set_xlim(0, 4096)
ax.set_ylim(1080, 0)

ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.set_title("Ball Tracking V1 — Frames 4100–4140")

ax.legend()
ax.grid(alpha=0.2)

plt.show()

In [ ]:
trajectory["dx"] = trajectory.center_x.diff()
trajectory["dy"] = trajectory.center_y.diff()

trajectory["frame_gap"] = trajectory.frame.diff()

trajectory["displacement_px"] = np.sqrt(
    trajectory.dx**2 + trajectory.dy**2
)

trajectory["speed_px_per_frame"] = (
    trajectory.displacement_px / trajectory.frame_gap
)

print("Trajectory summary")
print("------------------")
print("Observations:", len(trajectory))
print("Mean confidence:", round(trajectory.confidence.mean(), 3))
print(
    "Mean motion (px/frame):",
    round(trajectory.speed_px_per_frame.mean(), 2)
)
print(
    "Maximum motion (px/frame):",
    round(trajectory.speed_px_per_frame.max(), 2)
)

display(
    trajectory[
        [
            "frame",
            "confidence",
            "frame_gap",
            "displacement_px",
            "speed_px_per_frame"
        ]
    ]
)

In [ ]:
import cv2
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path("/content/football_outputs/ball_tracking")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_VIDEO = OUTPUT_DIR / "ball_tracking_v1_raw.mp4"
FINAL_VIDEO = OUTPUT_DIR / "ball_tracking_v1.mp4"

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    str(RAW_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)
assert writer.isOpened()

trajectory_lookup = {
    int(row.frame): row
    for row in trajectory.itertuples(index=False)
}

# Draw every original frame; observations are available every 2 frames.
last_position = None

cap.set(cv2.CAP_PROP_POS_FRAMES, 4100)

for frame_idx in range(4100, 4141):
    success, frame = cap.read()
    if not success:
        break

    record = trajectory_lookup.get(frame_idx)

    if record is not None:
        center = (
            int(record.center_x),
            int(record.center_y)
        )
        last_position = center

        cv2.circle(frame, center, 20, (0, 0, 255), 3)
        cv2.circle(frame, center, 4, (0, 0, 255), -1)

        cv2.putText(
            frame,
            f"BALL {record.confidence:.2f}",
            (center[0] + 22, center[1] - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7, (0, 0, 255), 2
        )

    cv2.putText(
        frame,
        f"Frame {frame_idx}",
        (30, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2, (255, 255, 255), 2
    )

    writer.write(frame)

cap.release()
writer.release()

# Browser-compatible H.264
import subprocess

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(RAW_VIDEO),
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "23",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(FINAL_VIDEO)
    ],
    check=True,
    capture_output=True,
    text=True
)

print("Video saved:", FINAL_VIDEO)

In [ ]:
from IPython.display import Video, display

display(
    Video(
        str(FINAL_VIDEO),
        embed=True,
        width=1100
    )
)

In [ ]:
from pathlib import Path

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v1"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

csv_path = SAVE_DIR / "ball_trajectory_v1_4100_4140.csv"

trajectory.to_csv(csv_path, index=False)

print("Trajectory saved:", csv_path)

In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

assert "ball_df" in globals(), "Run the previous ball-detection notebook cells first."
assert "trajectory" in globals(), "Ball Tracking V1 trajectory not found."

FPS = 25.0
START_FRAME = 4100
END_FRAME = 4140
ANCHOR_FRAME = 4116

MAX_MISSING = 6
MAX_ASSOCIATION_DISTANCE = 65.0

def make_kalman(x, y, vx=0.0, vy=0.0):
    kf = cv2.KalmanFilter(4, 2)

    kf.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ], dtype=np.float32)

    kf.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0]
    ], dtype=np.float32)

    kf.processNoiseCov = np.diag(
        [1.0, 1.0, 9.0, 9.0]
    ).astype(np.float32)

    kf.measurementNoiseCov = np.eye(
        2, dtype=np.float32
    ) * 16.0

    kf.errorCovPost = np.diag(
        [16.0, 16.0, 100.0, 100.0]
    ).astype(np.float32)

    kf.statePost = np.array(
        [[x], [y], [vx], [vy]],
        dtype=np.float32
    )

    return kf

print("Kalman filter configuration ready.")


In [ ]:

observations = ball_df.copy()

observations = observations[
    observations["frame"].between(START_FRAME, END_FRAME)
].copy()

observations["frame"] = observations["frame"].astype(int)

anchor_matches = trajectory[
    trajectory["frame"] == ANCHOR_FRAME
]

assert not anchor_matches.empty, "Anchor frame not found in V1 trajectory."

anchor = anchor_matches.iloc[0]

anchor_x = float(anchor.center_x)
anchor_y = float(anchor.center_y)

# Estimate velocity using nearby V1 detections.
before = trajectory[trajectory.frame < ANCHOR_FRAME].sort_values("frame")
after = trajectory[trajectory.frame > ANCHOR_FRAME].sort_values("frame")

if not before.empty and not after.empty:
    p0 = before.iloc[-1]
    p1 = after.iloc[0]
elif not before.empty:
    p0 = before.iloc[-1]
    p1 = anchor
elif not after.empty:
    p0 = anchor
    p1 = after.iloc[0]
else:
    p0 = anchor
    p1 = anchor

dt = float(p1.frame - p0.frame)

if dt > 0:
    vx = (float(p1.center_x) - float(p0.center_x)) / dt
    vy = (float(p1.center_y) - float(p0.center_y)) / dt
else:
    vx = vy = 0.0

print("Anchor position:", (round(anchor_x, 2), round(anchor_y, 2)))
print("Initial velocity:", (round(vx, 2), round(vy, 2)), "px/frame")
print("Candidate observations:", len(observations))


In [ ]:

def kalman_track_direction(direction):
    """
    direction=1  -> forward
    direction=-1 -> backward

    Returns one state per processed frame.
    """

    kf = make_kalman(
        anchor_x, anchor_y,
        direction * vx, direction * vy
    )

    frames = (
        range(ANCHOR_FRAME + 1, END_FRAME + 1)
        if direction == 1
        else range(ANCHOR_FRAME - 1, START_FRAME - 1, -1)
    )

    records = []
    missing_count = 0

    for frame_idx in frames:

        prediction = kf.predict()

        pred_x = float(prediction[0, 0])
        pred_y = float(prediction[1, 0])

        candidates = observations[
            observations.frame == frame_idx
        ].copy()

        observed = False
        confidence = np.nan
        measured_x = np.nan
        measured_y = np.nan

        if not candidates.empty:

            positions = candidates[
                ["center_x", "center_y"]
            ].to_numpy(dtype=float)

            distances = np.linalg.norm(
                positions - np.array([pred_x, pred_y]),
                axis=1
            )

            candidates["distance"] = distances

            candidates = candidates[
                candidates.distance <= MAX_ASSOCIATION_DISTANCE
            ].copy()

            if not candidates.empty:

                # Favor motion consistency; confidence breaks close choices.
                candidates["cost"] = (
                    candidates.distance / MAX_ASSOCIATION_DISTANCE
                    - 0.20 * candidates.confidence
                )

                selected = candidates.loc[
                    candidates.cost.idxmin()
                ]

                measured_x = float(selected.center_x)
                measured_y = float(selected.center_y)
                confidence = float(selected.confidence)

                measurement = np.array(
                    [[measured_x], [measured_y]],
                    dtype=np.float32
                )

                corrected = kf.correct(measurement)

                est_x = float(corrected[0, 0])
                est_y = float(corrected[1, 0])

                observed = True
                missing_count = 0

        if not observed:
            est_x, est_y = pred_x, pred_y
            missing_count += 1

        records.append({
            "frame": frame_idx,
            "timestamp_sec": frame_idx / FPS,
            "x": est_x,
            "y": est_y,
            "predicted_x": pred_x,
            "predicted_y": pred_y,
            "measured_x": measured_x,
            "measured_y": measured_y,
            "confidence": confidence,
            "source": "observed" if observed else "predicted",
            "missing_streak": missing_count
        })

        if missing_count > MAX_MISSING:
            print(
                f"Tracking stopped in direction {direction} "
                f"after frame {frame_idx}: prolonged detection gap."
            )
            break

    return records


forward_records = kalman_track_direction(direction=1)
backward_records = kalman_track_direction(direction=-1)

anchor_record = {
    "frame": ANCHOR_FRAME,
    "timestamp_sec": ANCHOR_FRAME / FPS,
    "x": anchor_x,
    "y": anchor_y,
    "predicted_x": anchor_x,
    "predicted_y": anchor_y,
    "measured_x": anchor_x,
    "measured_y": anchor_y,
    "confidence": float(anchor.confidence),
    "source": "observed",
    "missing_streak": 0
}

kalman_df = pd.DataFrame(
    backward_records + [anchor_record] + forward_records
)

kalman_df = kalman_df.sort_values(
    "frame"
).reset_index(drop=True)

print("Total frames tracked:", len(kalman_df))
print("Observed frames:",
      (kalman_df.source == "observed").sum())
print("Predicted frames:",
      (kalman_df.source == "predicted").sum())

display(kalman_df.head(10))


In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))

# All detector candidates
ax.scatter(
    ball_df.center_x,
    ball_df.center_y,
    s=20,
    color="gray",
    alpha=0.25,
    label="All YOLO candidates"
)

# Original V1
ax.plot(
    trajectory.center_x,
    trajectory.center_y,
    "--",
    color="royalblue",
    linewidth=1.5,
    label="V1 trajectory"
)

# Kalman V2
ax.plot(
    kalman_df.x,
    kalman_df.y,
    "-",
    color="red",
    linewidth=2.5,
    label="Kalman V2 trajectory"
)

observed = kalman_df[kalman_df.source == "observed"]
predicted = kalman_df[kalman_df.source == "predicted"]

ax.scatter(
    observed.x, observed.y,
    color="green", s=35,
    label="Observed / corrected"
)

ax.scatter(
    predicted.x, predicted.y,
    color="orange", s=35,
    marker="x",
    label="Predicted only"
)

ax.set_xlim(0, 4096)
ax.set_ylim(1080, 0)
ax.set_xlabel("Image X (pixels)")
ax.set_ylabel("Image Y (pixels)")
ax.set_title("Football Ball Tracking — V1 vs Kalman V2")
ax.legend()
ax.grid(alpha=0.2)

plt.show()


In [ ]:

kalman_df["dx"] = kalman_df.x.diff()
kalman_df["dy"] = kalman_df.y.diff()

kalman_df["displacement_px"] = np.sqrt(
    kalman_df.dx**2 + kalman_df.dy**2
)

kalman_df["speed_px_per_frame"] = kalman_df.displacement_px

summary = {
    "Frames covered": len(kalman_df),
    "Observed detections": int((kalman_df.source == "observed").sum()),
    "Predicted positions": int((kalman_df.source == "predicted").sum()),
    "Observed percentage": round(
        100 * (kalman_df.source == "observed").mean(), 2
    ),
    "Mean displacement (px/frame)": round(
        kalman_df.speed_px_per_frame.mean(), 2
    ),
    "Maximum displacement (px/frame)": round(
        kalman_df.speed_px_per_frame.max(), 2
    )
}

display(pd.DataFrame(
    summary.items(),
    columns=["Metric", "Value"]
))

print("\nLargest frame-to-frame movements:")

display(
    kalman_df.sort_values(
        "speed_px_per_frame",
        ascending=False
    )[
        ["frame", "source", "confidence",
         "speed_px_per_frame", "missing_streak"]
    ].head(10)
)


In [ ]:

import cv2
import subprocess
from pathlib import Path

VIDEO_OUT_DIR = Path(
    "/content/football_outputs/ball_tracking_v2"
)
VIDEO_OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = VIDEO_OUT_DIR / "ball_tracking_v2_raw.mp4"
FINAL_PATH = VIDEO_OUT_DIR / "ball_tracking_v2_H264.mp4"

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

video_fps = cap.get(cv2.CAP_PROP_FPS)
video_fps = video_fps if video_fps > 0 else FPS

W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    str(RAW_PATH),
    cv2.VideoWriter_fourcc(*"mp4v"),
    video_fps,
    (W, H)
)

assert writer.isOpened()

position_by_frame = {
    int(row.frame): row
    for row in kalman_df.itertuples(index=False)
}

history = []

cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

for frame_idx in range(START_FRAME, END_FRAME + 1):

    success, frame = cap.read()
    if not success:
        break

    record = position_by_frame.get(frame_idx)

    if record is not None:

        center = (
            int(round(record.x)),
            int(round(record.y))
        )

        history.append(center)
        history = history[-15:]

        for p1, p2 in zip(history[:-1], history[1:]):
            cv2.line(
                frame, p1, p2,
                (255, 255, 255), 2
            )

        color = (
            (0, 255, 0) if record.source == "observed"
            else (0, 165, 255)
        )

        cv2.circle(frame, center, 17, color, 3)
        cv2.circle(frame, center, 4, color, -1)

        label = (
            "BALL OBSERVED" if record.source == "observed"
            else "BALL PREDICTED"
        )

        cv2.putText(
            frame, label,
            (center[0] + 20, max(25, center[1] - 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6, color, 2
        )

    cv2.putText(
        frame,
        f"Ball Tracking V2 | Frame {frame_idx}",
        (30, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 255, 255),
        2
    )

    writer.write(frame)

cap.release()
writer.release()

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(RAW_PATH),
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "23",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(FINAL_PATH)
    ],
    check=True,
    capture_output=True,
    text=True
)

print("V2 video created:", FINAL_PATH)
print("File size:", round(FINAL_PATH.stat().st_size / 1e6, 2), "MB")


In [ ]:

from IPython.display import Video, display

display(Video(
    str(FINAL_PATH),
    embed=True,
    width=1100
))


In [ ]:

import cv2
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v2"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_CSV = SAVE_DIR / "segment_C_ball_candidates.csv"
SCAN_CSV = SAVE_DIR / "segment_C_detection_summary.csv"

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Cannot open SoccerTrack video"

# Use the existing tiled detector with CONF_THRESHOLD = 0.01
CONF_THRESHOLD = 0.01

records = []
scan_records = []

for frame_idx in range(4000, 4300, 2):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        print("Could not read frame:", frame_idx)
        continue

    boxes, scores = detect_ball_tiled(frame)

    scan_records.append({
        "frame": frame_idx,
        "detections": len(scores),
        "max_confidence": float(scores.max()) if len(scores) else 0.0
    })

    for box, score in zip(boxes, scores):
        x1, y1, x2, y2 = map(float, box)

        records.append({
            "frame": frame_idx,
            "timestamp_sec": frame_idx / 25.0,
            "center_x": (x1 + x2) / 2,
            "center_y": (y1 + y2) / 2,
            "width": x2 - x1,
            "height": y2 - y1,
            "confidence": float(score)
        })

    # Incremental backup in case the runtime disconnects
    if (frame_idx - 4000) % 40 == 0:
        pd.DataFrame(records).to_csv(
            CANDIDATE_CSV, index=False
        )
        pd.DataFrame(scan_records).to_csv(
            SCAN_CSV, index=False
        )
        print(
            f"Frame {frame_idx} | "
            f"{len(scores)} candidates | "
            f"Total {len(records)}"
        )

cap.release()

ball_candidates_C = pd.DataFrame(
    records,
    columns=[
        "frame", "timestamp_sec",
        "center_x", "center_y",
        "width", "height", "confidence"
    ]
)

scan_df = pd.DataFrame(scan_records)

ball_candidates_C.to_csv(CANDIDATE_CSV, index=False)
scan_df.to_csv(SCAN_CSV, index=False)

print("\nSegment C detection complete.")
print("Frames scanned:", len(scan_df))
print("Frames with candidates:", (scan_df.detections > 0).sum())
print("Total candidates:", len(ball_candidates_C))
print("Mean candidates per scanned frame:", round(scan_df.detections.mean(), 2))
print("Saved:", CANDIDATE_CSV)


In [ ]:

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(
    scan_df.frame,
    scan_df.detections,
    marker=".",
    linewidth=1
)

ax.set_title("Segment C — Ball detections per sampled frame")
ax.set_xlabel("Video frame")
ax.set_ylabel("Number of candidates")
ax.grid(alpha=0.25)

plt.show()

print("Detection coverage:",
      round(100 * (scan_df.detections > 0).mean(), 2),
      "% of scanned frames")

print("Highest-confidence frames:")
display(
    scan_df.sort_values(
        "max_confidence",
        ascending=False
    ).head(15)
)


In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v2"
)

CANDIDATE_CSV = SAVE_DIR / "segment_C_ball_candidates.csv"

candidates = pd.read_csv(CANDIDATE_CSV)

candidates["frame"] = candidates["frame"].astype(int)

print("Total candidates:", len(candidates))
print("Frames containing candidates:", candidates.frame.nunique())

display(candidates.head())


In [ ]:

candidate_groups = {
    int(frame): group
    for frame, group in candidates.groupby("frame")
}

def run_v3_direction(direction=1):
    kf = create_kalman(
        anchor_x, anchor_y,
        vx, vy,
        direction=direction
    )

    if direction == 1:
        frames = range(ANCHOR_FRAME + 1, END_FRAME + 1)
    else:
        frames = range(ANCHOR_FRAME - 1, START_FRAME - 1, -1)

    records = []
    missing = 0
    active = True

    for frame_idx in frames:

        if not active:
            records.append({
                "frame": frame_idx,
                "x": np.nan,
                "y": np.nan,
                "confidence": np.nan,
                "status": "lost",
                "missing_streak": missing
            })
            continue

        prediction = kf.predict()

        px = float(prediction[0, 0])
        py = float(prediction[1, 0])

        frame_candidates = candidate_groups.get(frame_idx)

        selected = None

        if frame_candidates is not None:
            pool = frame_candidates.copy()

            # Remove extremely weak or implausibly sized boxes.
            pool = pool[
                (pool.confidence >= MIN_CONFIDENCE) &
                (pool.width >= 2) &
                (pool.height >= 2) &
                (pool.width <= 30) &
                (pool.height <= 30)
            ].copy()

            if not pool.empty:
                positions = pool[
                    ["center_x", "center_y"]
                ].to_numpy(dtype=float)

                distances = np.linalg.norm(
                    positions - [px, py],
                    axis=1
                )

                gate = BASE_GATE_PX + min(70, 5 * missing)

                pool["distance"] = distances
                pool = pool[pool.distance <= gate].copy()

                if not pool.empty:
                    pool["cost"] = (
                        pool.distance / gate
                        - 0.15 * pool.confidence
                    )

                    selected = pool.loc[pool.cost.idxmin()]

        if selected is not None:
            measurement = np.array([
                [float(selected.center_x)],
                [float(selected.center_y)]
            ], dtype=np.float32)

            corrected = kf.correct(measurement)

            x = float(corrected[0, 0])
            y = float(corrected[1, 0])

            confidence = float(selected.confidence)
            status = "observed"
            missing = 0

        else:
            missing += 1

            if missing <= MAX_MISSING_FRAMES:
                x, y = px, py
                confidence = np.nan
                status = "predicted"
            else:
                x = y = np.nan
                confidence = np.nan
                status = "lost"
                active = False

        records.append({
            "frame": frame_idx,
            "x": x,
            "y": y,
            "confidence": confidence,
            "status": status,
            "missing_streak": missing
        })

    return records


backward = run_v3_direction(-1)
forward = run_v3_direction(1)

anchor_record = {
    "frame": ANCHOR_FRAME,
    "x": anchor_x,
    "y": anchor_y,
    "confidence": float(anchor.confidence),
    "status": "observed",
    "missing_streak": 0
}

ball_v3 = pd.DataFrame(
    backward + [anchor_record] + forward
)

ball_v3 = ball_v3.sort_values("frame").reset_index(drop=True)

ball_v3["timestamp_sec"] = ball_v3.frame / 25.0

assert len(ball_v3) == 300
assert ball_v3.frame.is_unique

print("Ball Tracking V3 complete!")
print(ball_v3.status.value_counts())

display(ball_v3.head())


In [ ]:

fig, ax = plt.subplots(figsize=(16, 7))

observed = ball_v3[ball_v3.status == "observed"]
predicted = ball_v3[ball_v3.status == "predicted"]
lost = ball_v3[ball_v3.status == "lost"]

ax.scatter(
    candidates.center_x,
    candidates.center_y,
    color="gray",
    alpha=0.15,
    s=12,
    label="All YOLO candidates"
)

ax.plot(
    ball_v3.x,
    ball_v3.y,
    color="red",
    linewidth=2,
    label="V3 estimated trajectory"
)

ax.scatter(
    observed.x, observed.y,
    color="green",
    s=20,
    label="Observed"
)

ax.scatter(
    predicted.x, predicted.y,
    color="orange",
    marker="x",
    s=25,
    label="Predicted"
)

ax.scatter(
    [anchor_x], [anchor_y],
    marker="*",
    s=180,
    color="blue",
    label="Anchor"
)

ax.set_xlim(1800, 3600)
ax.set_ylim(750, 80)

ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.set_title("Ball Tracking V3 — Segment C")
ax.grid(alpha=0.2)
ax.legend()

plt.show()


In [ ]:

total = len(ball_v3)

observed_count = (ball_v3.status == "observed").sum()
predicted_count = (ball_v3.status == "predicted").sum()
lost_count = (ball_v3.status == "lost").sum()

summary = pd.DataFrame({
    "Metric": [
        "Total video frames",
        "Observed frames",
        "Predicted frames",
        "Lost frames",
        "Estimated coverage (%)",
        "Observed coverage (%)"
    ],
    "Value": [
        total,
        observed_count,
        predicted_count,
        lost_count,
        round(100 * (observed_count + predicted_count) / total, 2),
        round(100 * observed_count / total, 2)
    ]
})

display(summary)

print("\nFirst observed frame:", observed.frame.min())
print("Last observed frame:", observed.frame.max())

print("\nLongest missing streak:", ball_v3.missing_streak.max())

print("\nFrame status near the start:")
display(ball_v3.head(20))

print("\nFrame status near the end:")
display(ball_v3.tail(20))


In [ ]:

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v3"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "segment_C_ball_trajectory_v3.csv"

ball_v3.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)


In [ ]:

import pandas as pd

diagnostic = ball_v3.copy()

diagnostic["previous_status"] = (
    diagnostic["status"].shift(1)
)

changes = diagnostic[
    diagnostic["status"] != diagnostic["previous_status"]
][
    ["frame", "previous_status", "status",
     "x", "y", "confidence", "missing_streak"]
]

print("Tracking status transitions:")
display(changes)

print("\nLongest continuous observed sequences:")

diagnostic["group"] = (
    diagnostic["status"] != diagnostic["status"].shift()
).cumsum()

runs = (
    diagnostic.groupby("group")
    .agg(
        start_frame=("frame", "min"),
        end_frame=("frame", "max"),
        status=("status", "first"),
        length=("frame", "size")
    )
    .reset_index(drop=True)
)

display(runs.sort_values("length", ascending=False).head(15))


In [ ]:

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v2"
)

FULL_CSV = SAVE_DIR / "segment_C_ball_candidates_allframes.csv"
SCAN_FULL_CSV = SAVE_DIR / "segment_C_detection_summary_allframes.csv"

# Recover previous partial progress if this cell was interrupted.
if FULL_CSV.exists():
    existing = pd.read_csv(FULL_CSV)
else:
    existing = ball_candidates_C.copy()

if SCAN_FULL_CSV.exists():
    scan_existing = pd.read_csv(SCAN_FULL_CSV)
else:
    scan_existing = scan_df.copy()

processed_frames = set(scan_existing.frame.astype(int))

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

new_records = []
new_scan = []

# Ensure the existing detector uses the diagnostic threshold.
CONF_THRESHOLD = 0.01

for frame_idx in range(4000, 4300):

    if frame_idx in processed_frames:
        continue

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, image = cap.read()

    if not ok:
        continue

    boxes, scores = detect_ball_tiled(image)

    new_scan.append({
        "frame": frame_idx,
        "detections": len(scores),
        "max_confidence": float(scores.max()) if len(scores) else 0.0
    })

    for box, score in zip(boxes, scores):
        x1, y1, x2, y2 = map(float, box)

        new_records.append({
            "frame": frame_idx,
            "timestamp_sec": frame_idx / 25.0,
            "center_x": (x1 + x2) / 2,
            "center_y": (y1 + y2) / 2,
            "width": x2 - x1,
            "height": y2 - y1,
            "confidence": float(score)
        })

    # Back up progress every 20 frames.
    if len(new_scan) % 20 == 0:
        pd.concat(
            [existing, pd.DataFrame(new_records)],
            ignore_index=True
        ).to_csv(FULL_CSV, index=False)

        pd.concat(
            [scan_existing, pd.DataFrame(new_scan)],
            ignore_index=True
        ).to_csv(SCAN_FULL_CSV, index=False)

        print("Processed additional frames:", len(new_scan))

cap.release()

all_candidates = pd.concat(
    [existing, pd.DataFrame(new_records)],
    ignore_index=True
).drop_duplicates()

full_scan = pd.concat(
    [scan_existing, pd.DataFrame(new_scan)],
    ignore_index=True
).drop_duplicates(subset=["frame"], keep="last")

all_candidates = all_candidates.sort_values("frame")
full_scan = full_scan.sort_values("frame")

all_candidates.to_csv(FULL_CSV, index=False)
full_scan.to_csv(SCAN_FULL_CSV, index=False)

print("Frames scanned:", len(full_scan))
print("Frames with candidates:",
      (full_scan.detections > 0).sum())
print("Total detections:", len(all_candidates))
print("Saved:", FULL_CSV)


In [ ]:

import pandas as pd

comparison = pd.DataFrame({
    "Experiment": [
        "V3 — every 2 frames",
        "V4 input — every frame"
    ],
    "Frames scanned": [
        len(scan_df),
        len(full_scan)
    ],
    "Frames with candidates": [
        int((scan_df.detections > 0).sum()),
        int((full_scan.detections > 0).sum())
    ]
})

comparison["Candidate coverage (%)"] = (
    100 * comparison["Frames with candidates"] /
    comparison["Frames scanned"]
).round(2)

display(comparison)


In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v2"
)

FULL_CSV = DATA_DIR / "segment_C_ball_candidates_allframes.csv"

assert FULL_CSV.exists(), "All-frame candidate CSV not found"

candidates_v4 = pd.read_csv(FULL_CSV)

candidates_v4 = (
    candidates_v4
    .dropna(subset=["frame", "center_x", "center_y", "confidence"])
    .drop_duplicates()
    .sort_values("frame")
    .reset_index(drop=True)
)

candidates_v4["frame"] = candidates_v4["frame"].astype(int)

print("Candidates:", len(candidates_v4))
print("Frames containing candidates:", candidates_v4.frame.nunique())

display(candidates_v4.head())

In [ ]:
START_FRAME = 4000
END_FRAME = 4299
ANCHOR_FRAME = 4116
FPS = 25.0

MIN_CONFIDENCE = 0.03
MAX_MISSING = 12
BASE_GATE = 35.0
MAX_GATE = 95.0

# Select the anchor that was verified in our earlier experiment
anchor_candidates = candidates_v4[
    candidates_v4.frame == ANCHOR_FRAME
]

assert not anchor_candidates.empty

anchor = anchor_candidates.loc[
    anchor_candidates.confidence.idxmax()
]

anchor_x = float(anchor.center_x)
anchor_y = float(anchor.center_y)

vx, vy = 0.0, 0.0

if "trajectory" in globals():
    before = trajectory[
        trajectory.frame < ANCHOR_FRAME
    ].sort_values("frame")

    after = trajectory[
        trajectory.frame > ANCHOR_FRAME
    ].sort_values("frame")

    if not before.empty and not after.empty:
        p0 = before.iloc[-1]
        p1 = after.iloc[0]

        delta = p1.frame - p0.frame

        vx = float((p1.center_x - p0.center_x) / delta)
        vy = float((p1.center_y - p0.center_y) / delta)

print("Anchor:", ANCHOR_FRAME)
print("Position:", anchor_x, anchor_y)
print("Estimated forward velocity:", vx, vy)

In [ ]:
def create_ball_kalman(x, y, vx, vy, direction):
    kf = cv2.KalmanFilter(4, 2)

    dt = float(direction)

    kf.transitionMatrix = np.array([
        [1, 0, dt, 0],
        [0, 1, 0, dt],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ], dtype=np.float32)

    kf.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0]
    ], dtype=np.float32)

    kf.processNoiseCov = np.diag(
        [1, 1, 12, 12]
    ).astype(np.float32)

    kf.measurementNoiseCov = (
        np.eye(2, dtype=np.float32) * 16
    )

    kf.errorCovPost = np.diag(
        [16, 16, 100, 100]
    ).astype(np.float32)

    kf.statePost = np.array(
        [[x], [y], [vx], [vy]],
        dtype=np.float32
    )

    return kf

In [ ]:
candidate_groups = {
    int(frame): group
    for frame, group in candidates_v4.groupby("frame")
}

def track_v4_direction(direction):

    kf = create_ball_kalman(
        anchor_x, anchor_y, vx, vy, direction
    )

    frames = (
        range(ANCHOR_FRAME + 1, END_FRAME + 1)
        if direction == 1
        else range(ANCHOR_FRAME - 1, START_FRAME - 1, -1)
    )

    records = []
    missing = 0
    active = True

    for frame_idx in frames:

        pool = candidate_groups.get(frame_idx)
        raw_count = len(pool) if pool is not None else 0

        if not active:
            records.append({
                "frame": frame_idx,
                "x": np.nan,
                "y": np.nan,
                "confidence": np.nan,
                "status": "lost",
                "missing_streak": missing,
                "raw_candidates": raw_count,
                "eligible_candidates": 0,
                "nearest_distance": np.nan,
                "reason": "track_terminated"
            })
            continue

        prediction = kf.predict()

        px = float(prediction[0, 0])
        py = float(prediction[1, 0])

        selected = None
        eligible_count = 0
        nearest_distance = np.nan
        reason = "no_candidates"

        if raw_count:
            valid = pool[
                (pool.confidence >= MIN_CONFIDENCE) &
                (pool.width.between(2, 30)) &
                (pool.height.between(2, 30))
            ].copy()

            eligible_count = len(valid)

            if valid.empty:
                reason = "filtered_candidates"
            else:
                distances = np.hypot(
                    valid.center_x.to_numpy() - px,
                    valid.center_y.to_numpy() - py
                )

                valid["distance"] = distances
                nearest_distance = float(distances.min())

                gate = min(
                    MAX_GATE,
                    BASE_GATE + 5 * missing
                )

                valid = valid[
                    valid.distance <= gate
                ].copy()

                if valid.empty:
                    reason = "outside_gate"
                else:
                    valid["cost"] = (
                        valid.distance / gate
                        - 0.15 * valid.confidence
                    )

                    selected = valid.loc[
                        valid.cost.idxmin()
                    ]

        if selected is not None:
            measurement = np.array([
                [float(selected.center_x)],
                [float(selected.center_y)]
            ], dtype=np.float32)

            corrected = kf.correct(measurement)

            x = float(corrected[0, 0])
            y = float(corrected[1, 0])

            confidence = float(selected.confidence)
            status = "observed"
            missing = 0
            reason = "matched"

        else:
            missing += 1
            confidence = np.nan

            if missing <= MAX_MISSING:
                x, y = px, py
                status = "predicted"
            else:
                x, y = np.nan, np.nan
                status = "lost"
                active = False

        records.append({
            "frame": frame_idx,
            "x": x,
            "y": y,
            "confidence": confidence,
            "status": status,
            "missing_streak": missing,
            "raw_candidates": raw_count,
            "eligible_candidates": eligible_count,
            "nearest_distance": nearest_distance,
            "reason": reason
        })

    return records


forward_v4 = track_v4_direction(1)
backward_v4 = track_v4_direction(-1)

anchor_record = {
    "frame": ANCHOR_FRAME,
    "x": anchor_x,
    "y": anchor_y,
    "confidence": float(anchor.confidence),
    "status": "observed",
    "missing_streak": 0,
    "raw_candidates": len(anchor_candidates),
    "eligible_candidates": len(anchor_candidates),
    "nearest_distance": 0.0,
    "reason": "anchor"
}

ball_v4 = pd.DataFrame(
    backward_v4 + [anchor_record] + forward_v4
).sort_values("frame").reset_index(drop=True)

ball_v4["timestamp_sec"] = ball_v4.frame / FPS

assert len(ball_v4) == 300
assert ball_v4.frame.is_unique

print("V4 results:")
print(ball_v4.status.value_counts())

In [ ]:
def tracker_summary(df, name):
    counts = df.status.value_counts()

    observed = int(counts.get("observed", 0))
    predicted = int(counts.get("predicted", 0))
    lost = int(counts.get("lost", 0))

    return {
        "Tracker": name,
        "Observed": observed,
        "Predicted": predicted,
        "Lost": lost,
        "Observed coverage (%)": round(100 * observed / len(df), 2),
        "Estimated coverage (%)": round(
            100 * (observed + predicted) / len(df), 2
        )
    }

comparison = pd.DataFrame([
    tracker_summary(ball_v3, "V3"),
    tracker_summary(ball_v4, "V4")
])

display(comparison)

print("\nV4 observation range:")
observed_v4 = ball_v4[ball_v4.status == "observed"]

if len(observed_v4):
    print(
        observed_v4.frame.min(),
        "to",
        observed_v4.frame.max()
    )

print("\nV4 reasons:")
display(ball_v4.reason.value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

ax.scatter(
    candidates_v4.center_x,
    candidates_v4.center_y,
    color="lightgray",
    s=8,
    alpha=0.25,
    label="YOLO candidates"
)

ax.plot(
    ball_v3.x, ball_v3.y,
    "--", color="royalblue",
    linewidth=2, label="V3"
)

ax.plot(
    ball_v4.x, ball_v4.y,
    "-", color="red",
    linewidth=2, label="V4"
)

obs = ball_v4[ball_v4.status == "observed"]

ax.scatter(
    obs.x, obs.y,
    color="green",
    s=20,
    label="V4 observed"
)

ax.scatter(
    [anchor_x], [anchor_y],
    color="orange",
    marker="*",
    s=160,
    label="Verified anchor"
)

ax.set_xlim(1800, 3600)
ax.set_ylim(750, 80)
ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.set_title("Ball Tracking — V3 vs V4")
ax.grid(alpha=0.2)
ax.legend()

plt.show()

In [ ]:
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v4"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ball_v4.to_csv(
    OUTPUT_DIR / "segment_C_ball_trajectory_v4.csv",
    index=False
)

comparison.to_csv(
    OUTPUT_DIR / "v3_v4_comparison.csv",
    index=False
)

print("V4 experiment saved!")

In [ ]:
import numpy as np
import pandas as pd

comparison_df = ball_v3[
    ["frame", "x", "y", "status"]
].merge(
    ball_v4[
        ["frame", "x", "y", "status", "confidence", "reason"]
    ],
    on="frame",
    suffixes=("_v3", "_v4")
)

comparison_df["difference_px"] = np.hypot(
    comparison_df.x_v4 - comparison_df.x_v3,
    comparison_df.y_v4 - comparison_df.y_v3
)

divergent = comparison_df[
    comparison_df.difference_px > 30
].copy()

print("Frames with >30px disagreement:", len(divergent))

display(
    divergent[
        [
            "frame",
            "x_v3", "y_v3",
            "x_v4", "y_v4",
            "status_v4",
            "confidence",
            "difference_px"
        ]
    ].head(20)
)

print("\nLargest disagreements:")
display(
    comparison_df.sort_values(
        "difference_px", ascending=False
    ).head(10)
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Focus on observed V4 positions descending toward the near side
suspect = ball_v4[
    (ball_v4.status == "observed") &
    (ball_v4.y > 460)
].sort_values("frame")

if suspect.empty:
    print("No observed V4 detections in the lower branch.")
else:
    indices = np.linspace(
        0, len(suspect) - 1,
        min(8, len(suspect)),
        dtype=int
    )

    samples = suspect.iloc[np.unique(indices)]

    cap = cv2.VideoCapture(str(VIDEO_PATH))
    assert cap.isOpened()

    fig, axes = plt.subplots(
        2, 4, figsize=(16, 9)
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, row in zip(
        axes.flat,
        samples.itertuples(index=False)
    ):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.frame))
        ok, frame = cap.read()

        if not ok:
            continue

        H, W = frame.shape[:2]
        cx, cy = int(row.x), int(row.y)
        radius = 85

        x1 = max(0, cx - radius)
        x2 = min(W, cx + radius)
        y1 = max(0, cy - radius)
        y2 = min(H, cy + radius)

        crop = frame[y1:y2, x1:x2].copy()

        cv2.circle(
            crop,
            (cx - x1, cy - y1),
            12,
            (0, 0, 255),
            2
        )

        ax.imshow(
            cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        )
        ax.set_title(
            f"Frame {row.frame} | Conf {row.confidence:.3f}",
            fontsize=10
        )
        ax.axis("off")

    cap.release()

    plt.suptitle("V4 lower-branch detections")
    plt.tight_layout()
    plt.show()

In [ ]:
print("V3 statuses:")
print(ball_v3.status.value_counts())

print("\nV4 statuses:")
print(ball_v4.status.value_counts())

print("\nV4 association reasons:")
print(ball_v4.reason.value_counts())

print("\nV4 observed frame range:")
observed = ball_v4[ball_v4.status == "observed"]

if not observed.empty:
    print(observed.frame.min(), "to", observed.frame.max())

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

FRAMES_TO_INSPECT = [
    4180, 4184, 4188, 4192,
    4196, 4200, 4204, 4208
]

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Cannot open SoccerTrack video"

fig, axes = plt.subplots(2, 4, figsize=(18, 10))

for ax, frame_idx in zip(axes.flat, FRAMES_TO_INSPECT):
    ax.axis("off")

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        ax.set_title(f"Frame {frame_idx}: Read failed")
        continue

    H, W = frame.shape[:2]

    frame_candidates = candidates_v4[
        candidates_v4.frame == frame_idx
    ].copy()

    matches = ball_v4[ball_v4.frame == frame_idx]

    if matches.empty:
        continue

    track_row = matches.iloc[0]

    if np.isfinite(track_row.x) and np.isfinite(track_row.y):
        cx = int(track_row.x)
        cy = int(track_row.y)

    elif not frame_candidates.empty:
        candidate = frame_candidates.loc[
            frame_candidates.confidence.idxmax()
        ]

        cx = int(candidate.center_x)
        cy = int(candidate.center_y)

    else:
        ax.set_title(f"Frame {frame_idx}: No position")
        continue

    radius = 140

    x1 = max(0, cx - radius)
    x2 = min(W, cx + radius)
    y1 = max(0, cy - radius)
    y2 = min(H, cy + radius)

    crop = frame[y1:y2, x1:x2].copy()

    # Green circle: Kalman tracker position
    if np.isfinite(track_row.x) and np.isfinite(track_row.y):
        tx = int(track_row.x)
        ty = int(track_row.y)

        cv2.circle(
            crop,
            (tx - x1, ty - y1),
            12,
            (0, 255, 0),
            2
        )

    # Red circles: YOLO ball candidates
    for candidate in frame_candidates.itertuples(index=False):
        px = int(candidate.center_x)
        py = int(candidate.center_y)

        if x1 <= px < x2 and y1 <= py < y2:
            cv2.circle(
                crop,
                (px - x1, py - y1),
                9,
                (0, 0, 255),
                2
            )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

    ax.set_title(
        f"Frame {frame_idx} | {track_row.status} | "
        f"{len(frame_candidates)} candidates",
        fontsize=10
    )

cap.release()

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

SAVE_V4 = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v4"
)

SAVE_V4.mkdir(parents=True, exist_ok=True)

ball_v4.to_csv(
    SAVE_V4 / "segment_C_ball_trajectory_v4.csv",
    index=False
)

candidates_v4.to_csv(
    SAVE_V4 / "segment_C_all_ball_candidates.csv",
    index=False
)

print("V4 trajectory and candidate detections backed up.")

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

TARGET_FRAME = 4208

frame_candidates = (
    candidates_v4[candidates_v4.frame == TARGET_FRAME]
    .sort_values("confidence", ascending=False)
    .reset_index(drop=True)
)

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, TARGET_FRAME)
ok, frame = cap.read()
cap.release()

assert ok, "Could not read frame"

print("Candidates:", len(frame_candidates))

fig, axes = plt.subplots(
    1, max(1, len(frame_candidates)),
    figsize=(4 * max(1, len(frame_candidates)), 4),
    squeeze=False
)

for ax in axes.flat:
    ax.axis("off")

for i, row in frame_candidates.iterrows():
    cx = int(row.center_x)
    cy = int(row.center_y)
    radius = 95

    h, w = frame.shape[:2]
    x1, x2 = max(0, cx-radius), min(w, cx+radius)
    y1, y2 = max(0, cy-radius), min(h, cy+radius)

    crop = frame[y1:y2, x1:x2].copy()

    cv2.circle(
        crop, (cx-x1, cy-y1),
        12, (0, 0, 255), 2
    )

    axes[0, i].imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    )
    axes[0, i].set_title(
        f"Candidate {i} | conf {row.confidence:.3f}"
    )

plt.tight_layout()
plt.show()

display(frame_candidates[
    ["center_x", "center_y", "confidence", "width", "height"]
])


In [ ]:
print("Ball candidates at frame 4208:")

display(
    frame_candidates[
        ["center_x", "center_y", "confidence", "width", "height"]
    ]
)

In [ ]:
CANDIDATE_INDEX = 0

assert 0 <= CANDIDATE_INDEX < len(frame_candidates)

verified = frame_candidates.iloc[CANDIDATE_INDEX]

new_anchor = {
    "frame": 4208,
    "center_x": float(verified.center_x),
    "center_y": float(verified.center_y),
    "confidence": float(verified.confidence),
    "verification": "visual"
}

print("New verified ball anchor:")
print(new_anchor)

In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

V5_ANCHOR_FRAME = 4208
V5_START = 4180
V5_END = 4250

V5_MIN_CONF = 0.01
V5_BASE_GATE = 35.0
V5_MAX_GATE = 80.0
V5_MAX_MISSING = 8

# Use the candidate already verified visually
v5_anchor = {
    "frame": 4208,
    "x": 3063.608764648437,
    "y": 588.7508544921875,
    "confidence": 0.0245396103709936
}

# Use the Kalman function from our previous V4 notebook
assert "create_ball_kalman" in globals()
assert "candidates_v4" in globals()

print("V5 anchor ready:", v5_anchor)


In [ ]:

candidate_groups_v5 = {
    int(f): group
    for f, group in candidates_v4.groupby("frame")
}

def run_v5_direction(direction):
    kf = create_ball_kalman(
        v5_anchor["x"],
        v5_anchor["y"],
        0.0,
        0.0,
        direction
    )

    frames = (
        range(V5_ANCHOR_FRAME + 1, V5_END + 1)
        if direction == 1
        else range(V5_ANCHOR_FRAME - 1, V5_START - 1, -1)
    )

    records = []
    missing = 0
    active = True

    for frame_idx in frames:
        if not active:
            records.append({
                "frame": frame_idx,
                "x": np.nan,
                "y": np.nan,
                "confidence": np.nan,
                "status": "lost"
            })
            continue

        prediction = kf.predict()
        px, py = float(prediction[0, 0]), float(prediction[1, 0])

        pool = candidate_groups_v5.get(frame_idx)
        selected = None

        if pool is not None:
            valid = pool[
                (pool.confidence >= V5_MIN_CONF) &
                (pool.width.between(2, 30)) &
                (pool.height.between(2, 30))
            ].copy()

            if not valid.empty:
                valid["distance"] = np.hypot(
                    valid.center_x - px,
                    valid.center_y - py
                )

                gate = min(
                    V5_MAX_GATE,
                    V5_BASE_GATE + 5 * missing
                )

                valid = valid[valid.distance <= gate].copy()

                if not valid.empty:
                    valid["cost"] = (
                        valid.distance / gate
                        - 0.10 * valid.confidence
                    )

                    selected = valid.loc[valid.cost.idxmin()]

        if selected is not None:
            measurement = np.array(
                [
                    [float(selected.center_x)],
                    [float(selected.center_y)]
                ],
                dtype=np.float32
            )

            corrected = kf.correct(measurement)

            x = float(corrected[0, 0])
            y = float(corrected[1, 0])
            confidence = float(selected.confidence)
            status = "observed"
            missing = 0

        else:
            missing += 1

            if missing <= V5_MAX_MISSING:
                x, y = px, py
                confidence = np.nan
                status = "predicted"
            else:
                x = y = np.nan
                confidence = np.nan
                status = "lost"
                active = False

        records.append({
            "frame": frame_idx,
            "x": x,
            "y": y,
            "confidence": confidence,
            "status": status
        })

    return records


backward_v5 = run_v5_direction(-1)
forward_v5 = run_v5_direction(1)

anchor_record = {
    "frame": V5_ANCHOR_FRAME,
    "x": v5_anchor["x"],
    "y": v5_anchor["y"],
    "confidence": v5_anchor["confidence"],
    "status": "observed"
}

ball_v5 = pd.DataFrame(
    backward_v5 + [anchor_record] + forward_v5
)

ball_v5 = ball_v5.sort_values("frame").reset_index(drop=True)
ball_v5["timestamp_sec"] = ball_v5.frame / 25.0

print("V5 status counts:")
print(ball_v5.status.value_counts())

observed_v5 = ball_v5[ball_v5.status == "observed"]

print("First observed frame:", observed_v5.frame.min())
print("Last observed frame:", observed_v5.frame.max())


In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))

window = candidates_v4[
    candidates_v4.frame.between(V5_START, V5_END)
]

ax.scatter(
    window.center_x,
    window.center_y,
    color="lightgray",
    s=15,
    alpha=0.4,
    label="YOLO candidates"
)

old = ball_v4[
    ball_v4.frame.between(V5_START, V5_END)
]

ax.plot(
    old.x, old.y,
    "--",
    color="royalblue",
    linewidth=2,
    label="V4 trajectory"
)

ax.plot(
    ball_v5.x,
    ball_v5.y,
    "-",
    color="red",
    linewidth=2,
    label="V5 trajectory"
)

observed = ball_v5[ball_v5.status == "observed"]
predicted = ball_v5[ball_v5.status == "predicted"]

ax.scatter(
    observed.x, observed.y,
    color="green",
    s=28,
    label="V5 observed"
)

ax.scatter(
    predicted.x, predicted.y,
    color="orange",
    marker="x",
    s=30,
    label="V5 predicted"
)

ax.scatter(
    [v5_anchor["x"]],
    [v5_anchor["y"]],
    marker="*",
    color="purple",
    s=180,
    label="V5 anchor"
)

ax.set_xlim(2400, 3700)
ax.set_ylim(850, 300)
ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.set_title("Ball Tracking V4 vs V5 - Reacquisition experiment")
ax.legend()
ax.grid(alpha=0.2)

plt.show()


In [ ]:

overlap = ball_v4[
    ["frame", "x", "y", "status"]
].merge(
    ball_v5[
        ["frame", "x", "y", "status"]
    ],
    on="frame",
    suffixes=("_v4", "_v5")
)

overlap["distance_px"] = np.hypot(
    overlap.x_v4 - overlap.x_v5,
    overlap.y_v4 - overlap.y_v5
)

print("Overlap around the transition:")

display(
    overlap[
        overlap.frame.between(4180, 4208)
    ][
        [
            "frame",
            "status_v4",
            "status_v5",
            "distance_px"
        ]
    ]
)

print("\nFrames where both trackers have observations:")

both_observed = overlap[
    (overlap.status_v4 == "observed") &
    (overlap.status_v5 == "observed")
]

display(both_observed)


In [ ]:

from pathlib import Path

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v5"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

ball_v5.to_csv(
    SAVE_DIR / "segment_C_ball_local_track_4208.csv",
    index=False
)

overlap.to_csv(
    SAVE_DIR / "v4_v5_overlap_diagnostics.csv",
    index=False
)

print("V5 local tracking results saved.")


In [ ]:

import numpy as np
import pandas as pd

ANCHOR_X = 3063.608764648437
ANCHOR_Y = 588.7508544921875

FRAME_START = 4196
FRAME_END = 4220

rows = []

for frame_idx in range(FRAME_START, FRAME_END + 1):
    pool = candidates_v4[
        candidates_v4.frame == frame_idx
    ].copy()

    if pool.empty:
        rows.append({
            "frame": frame_idx,
            "candidates": 0,
            "nearest_distance_px": np.nan,
            "nearest_confidence": np.nan,
            "nearest_x": np.nan,
            "nearest_y": np.nan
        })
        continue

    pool["distance"] = np.hypot(
        pool.center_x - ANCHOR_X,
        pool.center_y - ANCHOR_Y
    )

    nearest = pool.loc[pool.distance.idxmin()]

    rows.append({
        "frame": frame_idx,
        "candidates": len(pool),
        "nearest_distance_px": round(float(nearest.distance), 2),
        "nearest_confidence": round(float(nearest.confidence), 4),
        "nearest_x": round(float(nearest.center_x), 1),
        "nearest_y": round(float(nearest.center_y), 1)
    })

diagnostics = pd.DataFrame(rows)

display(diagnostics)

print("Candidates within 35 px of anchor:",
      int((diagnostics.nearest_distance_px <= 35).sum()))

print("Candidates within 80 px of anchor:",
      int((diagnostics.nearest_distance_px <= 80).sum()))


In [ ]:

import cv2
import matplotlib.pyplot as plt
import numpy as np

FRAMES = [
    4202, 4204, 4206, 4208,
    4210, 4212, 4214, 4216
]

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(2, 4, figsize=(18, 10))

RADIUS = 160

for ax, frame_idx in zip(axes.flat, FRAMES):
    ax.axis("off")

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        continue

    H, W = frame.shape[:2]

    x1 = max(0, int(ANCHOR_X) - RADIUS)
    x2 = min(W, int(ANCHOR_X) + RADIUS)
    y1 = max(0, int(ANCHOR_Y) - RADIUS)
    y2 = min(H, int(ANCHOR_Y) + RADIUS)

    crop = frame[y1:y2, x1:x2].copy()

    # Draw all YOLO candidates in the fixed region
    pool = candidates_v4[
        candidates_v4.frame == frame_idx
    ]

    for det in pool.itertuples(index=False):
        cx = int(det.center_x)
        cy = int(det.center_y)

        if x1 <= cx < x2 and y1 <= cy < y2:
            cv2.circle(
                crop,
                (cx - x1, cy - y1),
                10,
                (0, 0, 255),
                2
            )

    # Small blue cross marks the fixed anchor location
    ax_x = int(ANCHOR_X) - x1
    ax_y = int(ANCHOR_Y) - y1

    cv2.drawMarker(
        crop,
        (ax_x, ax_y),
        (255, 0, 0),
        cv2.MARKER_CROSS,
        12,
        1
    )

    ax.imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    )

    ax.set_title(f"Frame {frame_idx}", fontsize=11)

cap.release()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Previously inspected approximate ball locations
known_positions = {
    4208: (3063.61, 588.75),
    4210: (2996.0, 582.7),
    4211: (2961.9, 580.5)
}

def find_nearest_candidate(frame_idx, expected, max_distance=18):
    pool = candidates_v4[
        candidates_v4.frame == frame_idx
    ].copy()

    assert not pool.empty, f"No candidates at frame {frame_idx}"

    pool["distance"] = np.hypot(
        pool.center_x - expected[0],
        pool.center_y - expected[1]
    )

    selected = pool.loc[pool.distance.idxmin()]

    assert selected.distance <= max_distance, (
        f"Candidate at frame {frame_idx} is too far "
        f"from the inspected position"
    )

    return selected


anchors = {
    frame: find_nearest_candidate(frame, pos)
    for frame, pos in known_positions.items()
}

p0 = anchors[4210]
p1 = anchors[4211]

vx = float(p1.center_x - p0.center_x)
vy = float(p1.center_y - p0.center_y)

print("Initial velocity:")
print("vx:", round(vx, 2), "px/frame")
print("vy:", round(vy, 2), "px/frame")
print("Initial anchor:", (float(p1.center_x), float(p1.center_y)))

In [ ]:
assert "create_ball_kalman" in globals()

START = 4211
END = 4250

GATE_PX = 50.0
MAX_MISSING = 6
MIN_CONF = 0.01

kf = create_ball_kalman(
    float(p1.center_x),
    float(p1.center_y),
    vx,
    vy,
    direction=1
)

records = []
missing = 0
active = True

def add_record(frame_idx, x, y, confidence, status):
    records.append({
        "frame": frame_idx,
        "timestamp_sec": frame_idx / 25.0,
        "x": x,
        "y": y,
        "confidence": confidence,
        "status": status
    })


# Preserve known observations and explicitly interpolated frame 4209
add_record(
    4208,
    float(anchors[4208].center_x),
    float(anchors[4208].center_y),
    float(anchors[4208].confidence),
    "observed"
)

add_record(
    4209,
    float((anchors[4208].center_x + p0.center_x) / 2),
    float((anchors[4208].center_y + p0.center_y) / 2),
    np.nan,
    "interpolated"
)

for f, point in [(4210, p0), (4211, p1)]:
    add_record(
        f,
        float(point.center_x),
        float(point.center_y),
        float(point.confidence),
        "observed"
    )

for frame_idx in range(START + 1, END + 1):

    if not active:
        add_record(frame_idx, np.nan, np.nan, np.nan, "lost")
        continue

    prediction = kf.predict()
    px, py = float(prediction[0, 0]), float(prediction[1, 0])

    pool = candidates_v4[
        candidates_v4.frame == frame_idx
    ].copy()

    pool = pool[
        (pool.confidence >= MIN_CONF) &
        (pool.width.between(2, 30)) &
        (pool.height.between(2, 30))
    ].copy()

    selected = None

    if not pool.empty:
        pool["distance"] = np.hypot(
            pool.center_x - px,
            pool.center_y - py
        )

        # Grow gate modestly during missed detections
        gate = min(80.0, GATE_PX + 5 * missing)

        pool = pool[pool.distance <= gate]

        if not pool.empty:
            pool["cost"] = (
                pool.distance / gate
                - 0.12 * pool.confidence
            )
            selected = pool.loc[pool.cost.idxmin()]

    if selected is not None:
        measurement = np.array([
            [float(selected.center_x)],
            [float(selected.center_y)]
        ], dtype=np.float32)

        corrected = kf.correct(measurement)

        add_record(
            frame_idx,
            float(corrected[0, 0]),
            float(corrected[1, 0]),
            float(selected.confidence),
            "observed"
        )

        missing = 0

    else:
        missing += 1

        if missing <= MAX_MISSING:
            add_record(
                frame_idx, px, py, np.nan, "predicted"
            )
        else:
            add_record(
                frame_idx, np.nan, np.nan, np.nan, "lost"
            )
            active = False

ball_v51 = pd.DataFrame(records)

print("V5.1 tracking results:")
print(ball_v51.status.value_counts())

print("\nLast observed frame:")
obs = ball_v51[ball_v51.status == "observed"]
print(obs.frame.max())

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))

window = candidates_v4[
    candidates_v4.frame.between(4208, 4250)
]

ax.scatter(
    window.center_x,
    window.center_y,
    color="lightgray",
    alpha=0.3,
    s=15,
    label="YOLO candidates"
)

ax.plot(
    ball_v5.x, ball_v5.y,
    "--", color="royalblue",
    linewidth=2,
    label="V5: zero initial velocity"
)

ax.plot(
    ball_v51.x, ball_v51.y,
    "-o", color="red",
    linewidth=2,
    markersize=3,
    label="V5.1: estimated initial velocity"
)

observed = ball_v51[ball_v51.status == "observed"]

ax.scatter(
    observed.x, observed.y,
    color="green",
    s=30,
    label="V5.1 observations"
)

ax.set_xlim(1900, 3400)
ax.set_ylim(750, 350)
ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.set_title("Football Tracking: V5 vs V5.1")
ax.legend()
ax.grid(alpha=0.2)

plt.show()

In [ ]:
SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v5"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

output_path = SAVE_DIR / "ball_trajectory_v51_4208_4250.csv"

ball_v51.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:

import cv2
import numpy as np
import subprocess
from pathlib import Path
from IPython.display import Video, display

OUT_DIR = Path("/content/football_outputs/ball_tracking_v51")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_VIDEO = OUT_DIR / "v51_zoom_raw.mp4"
FINAL_VIDEO = OUT_DIR / "v51_zoom_H264.mp4"

FRAME_START = 4208
FRAME_END = 4250

# Fixed crop covering the V5.1 trajectory
X1, Y1 = 1750, 350
X2, Y2 = 3350, 850

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fps = cap.get(cv2.CAP_PROP_FPS)
fps = fps if fps > 0 else 25.0

crop_width = X2 - X1
crop_height = Y2 - Y1

writer = cv2.VideoWriter(
    str(RAW_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (crop_width, crop_height)
)
assert writer.isOpened()

track_lookup = {
    int(row.frame): row
    for row in ball_v51.itertuples(index=False)
}

candidate_groups = {
    int(f): group
    for f, group in candidates_v4.groupby("frame")
}

colors = {
    "observed": (0, 255, 0),
    "predicted": (0, 165, 255),
    "interpolated": (255, 120, 0)
}

history = []

cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_START)

for frame_idx in range(FRAME_START, FRAME_END + 1):
    ok, frame = cap.read()

    if not ok:
        break

    crop = frame[Y1:Y2, X1:X2].copy()

    # Mark all raw YOLO ball candidates
    pool = candidate_groups.get(frame_idx)

    if pool is not None:
        for candidate in pool.itertuples(index=False):
            px = int(round(candidate.center_x)) - X1
            py = int(round(candidate.center_y)) - Y1

            if 0 <= px < crop_width and 0 <= py < crop_height:
                cv2.circle(
                    crop, (px, py),
                    9, (0, 0, 255), 1
                )

    record = track_lookup.get(frame_idx)

    if record is not None and np.isfinite(record.x) and np.isfinite(record.y):
        px = int(round(record.x)) - X1
        py = int(round(record.y)) - Y1

        color = colors.get(record.status, (128, 128, 128))

        if 0 <= px < crop_width and 0 <= py < crop_height:
            history.append((px, py))
            history = history[-12:]

            for a, b in zip(history[:-1], history[1:]):
                cv2.line(crop, a, b, (255, 255, 255), 2)

            cv2.circle(crop, (px, py), 18, color, 3)
            cv2.circle(crop, (px, py), 3, color, -1)

            cv2.putText(
                crop,
                record.status.upper(),
                (min(px + 22, crop_width - 160), max(30, py - 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65, color, 2
            )

    cv2.putText(
        crop,
        f"Ball Tracking V5.1 | Frame {frame_idx}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9, (255, 255, 255), 2
    )

    writer.write(crop)

cap.release()
writer.release()

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(RAW_VIDEO),
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "23",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(FINAL_VIDEO)
    ],
    check=True,
    capture_output=True,
    text=True
)

print("Video created:", FINAL_VIDEO)
print("Size (MB):", round(FINAL_VIDEO.stat().st_size / 1e6, 2))

display(Video(str(FINAL_VIDEO), embed=True, width=1200))


In [ ]:

import shutil
from pathlib import Path

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v5"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

destination = SAVE_DIR / "ball_tracking_v51_zoom.mp4"

shutil.copy2(FINAL_VIDEO, destination)

print("Saved video:", destination)


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

frames_to_check = [4234, 4244, 4246, 4250]

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, frame_idx in zip(axes, frames_to_check):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    ax.axis("off")
    if not ok:
        continue

    row = ball_v51[ball_v51.frame == frame_idx]
    if row.empty:
        continue

    record = row.iloc[0]
    if not np.isfinite(record.x) or not np.isfinite(record.y):
        continue

    cx, cy = int(record.x), int(record.y)
    H, W = frame.shape[:2]
    r = 75

    x1, x2 = max(0, cx-r), min(W, cx+r)
    y1, y2 = max(0, cy-r), min(H, cy+r)

    crop = frame[y1:y2, x1:x2].copy()

    cv2.circle(
        crop, (cx-x1, cy-y1),
        12, (0, 0, 255), 2
    )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(f"F{frame_idx} | {record.status}")

cap.release()
plt.tight_layout()
plt.show()


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

TARGET_FRAME = 4250

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

cap.set(cv2.CAP_PROP_POS_FRAMES, TARGET_FRAME)
ok, frame = cap.read()
cap.release()

assert ok, "Could not read frame 4250"

# Use the V5.1 estimated position as a search center.
record = ball_v51.loc[
    ball_v51.frame == TARGET_FRAME
].iloc[0]

cx = int(record.x)
cy = int(record.y)

H, W = frame.shape[:2]
RADIUS = 180

x1 = max(0, cx - RADIUS)
y1 = max(0, cy - RADIUS)
x2 = min(W, cx + RADIUS)
y2 = min(H, cy + RADIUS)

roi = frame[y1:y2, x1:x2].copy()

result = ball_model.predict(
    source=roi,
    imgsz=1280,
    conf=0.005,
    device=DEVICE,
    verbose=False
)[0]

local_candidates = []

for box in result.boxes:
    coords = box.xyxy[0].cpu().numpy()

    local_candidates.append({
        "center_x": float((coords[0] + coords[2]) / 2 + x1),
        "center_y": float((coords[1] + coords[3]) / 2 + y1),
        "confidence": float(box.conf[0])
    })

print("Local candidate count:", len(local_candidates))

visual = roi.copy()

for i, candidate in enumerate(local_candidates):
    px = int(candidate["center_x"] - x1)
    py = int(candidate["center_y"] - y1)

    cv2.circle(visual, (px, py), 12, (0, 0, 255), 2)
    cv2.putText(
        visual, str(i),
        (px + 15, py - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7, (0, 0, 255), 2
    )

# Blue cross: previous Kalman prediction
cv2.drawMarker(
    visual,
    (cx - x1, cy - y1),
    (255, 0, 0),
    cv2.MARKER_CROSS,
    20, 2
)

plt.figure(figsize=(9, 9))
plt.imshow(cv2.cvtColor(visual, cv2.COLOR_BGR2RGB))
plt.title("Frame 4250: Local YOLO candidates and V5.1 prediction")
plt.axis("off")
plt.show()

import pandas as pd
display(pd.DataFrame(local_candidates))


In [ ]:
if local_candidates:
    candidate_df = pd.DataFrame(local_candidates)

    candidate_df["distance_from_prediction"] = np.hypot(
        candidate_df.center_x - record.x,
        candidate_df.center_y - record.y
    )

    display(
        candidate_df.sort_values("distance_from_prediction")
    )
else:
    print("No local YOLO detections. Manual verification is needed.")

In [ ]:

import pandas as pd
from pathlib import Path

assert len(local_candidates) >= 1

verified_candidate = local_candidates[0]

recovery_anchor = {
    "frame": 4250,
    "x": float(verified_candidate["center_x"]),
    "y": float(verified_candidate["center_y"]),
    "confidence": float(verified_candidate["confidence"]),
    "source": "local_yolo_visual_verified"
}

print("Verified recovery anchor:")
print(recovery_anchor)

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v5"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame([recovery_anchor]).to_csv(
    SAVE_DIR / "verified_anchor_frame4250.csv",
    index=False
)

print("Anchor saved.")


In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FRAMES = list(range(4246, 4257))

# Fixed search region around the visually verified ball
AX = int(recovery_anchor["x"])
AY = int(recovery_anchor["y"])

RADIUS = 160

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

local_records = []

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for ax in axes.flat:
    ax.axis("off")

for ax, frame_idx in zip(axes.flat, FRAMES):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, image = cap.read()

    if not ok:
        continue

    H, W = image.shape[:2]

    x1 = max(0, AX - RADIUS)
    x2 = min(W, AX + RADIUS)
    y1 = max(0, AY - RADIUS)
    y2 = min(H, AY + RADIUS)

    crop = image[y1:y2, x1:x2].copy()

    result = ball_model.predict(
        source=crop,
        imgsz=1280,
        conf=0.005,
        device=DEVICE,
        verbose=False
    )[0]

    for det in result.boxes:

        coords = det.xyxy[0].cpu().numpy()

        lx = float((coords[0] + coords[2]) / 2)
        ly = float((coords[1] + coords[3]) / 2)

        confidence = float(det.conf[0])

        local_records.append({
            "frame": frame_idx,
            "center_x": lx + x1,
            "center_y": ly + y1,
            "confidence": confidence,
            "width": float(coords[2] - coords[0]),
            "height": float(coords[3] - coords[1])
        })

        cv2.circle(
            crop,
            (int(lx), int(ly)),
            10,
            (0, 0, 255),
            2
        )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"Frame {frame_idx} | {len(result.boxes)} candidates"
    )

cap.release()

plt.tight_layout()
plt.show()

local_df = pd.DataFrame(
    local_records,
    columns=[
        "frame", "center_x", "center_y",
        "confidence", "width", "height"
    ]
)

print("Total local candidates:", len(local_df))
print("Frames with candidates:", local_df.frame.nunique())

display(local_df)


In [ ]:

local_df.to_csv(
    SAVE_DIR / "local_ball_candidates_4246_4256.csv",
    index=False
)

print("Local detection results saved.")


In [ ]:
import numpy as np
import pandas as pd

assert not local_df.empty

# Approximate positions from the inspected ball sequence.
reference_points = {
    4248: (2007.35, 577.73),
    4250: (1995.50, 570.09),
    4251: (1996.61, 568.98),
    4252: (1997.21, 566.83),
    4253: (1998.54, 565.09),
    4254: (1998.63, 563.30),
    4255: (1998.91, 560.73),
    4256: (1999.10, 559.21)
}

selected_records = []

for frame_idx, (expected_x, expected_y) in reference_points.items():

    pool = local_df[local_df.frame == frame_idx].copy()

    if pool.empty:
        continue

    pool["distance"] = np.hypot(
        pool.center_x - expected_x,
        pool.center_y - expected_y
    )

    # Avoid the small, near-static false-positive sequence.
    pool = pool[
        (pool.width >= 9) &
        (pool.height >= 10) &
        (pool.distance <= 15)
    ].copy()

    if pool.empty:
        continue

    pool = pool.sort_values(
        ["distance", "confidence"],
        ascending=[True, False]
    )

    best = pool.iloc[0]

    selected_records.append({
        "frame": int(frame_idx),
        "timestamp_sec": frame_idx / 25.0,
        "x": float(best.center_x),
        "y": float(best.center_y),
        "confidence": float(best.confidence),
        "status": "observed"
    })

ball_v6_observed = pd.DataFrame(selected_records)

display(ball_v6_observed)

In [ ]:
full_frames = pd.DataFrame({
    "frame": np.arange(4248, 4257)
})

ball_v6 = full_frames.merge(
    ball_v6_observed,
    on="frame",
    how="left"
)

ball_v6["timestamp_sec"] = ball_v6.frame / 25.0

ball_v6["x"] = ball_v6["x"].interpolate(
    method="linear",
    limit=1,
    limit_area="inside"
)

ball_v6["y"] = ball_v6["y"].interpolate(
    method="linear",
    limit=1,
    limit_area="inside"
)

filled = (
    ball_v6.status.isna() &
    ball_v6.x.notna() &
    ball_v6.y.notna()
)

ball_v6.loc[filled, "status"] = "interpolated"
ball_v6["status"] = ball_v6["status"].fillna("lost")

print("V6 status counts:")
print(ball_v6.status.value_counts())

display(ball_v6)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 6))

# Raw local YOLO candidates
ax.scatter(
    local_df.center_x,
    local_df.center_y,
    color="lightgray",
    s=45,
    label="All local YOLO candidates"
)

# Selected ball path
ax.plot(
    ball_v6.x,
    ball_v6.y,
    "-o",
    color="red",
    linewidth=2,
    label="V6 selected trajectory"
)

observed = ball_v6[ball_v6.status == "observed"]
interpolated = ball_v6[ball_v6.status == "interpolated"]

ax.scatter(
    observed.x,
    observed.y,
    color="green",
    s=75,
    label="Selected detections"
)

ax.scatter(
    interpolated.x,
    interpolated.y,
    marker="x",
    color="orange",
    s=90,
    label="Interpolated"
)

for row in ball_v6.itertuples(index=False):
    if np.isfinite(row.x) and np.isfinite(row.y):
        ax.annotate(
            str(row.frame),
            (row.x, row.y),
            xytext=(7, 5),
            textcoords="offset points",
            fontsize=9
        )

ax.set_xlim(1920, 2050)
ax.set_ylim(620, 530)
ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.set_title("Ball Tracking V6 — Frames 4248–4256")
ax.legend()
ax.grid(alpha=0.25)

plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(3, 3, figsize=(13, 12))

for ax, record in zip(
    axes.flat,
    ball_v6.itertuples(index=False)
):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(record.frame))
    ok, image = cap.read()

    ax.axis("off")

    if not ok or not np.isfinite(record.x):
        continue

    H, W = image.shape[:2]
    cx = int(round(record.x))
    cy = int(round(record.y))

    radius = 70

    x1 = max(0, cx - radius)
    x2 = min(W, cx + radius)
    y1 = max(0, cy - radius)
    y2 = min(H, cy + radius)

    crop = image[y1:y2, x1:x2].copy()

    color = (
        (0, 255, 0)
        if record.status == "observed"
        else (0, 165, 255)
    )

    cv2.circle(
        crop,
        (cx-x1, cy-y1),
        13,
        color,
        2
    )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"Frame {record.frame} | {record.status}",
        fontsize=10
    )

cap.release()
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v6"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

ball_v6.to_csv(
    SAVE_DIR / "ball_trajectory_v6_4248_4256.csv",
    index=False
)

local_df.to_csv(
    SAVE_DIR / "local_candidates_4246_4256.csv",
    index=False
)

print("V6 results saved.")

In [ ]:

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# Previously inspected V6 detections
seed = ball_v6_observed.set_index("frame")

p0 = seed.loc[4248]
p1 = seed.loc[4250]

vx = (float(p1.x) - float(p0.x)) / 2
vy = (float(p1.y) - float(p0.y)) / 2

START_FRAME = 4251
END_FRAME = 4280

SEARCH_RADIUS = 110
MAX_GATE = 55
MAX_MISSING = 5
MIN_CONF = 0.005

# Uses our existing Kalman helper
kf = create_ball_kalman(
    float(p1.x),
    float(p1.y),
    vx, vy,
    direction=1
)

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

records = []
missing = 0
active = True

for frame_idx in range(START_FRAME, END_FRAME + 1):

    if not active:
        records.append({
            "frame": frame_idx,
            "x": np.nan,
            "y": np.nan,
            "confidence": np.nan,
            "status": "lost",
            "candidate_count": 0
        })
        continue

    prediction = kf.predict()
    px, py = float(prediction[0, 0]), float(prediction[1, 0])

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        break

    H, W = frame.shape[:2]

    x1 = max(0, int(px - SEARCH_RADIUS))
    x2 = min(W, int(px + SEARCH_RADIUS))
    y1 = max(0, int(py - SEARCH_RADIUS))
    y2 = min(H, int(py + SEARCH_RADIUS))

    crop = frame[y1:y2, x1:x2]

    result = ball_model.predict(
        source=crop,
        imgsz=1280,
        conf=MIN_CONF,
        device=DEVICE,
        verbose=False
    )[0]

    candidates = []

    for det in result.boxes:
        box = det.xyxy[0].cpu().numpy()

        bw = float(box[2] - box[0])
        bh = float(box[3] - box[1])

        if not (9 <= bw <= 35 and 10 <= bh <= 35):
            continue

        cx = float((box[0] + box[2]) / 2 + x1)
        cy = float((box[1] + box[3]) / 2 + y1)
        conf = float(det.conf[0])

        distance = np.hypot(cx - px, cy - py)

        if distance <= MAX_GATE:
            candidates.append({
                "x": cx,
                "y": cy,
                "confidence": conf,
                "distance": distance,
                "cost": distance / MAX_GATE - 0.15 * conf
            })

    if candidates:
        best = min(candidates, key=lambda c: c["cost"])

        measurement = np.array(
            [[best["x"]], [best["y"]]],
            dtype=np.float32
        )

        corrected = kf.correct(measurement)

        x = float(corrected[0, 0])
        y = float(corrected[1, 0])
        confidence = best["confidence"]

        status = "observed"
        missing = 0

    else:
        missing += 1
        x, y = px, py
        confidence = np.nan

        if missing <= MAX_MISSING:
            status = "predicted"
        else:
            x = y = np.nan
            status = "lost"
            active = False

    records.append({
        "frame": frame_idx,
        "x": x,
        "y": y,
        "confidence": confidence,
        "status": status,
        "candidate_count": len(candidates)
    })

cap.release()

ball_v7 = pd.DataFrame(records)
ball_v7["timestamp_sec"] = ball_v7.frame / 25.0

print("V7 status counts:")
print(ball_v7.status.value_counts())

observed_v7 = ball_v7[ball_v7.status == "observed"]

print(
    "Last observed frame:",
    observed_v7.frame.max() if len(observed_v7) else "None"
)


In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import subprocess
import shutil

from pathlib import Path
from IPython.display import Video, display

MODEL_PATH = Path(
    "/content/football_models/ball_detector_v1/best.pt"
)

VIDEO_PATH = Path(
    "/content/football_dataset/mot/clips/117093.mp4"
)

assert MODEL_PATH.exists(), "Missing best.pt"
assert VIDEO_PATH.exists(), "Missing SoccerTrack video"

from ultralytics import YOLO

ball_model = YOLO(str(MODEL_PATH))

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Model:", MODEL_PATH)
print("Classes:", ball_model.names)
print("Video:", VIDEO_PATH)
print("Device:", DEVICE)

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Cannot open video"

FPS = cap.get(cv2.CAP_PROP_FPS)
FPS = FPS if FPS > 0 else 25.0

WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.release()

print("Resolution:", WIDTH, "x", HEIGHT)
print("FPS:", FPS)


In [ ]:

def create_ball_kalman(x, y, vx=0.0, vy=0.0):
    kf = cv2.KalmanFilter(4, 2)

    kf.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ], dtype=np.float32)

    kf.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0]
    ], dtype=np.float32)

    kf.processNoiseCov = np.diag(
        [1.0, 1.0, 12.0, 12.0]
    ).astype(np.float32)

    kf.measurementNoiseCov = (
        np.eye(2, dtype=np.float32) * 16.0
    )

    kf.errorCovPost = np.diag(
        [16.0, 16.0, 100.0, 100.0]
    ).astype(np.float32)

    kf.statePost = np.array(
        [[x], [y], [vx], [vy]],
        dtype=np.float32
    )

    return kf

print("Kalman filter ready.")


In [ ]:

START_FRAME = 4251
END_FRAME = 4280

SEARCH_RADIUS = 110
MAX_GATE = 55.0
MAX_MISSING = 5
MIN_CONFIDENCE = 0.005

SEED_POSITIONS = {
    4248: (2007.349869, 577.725037),
    4250: (1995.503845, 570.093155)
}

x0, y0 = SEED_POSITIONS[4248]
x1, y1 = SEED_POSITIONS[4250]

dt = 4250 - 4248

vx = (x1 - x0) / dt
vy = (y1 - y0) / dt

kf = create_ball_kalman(
    x=x1,
    y=y1,
    vx=vx,
    vy=vy
)

print("Starting frame:", START_FRAME)
print("Initial position:", round(x1, 2), round(y1, 2))
print("Initial velocity:", round(vx, 2), round(vy, 2))


In [ ]:

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Video could not be opened"

cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

records = []
missing = 0
active = True

for frame_idx in range(START_FRAME, END_FRAME + 1):

    if not active:
        records.append({
            "frame": frame_idx,
            "timestamp_sec": frame_idx / FPS,
            "x": np.nan,
            "y": np.nan,
            "measured_x": np.nan,
            "measured_y": np.nan,
            "confidence": np.nan,
            "status": "lost",
            "candidate_count": 0,
            "missing_streak": missing
        })
        continue

    ok, frame = cap.read()

    if not ok:
        raise RuntimeError(f"Could not read frame {frame_idx}")

    prediction = kf.predict()

    px = float(prediction[0, 0])
    py = float(prediction[1, 0])

    H, W = frame.shape[:2]

    crop_x1 = max(0, int(px - SEARCH_RADIUS))
    crop_x2 = min(W, int(px + SEARCH_RADIUS))

    crop_y1 = max(0, int(py - SEARCH_RADIUS))
    crop_y2 = min(H, int(py + SEARCH_RADIUS))

    crop = frame[
        crop_y1:crop_y2,
        crop_x1:crop_x2
    ]

    selected = None
    valid_candidates = []

    if crop.size > 0:

        result = ball_model.predict(
            source=crop,
            imgsz=1280,
            conf=MIN_CONFIDENCE,
            device=DEVICE,
            verbose=False
        )[0]

        for det in result.boxes:

            box = det.xyxy[0].cpu().numpy()

            bw = float(box[2] - box[0])
            bh = float(box[3] - box[1])

            # Ball-size filter in original frame coordinates
            if not (9 <= bw <= 35 and 10 <= bh <= 35):
                continue

            cx = float(
                (box[0] + box[2]) / 2 + crop_x1
            )

            cy = float(
                (box[1] + box[3]) / 2 + crop_y1
            )

            confidence = float(det.conf[0])

            distance = float(np.hypot(
                cx - px,
                cy - py
            ))

            gate = min(
                80.0,
                MAX_GATE + 5.0 * missing
            )

            if distance > gate:
                continue

            valid_candidates.append({
                "x": cx,
                "y": cy,
                "confidence": confidence,
                "distance": distance,
                "cost": distance / gate - 0.15 * confidence
            })

    if valid_candidates:
        selected = min(
            valid_candidates,
            key=lambda c: c["cost"]
        )

    if selected is not None:

        measurement = np.array([
            [selected["x"]],
            [selected["y"]]
        ], dtype=np.float32)

        corrected = kf.correct(measurement)

        x = float(corrected[0, 0])
        y = float(corrected[1, 0])

        measured_x = selected["x"]
        measured_y = selected["y"]

        confidence = selected["confidence"]
        status = "observed"
        missing = 0

    else:

        missing += 1

        measured_x = np.nan
        measured_y = np.nan
        confidence = np.nan

        if missing <= MAX_MISSING:
            x = px
            y = py
            status = "predicted"

        else:
            x = np.nan
            y = np.nan
            status = "lost"
            active = False

    records.append({
        "frame": frame_idx,
        "timestamp_sec": frame_idx / FPS,
        "x": x,
        "y": y,
        "measured_x": measured_x,
        "measured_y": measured_y,
        "confidence": confidence,
        "status": status,
        "candidate_count": len(valid_candidates),
        "missing_streak": missing
    })

    print(
        f"Frame {frame_idx}: "
        f"{status.upper()} | "
        f"confidence={confidence if np.isfinite(confidence) else 0:.3f} | "
        f"candidates={len(valid_candidates)}"
    )

cap.release()

ball_v7 = pd.DataFrame(records)

assert len(ball_v7) == END_FRAME - START_FRAME + 1

print("\nV7 TRACKING SUMMARY")
print(ball_v7.status.value_counts())

observed_v7 = ball_v7[
    ball_v7.status == "observed"
]

print(
    "Last observed frame:",
    int(observed_v7.frame.max())
    if not observed_v7.empty else "None"
)


In [ ]:

fig, ax = plt.subplots(figsize=(14, 6))

observed = ball_v7[
    ball_v7.status == "observed"
]

predicted = ball_v7[
    ball_v7.status == "predicted"
]

ax.plot(
    ball_v7.x,
    ball_v7.y,
    "-",
    color="red",
    linewidth=2,
    label="V7 estimated trajectory"
)

ax.scatter(
    observed.x,
    observed.y,
    color="green",
    s=40,
    label="Observed"
)

ax.scatter(
    predicted.x,
    predicted.y,
    color="orange",
    marker="x",
    s=50,
    label="Predicted"
)

ax.set_xlim(1500, 2300)
ax.set_ylim(800, 350)

ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.set_title("Automatic Ball Tracking V7 — Frames 4251–4280")
ax.legend()
ax.grid(alpha=0.25)

plt.show()


In [ ]:

frames_to_check = list(range(4252, 4281, 4))

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(2, 4, figsize=(16, 9))

for ax in axes.flat:
    ax.axis("off")

for ax, frame_idx in zip(axes.flat, frames_to_check):

    match = ball_v7[
        ball_v7.frame == frame_idx
    ]

    if match.empty:
        continue

    row = match.iloc[0]

    if not np.isfinite(row.x) or not np.isfinite(row.y):
        ax.set_title(f"Frame {frame_idx} | LOST")
        continue

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

    ok, frame = cap.read()

    if not ok:
        continue

    H, W = frame.shape[:2]
    cx = int(round(row.x))
    cy = int(round(row.y))

    radius = 100

    x1 = max(0, cx - radius)
    x2 = min(W, cx + radius)
    y1 = max(0, cy - radius)
    y2 = min(H, cy + radius)

    crop = frame[y1:y2, x1:x2].copy()

    color = (
        (0, 255, 0)
        if row.status == "observed"
        else (0, 165, 255)
    )

    cv2.circle(
        crop,
        (cx - x1, cy - y1),
        15, color, 2
    )

    ax.imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    )

    ax.set_title(
        f"F{frame_idx} | {row.status}",
        fontsize=10
    )

cap.release()

plt.tight_layout()
plt.show()


In [ ]:

OUT_DIR = Path(
    "/content/football_outputs/ball_tracking_v7"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_VIDEO = OUT_DIR / "ball_v7_raw.mp4"
FINAL_VIDEO = OUT_DIR / "ball_v7_H264.mp4"

CROP_X1, CROP_Y1 = 1350, 300
CROP_X2, CROP_Y2 = 2350, 850

crop_width = CROP_X2 - CROP_X1
crop_height = CROP_Y2 - CROP_Y1

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

writer = cv2.VideoWriter(
    str(RAW_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (crop_width, crop_height)
)
assert writer.isOpened(), "VideoWriter failed"

track_lookup = {
    int(row.frame): row
    for row in ball_v7.itertuples(index=False)
}

cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

history = []

for frame_idx in range(START_FRAME, END_FRAME + 1):

    ok, frame = cap.read()

    if not ok:
        break

    crop = frame[
        CROP_Y1:CROP_Y2,
        CROP_X1:CROP_X2
    ].copy()

    record = track_lookup.get(frame_idx)

    if record is not None and np.isfinite(record.x):

        px = int(round(record.x)) - CROP_X1
        py = int(round(record.y)) - CROP_Y1

        if 0 <= px < crop_width and 0 <= py < crop_height:

            history.append((px, py))
            history = history[-12:]

            for a, b in zip(history[:-1], history[1:]):
                cv2.line(
                    crop, a, b,
                    (255, 255, 255), 2
                )

            color = (
                (0, 255, 0)
                if record.status == "observed"
                else (0, 165, 255)
            )

            cv2.circle(
                crop, (px, py),
                18, color, 3
            )

            cv2.putText(
                crop,
                record.status.upper(),
                (min(px + 20, crop_width - 150), max(30, py - 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, color, 2
            )

    cv2.putText(
        crop,
        f"Ball Tracking V7 | Frame {frame_idx}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8, (255, 255, 255), 2
    )

    writer.write(crop)

cap.release()
writer.release()

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(RAW_VIDEO),
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "23",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(FINAL_VIDEO)
    ],
    check=True,
    capture_output=True,
    text=True
)

print("V7 video created:", FINAL_VIDEO)

display(
    Video(
        str(FINAL_VIDEO),
        embed=True,
        width=1100
    )
)


In [ ]:

from google.colab import drive

drive.mount("/content/drive")

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v7"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = SAVE_DIR / "ball_tracking_v7_4251_4280.csv"
VIDEO_BACKUP = SAVE_DIR / "ball_tracking_v7_4251_4280.mp4"

ball_v7.to_csv(CSV_PATH, index=False)

shutil.copy2(
    FINAL_VIDEO,
    VIDEO_BACKUP
)

print("CSV saved:", CSV_PATH)
print("Video saved:", VIDEO_BACKUP)

print("\nFinal V7 status counts:")
print(ball_v7.status.value_counts())


In [ ]:

import cv2
import numpy as np
import pandas as pd
import torch

assert "ball_v6_observed" in globals()
assert "ball_model" in globals()

seed = ball_v6_observed.set_index("frame")

p0 = seed.loc[4254]
p1 = seed.loc[4255]

vx = float(p1.x - p0.x)
vy = float(p1.y - p0.y)

print("Initial velocity:", round(vx, 2), round(vy, 2))

kf = cv2.KalmanFilter(4, 2)

kf.transitionMatrix = np.array([
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
], dtype=np.float32)

kf.measurementMatrix = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
], dtype=np.float32)

kf.processNoiseCov = np.diag(
    [2, 2, 25, 25]
).astype(np.float32)

kf.measurementNoiseCov = (
    np.eye(2, dtype=np.float32) * 16
)

kf.errorCovPost = np.diag(
    [16, 16, 100, 100]
).astype(np.float32)

kf.statePost = np.array([
    [float(p1.x)],
    [float(p1.y)],
    [vx],
    [vy]
], dtype=np.float32)

START = 4256
END = 4280

SEARCH_RADIUS = 170
MIN_CONF = 0.005
MAX_MISSING = 6

DEVICE = 0 if torch.cuda.is_available() else "cpu"

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

cap.set(cv2.CAP_PROP_POS_FRAMES, START)

records = []
missing = 0
active = True

for frame_idx in range(START, END + 1):

    ok, frame = cap.read()
    if not ok:
        raise RuntimeError(f"Could not read frame {frame_idx}")

    if not active:
        records.append({
            "frame": frame_idx,
            "x": np.nan,
            "y": np.nan,
            "confidence": np.nan,
            "status": "lost",
            "raw_candidates": 0,
            "valid_candidates": 0,
            "nearest_distance": np.nan,
            "reason": "terminated"
        })
        continue

    prediction = kf.predict()
    px = float(prediction[0, 0])
    py = float(prediction[1, 0])

    H, W = frame.shape[:2]

    x1 = max(0, int(px - SEARCH_RADIUS))
    x2 = min(W, int(px + SEARCH_RADIUS))
    y1 = max(0, int(py - SEARCH_RADIUS))
    y2 = min(H, int(py + SEARCH_RADIUS))

    crop = frame[y1:y2, x1:x2]

    raw_count = 0
    valid_count = 0
    nearest_distance = np.nan
    reason = "no_detection"
    selected = None

    if crop.size > 0:

        result = ball_model.predict(
            source=crop,
            imgsz=1280,
            conf=MIN_CONF,
            device=DEVICE,
            verbose=False
        )[0]

        raw_count = len(result.boxes)
        options = []

        for det in result.boxes:

            box = det.xyxy[0].cpu().numpy()

            bw = float(box[2] - box[0])
            bh = float(box[3] - box[1])

            # Wider size range for local inference
            if not (3 <= bw <= 40 and 3 <= bh <= 40):
                continue

            cx = float((box[0] + box[2]) / 2 + x1)
            cy = float((box[1] + box[3]) / 2 + y1)
            conf = float(det.conf[0])

            distance = float(np.hypot(cx - px, cy - py))

            options.append({
                "x": cx,
                "y": cy,
                "confidence": conf,
                "distance": distance,
                "width": bw,
                "height": bh
            })

        valid_count = len(options)

        if options:
            nearest_distance = min(o["distance"] for o in options)

            gate = min(95.0, 45.0 + 8.0 * missing)

            eligible = [
                o for o in options
                if o["distance"] <= gate
            ]

            if eligible:
                selected = min(
                    eligible,
                    key=lambda o: (
                        o["distance"] / gate
                        - 0.15 * o["confidence"]
                    )
                )
            else:
                reason = "outside_gate"
        elif raw_count:
            reason = "size_filtered"

    if selected is not None:

        measurement = np.array([
            [selected["x"]],
            [selected["y"]]
        ], dtype=np.float32)

        corrected = kf.correct(measurement)

        x = float(corrected[0, 0])
        y = float(corrected[1, 0])

        confidence = selected["confidence"]
        status = "observed"
        reason = "matched"
        missing = 0

    else:

        missing += 1
        confidence = np.nan

        if missing <= MAX_MISSING:
            x, y = px, py
            status = "predicted"
        else:
            x = y = np.nan
            status = "lost"
            active = False

    records.append({
        "frame": frame_idx,
        "x": x,
        "y": y,
        "confidence": confidence,
        "status": status,
        "raw_candidates": raw_count,
        "valid_candidates": valid_count,
        "nearest_distance": nearest_distance,
        "reason": reason
    })

cap.release()

ball_v71 = pd.DataFrame(records)
ball_v71["timestamp_sec"] = ball_v71.frame / 25.0

print("\nV7.1 STATUS COUNTS")
print(ball_v71.status.value_counts())

print("\nRejection reasons:")
print(ball_v71.reason.value_counts())

display(ball_v71)


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import cv2

frames_to_check = [4256, 4258, 4260, 4264,
                   4268, 4272, 4276, 4280]

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(2, 4, figsize=(17, 10))

for ax, frame_idx in zip(axes.flat, frames_to_check):
    ax.axis("off")

    row = ball_v71[
        ball_v71.frame == frame_idx
    ].iloc[0]

    if not np.isfinite(row.x):
        ax.set_title(f"F{frame_idx} | LOST")
        continue

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, image = cap.read()
    if not ok:
        continue

    H, W = image.shape[:2]
    cx, cy = int(row.x), int(row.y)
    radius = 110

    x1 = max(0, cx - radius)
    x2 = min(W, cx + radius)
    y1 = max(0, cy - radius)
    y2 = min(H, cy + radius)

    crop = image[y1:y2, x1:x2].copy()

    color = (
        (0, 255, 0) if row.status == "observed"
        else (0, 165, 255)
    )

    cv2.circle(
        crop,
        (cx-x1, cy-y1),
        15, color, 2
    )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"F{frame_idx} | {row.status} | "
        f"{row.reason}",
        fontsize=9
    )

cap.release()
plt.tight_layout()
plt.show()

print("Final status counts:")
print(ball_v71.status.value_counts())


In [ ]:

from pathlib import Path

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v7"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

path = SAVE_DIR / "ball_tracking_v71_4256_4280.csv"

ball_v71.to_csv(path, index=False)

print("Saved:", path)


In [ ]:

import cv2
import numpy as np
import pandas as pd
import torch

seed = ball_v6_observed.set_index("frame")
p0, p1 = seed.loc[4254], seed.loc[4255]

kf = create_ball_kalman(
    float(p1.x),
    float(p1.y),
    float(p1.x - p0.x),
    float(p1.y - p0.y)
)

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

cap.set(cv2.CAP_PROP_POS_FRAMES, 4256)

records = []
missing = 0
active = True
device = 0 if torch.cuda.is_available() else "cpu"

for frame_idx in range(4256, 4281):
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError(f"Unable to read frame {frame_idx}")

    if not active:
        records.append({
            "frame": frame_idx,
            "x": np.nan, "y": np.nan,
            "confidence": np.nan,
            "status": "lost",
            "reason": "terminated"
        })
        continue

    pred = kf.predict()
    px, py = float(pred[0, 0]), float(pred[1, 0])

    H, W = frame.shape[:2]
    radius = 170
    x1, x2 = max(0, int(px-radius)), min(W, int(px+radius))
    y1, y2 = max(0, int(py-radius)), min(H, int(py+radius))

    crop = frame[y1:y2, x1:x2]

    candidates = []

    if crop.size:
        result = ball_model.predict(
            crop, imgsz=1280, conf=0.005,
            device=device, verbose=False
        )[0]

        for det in result.boxes:
            box = det.xyxy[0].cpu().numpy()
            bw, bh = float(box[2]-box[0]), float(box[3]-box[1])
            conf = float(det.conf[0])

            if not (8 <= bw <= 35 and 8 <= bh <= 35):
                continue

            cx = float((box[0]+box[2])/2+x1)
            cy = float((box[1]+box[3])/2+y1)
            distance = float(np.hypot(cx-px, cy-py))

            gate = 40.0 if conf >= 0.30 else 25.0

            if conf >= 0.03 and distance <= gate:
                candidates.append({
                    "x": cx, "y": cy,
                    "confidence": conf,
                    "distance": distance,
                    "cost": distance/gate - 0.15*conf
                })

    if candidates:
        best = min(candidates, key=lambda c: c["cost"])
        correction = kf.correct(
            np.array([[best["x"]], [best["y"]]], dtype=np.float32)
        )
        x, y = float(correction[0, 0]), float(correction[1, 0])
        confidence = best["confidence"]
        status, reason = "observed", "matched"
        missing = 0
    else:
        missing += 1
        confidence = np.nan
        if missing <= 6:
            x, y = px, py
            status, reason = "predicted", "no_safe_match"
        else:
            x = y = np.nan
            status, reason = "lost", "missing_limit"
            active = False

    records.append({
        "frame": frame_idx,
        "x": x, "y": y,
        "confidence": confidence,
        "status": status,
        "reason": reason
    })

cap.release()

ball_v72 = pd.DataFrame(records)

print("V7.2 status counts:")
print(ball_v72.status.value_counts())

print("\nCritical frames:")
display(ball_v72[ball_v72.frame.between(4262, 4272)])


In [ ]:

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(ball_v71.x, ball_v71.y, "-o",
        label="V7.1", alpha=0.65, markersize=3)

ax.plot(ball_v72.x, ball_v72.y, "-o",
        label="V7.2", markersize=3)

ax.scatter(
    ball_v72.loc[ball_v72.status=="observed", "x"],
    ball_v72.loc[ball_v72.status=="observed", "y"],
    color="green", s=35, label="V7.2 observed"
)

ax.invert_yaxis()
ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.set_title("Ball Tracking V7.1 vs V7.2")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

frames_to_check = [
    4262, 4263, 4264, 4265,
    4266, 4267, 4268, 4269,
    4270, 4271, 4272, 4276
]

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(3, 4, figsize=(18, 12))

for ax, frame_idx in zip(axes.flat, frames_to_check):
    ax.axis("off")

    row = ball_v72[ball_v72.frame == frame_idx]

    if row.empty:
        continue

    record = row.iloc[0]

    if not np.isfinite(record.x) or not np.isfinite(record.y):
        ax.set_title(f"F{frame_idx} | LOST")
        continue

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, image = cap.read()

    if not ok:
        continue

    H, W = image.shape[:2]

    cx = int(round(record.x))
    cy = int(round(record.y))

    radius = 90

    x1 = max(0, cx-radius)
    x2 = min(W, cx+radius)
    y1 = max(0, cy-radius)
    y2 = min(H, cy+radius)

    crop = image[y1:y2, x1:x2].copy()

    color = (
        (0, 255, 0)
        if record.status == "observed"
        else (0, 165, 255)
    )

    cv2.circle(
        crop,
        (cx-x1, cy-y1),
        14,
        color,
        2
    )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

    ax.set_title(
        f"F{frame_idx} | {record.status}",
        fontsize=10
    )

cap.release()

plt.tight_layout()
plt.show()

print("V7.2 status counts:")
print(ball_v72.status.value_counts())

print("\nCritical frame details:")
display(
    ball_v72[
        ball_v72.frame.between(4262, 4272)
    ]
)


In [ ]:

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v7"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = SAVE_DIR / "ball_tracking_v72_4256_4280.csv"

ball_v72.to_csv(CSV_PATH, index=False)

print("V7.2 saved:", CSV_PATH)
print("Frame range:", ball_v72.frame.min(), ball_v72.frame.max())
print(ball_v72.status.value_counts())


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

frames_to_check = [4277, 4278, 4279, 4280]

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(1, 4, figsize=(17, 5))

for ax, frame_idx in zip(axes, frames_to_check):
    ax.axis("off")

    record = ball_v72[
        ball_v72.frame == frame_idx
    ].iloc[0]

    if not np.isfinite(record.x):
        ax.set_title(f"F{frame_idx} | LOST")
        continue

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        continue

    H, W = frame.shape[:2]
    cx, cy = int(round(record.x)), int(round(record.y))
    radius = 95

    x1, x2 = max(0, cx-radius), min(W, cx+radius)
    y1, y2 = max(0, cy-radius), min(H, cy+radius)

    crop = frame[y1:y2, x1:x2].copy()

    color = (
        (0, 255, 0)
        if record.status == "observed"
        else (0, 165, 255)
    )

    cv2.circle(
        crop, (cx-x1, cy-y1),
        15, color, 2
    )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(f"F{frame_idx} | {record.status}")

cap.release()
plt.tight_layout()
plt.show()


In [ ]:

import cv2
import numpy as np
import pandas as pd
import torch

assert "ball_v72" in globals()
assert "ball_model" in globals()
assert "VIDEO_PATH" in globals()

# Restart from two recent observations.
seed = ball_v72.set_index("frame")
p0 = seed.loc[4276]
p1 = seed.loc[4277]

assert p0.status == "observed"
assert p1.status == "observed"

def make_kf_extension(x, y, vx, vy):
    kf = cv2.KalmanFilter(4, 2)

    kf.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ], dtype=np.float32)

    kf.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0]
    ], dtype=np.float32)

    kf.processNoiseCov = np.diag(
        [2, 2, 25, 25]
    ).astype(np.float32)

    kf.measurementNoiseCov = (
        np.eye(2, dtype=np.float32) * 16
    )

    kf.errorCovPost = np.diag(
        [16, 16, 100, 100]
    ).astype(np.float32)

    kf.statePost = np.array([
        [float(x)], [float(y)],
        [float(vx)], [float(vy)]
    ], dtype=np.float32)

    return kf

kf_ext = make_kf_extension(
    p1.x,
    p1.y,
    p1.x - p0.x,
    p1.y - p0.y
)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

cap.set(cv2.CAP_PROP_POS_FRAMES, 4278)

records = []
missing = 0
active = True

for frame_idx in range(4278, 4300):
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError(f"Failed reading frame {frame_idx}")

    if not active:
        records.append({
            "frame": frame_idx,
            "x": np.nan,
            "y": np.nan,
            "confidence": np.nan,
            "status": "lost",
            "reason": "terminated"
        })
        continue

    prediction = kf_ext.predict()
    px = float(prediction[0, 0])
    py = float(prediction[1, 0])

    h, w = frame.shape[:2]
    radius = 170

    x1 = max(0, int(px - radius))
    y1 = max(0, int(py - radius))
    x2 = min(w, int(px + radius))
    y2 = min(h, int(py + radius))

    crop = frame[y1:y2, x1:x2]
    candidates = []

    if crop.size:
        result = ball_model.predict(
            source=crop,
            imgsz=1280,
            conf=0.005,
            device=DEVICE,
            verbose=False
        )[0]

        for det in result.boxes:
            box = det.xyxy[0].cpu().numpy()

            bw = float(box[2] - box[0])
            bh = float(box[3] - box[1])
            conf = float(det.conf[0])

            if not (8 <= bw <= 35 and 8 <= bh <= 35):
                continue

            cx = float((box[0] + box[2]) / 2 + x1)
            cy = float((box[1] + box[3]) / 2 + y1)

            distance = float(np.hypot(cx - px, cy - py))

            gate = 40.0 if conf >= 0.30 else 25.0

            if conf >= 0.03 and distance <= gate:
                candidates.append({
                    "x": cx,
                    "y": cy,
                    "confidence": conf,
                    "distance": distance,
                    "cost": distance / gate - 0.15 * conf
                })

    if candidates:
        best = min(candidates, key=lambda c: c["cost"])

        corrected = kf_ext.correct(
            np.array([
                [best["x"]],
                [best["y"]]
            ], dtype=np.float32)
        )

        x = float(corrected[0, 0])
        y = float(corrected[1, 0])
        conf = best["confidence"]

        status = "observed"
        reason = "matched"
        missing = 0

    else:
        missing += 1
        conf = np.nan

        if missing <= 6:
            x, y = px, py
            status = "predicted"
            reason = "no_safe_match"
        else:
            x = y = np.nan
            status = "lost"
            reason = "missing_limit"
            active = False

    records.append({
        "frame": frame_idx,
        "x": x,
        "y": y,
        "confidence": conf,
        "status": status,
        "reason": reason
    })

cap.release()

ball_extension = pd.DataFrame(records)
ball_extension["timestamp_sec"] = ball_extension.frame / 25.0

print("Extension status counts:")
print(ball_extension.status.value_counts())

print("\nLast 10 frames:")
display(ball_extension.tail(10))


In [ ]:

import matplotlib.pyplot as plt

frames_to_check = [
    4282, 4285, 4288, 4290,
    4293, 4295, 4297, 4299
]

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(2, 4, figsize=(17, 9))

last_known = (
    float(p1.x),
    float(p1.y)
)

for ax, frame_idx in zip(axes.flat, frames_to_check):
    ax.axis("off")

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        continue

    record = ball_extension[
        ball_extension.frame == frame_idx
    ].iloc[0]

    if np.isfinite(record.x) and np.isfinite(record.y):
        last_known = (record.x, record.y)

    cx, cy = map(int, last_known)
    h, w = frame.shape[:2]
    r = 120

    x1 = max(0, cx-r)
    x2 = min(w, cx+r)
    y1 = max(0, cy-r)
    y2 = min(h, cy+r)

    crop = frame[y1:y2, x1:x2].copy()

    if np.isfinite(record.x) and np.isfinite(record.y):
        color = (
            (0, 255, 0)
            if record.status == "observed"
            else (0, 165, 255)
        )

        cv2.circle(
            crop,
            (int(record.x)-x1, int(record.y)-y1),
            15, color, 2
        )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(f"F{frame_idx} | {record.status}")

cap.release()
plt.tight_layout()
plt.show()


In [ ]:

from pathlib import Path

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_v7"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

output_csv = SAVE_DIR / "ball_tracking_extension_4278_4299.csv"

ball_extension.to_csv(output_csv, index=False)

print("Saved:", output_csv)


In [ ]:
# Preserve original experiment
ball_extension_reviewed = ball_extension.copy()

# These positions need additional visual evidence.
uncertain_frames = [4293, 4294, 4295, 4296, 4299]

mask = ball_extension_reviewed.frame.isin(uncertain_frames)

ball_extension_reviewed.loc[mask, ["x", "y"]] = np.nan
ball_extension_reviewed.loc[mask, "status"] = "uncertain"

print(ball_extension_reviewed.tail(10)[
    ["frame", "x", "y", "status"]
])

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

combined = pd.DataFrame({
    "frame": range(4000, 4300)
})

combined["timestamp_sec"] = combined.frame / 25.0
combined["ball_x"] = np.nan
combined["ball_y"] = np.nan
combined["confidence"] = np.nan
combined["status"] = "lost"
combined["track_segment_id"] = pd.NA

def insert_segment(source, start, end, segment_id):
    global combined

    part = source[
        source.frame.between(start, end)
    ].copy()

    part = part.drop_duplicates("frame", keep="last")
    part = part.set_index("frame")

    for frame_idx, row in part.iterrows():
        idx = combined.index[combined.frame == frame_idx]

        if len(idx) == 0:
            continue

        i = idx[0]

        combined.loc[i, "ball_x"] = row.x
        combined.loc[i, "ball_y"] = row.y
        combined.loc[i, "confidence"] = row.get(
            "confidence", np.nan
        )
        combined.loc[i, "status"] = row.status
        combined.loc[i, "track_segment_id"] = segment_id

insert_segment(ball_v4, 4086, 4184, "V4")
insert_segment(ball_v51, 4208, 4244, "V5.1")
insert_segment(ball_v6, 4248, 4256, "V6")
insert_segment(ball_v72, 4257, 4277, "V7.2")
insert_segment(
    ball_extension_reviewed,
    4278, 4299,
    "V7.2_extension"
)

# Uncertain/lost locations must not contain coordinates.
invalid = combined.status.isin(["lost", "uncertain"])
combined.loc[invalid, ["ball_x", "ball_y"]] = np.nan

assert len(combined) == 300
assert combined.frame.is_unique

print("Final Segment C statuses:")
print(combined.status.value_counts())

print("\nExperiment segments:")
print(combined.track_segment_id.value_counts(dropna=False))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

OUT = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_consolidated"
)
OUT.mkdir(parents=True, exist_ok=True)

output_csv = OUT / "segment_C_ball_tracking_provisional.csv"

combined.to_csv(output_csv, index=False)

print("Saved:", output_csv)
print("Rows:", len(combined))

In [ ]:

import pandas as pd
import numpy as np

audit = combined.copy()

audit["needs_review"] = (
    audit["status"].isin(["lost", "uncertain"])
    | audit["ball_x"].isna()
    | audit["ball_y"].isna()
)

audit["run_id"] = (
    audit["needs_review"] != audit["needs_review"].shift()
).cumsum()

gap_summary = (
    audit[audit.needs_review]
    .groupby("run_id")
    .agg(
        start_frame=("frame", "min"),
        end_frame=("frame", "max"),
        frames=("frame", "size")
    )
    .reset_index(drop=True)
    .sort_values("frames", ascending=False)
)

gap_summary["duration_sec"] = (
    gap_summary["frames"] / 25.0
)

print("Missing or uncertain intervals:")
display(gap_summary)

print("Total frames requiring review:",
      int(audit.needs_review.sum()))


In [ ]:

import matplotlib.pyplot as plt

status_codes = {
    "observed": 4,
    "interpolated": 3,
    "predicted": 2,
    "uncertain": 1,
    "lost": 0
}

colors = {
    "observed": "green",
    "interpolated": "blue",
    "predicted": "orange",
    "uncertain": "red",
    "lost": "gray"
}

fig, ax = plt.subplots(figsize=(16, 4))

for status, group in combined.groupby("status"):
    ax.scatter(
        group["frame"],
        group["status"].map(status_codes),
        s=16,
        color=colors.get(status, "black"),
        label=status
    )

ax.set_yticks(list(status_codes.values()))
ax.set_yticklabels(list(status_codes.keys()))
ax.set_xlabel("Frame number")
ax.set_title("Segment C - Ball tracking coverage and gaps")
ax.set_xlim(3995, 4305)
ax.grid(axis="x", alpha=0.2)
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Inspect frames spanning the V4-to-V5.1 gap
FRAMES = [
    4184, 4186, 4188, 4190,
    4192, 4194, 4196, 4198,
    4200, 4202, 4204, 4206,
    4208
]

# Same fixed region for every frame
X1, Y1 = 2700, 300
X2, Y2 = 3550, 820

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Could not open match video"

fig, axes = plt.subplots(4, 4, figsize=(20, 15))

for ax in axes.flat:
    ax.axis("off")

for ax, frame_idx in zip(axes.flat, FRAMES):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        ax.set_title(f"Frame {frame_idx}: Read failed")
        continue

    crop = frame[Y1:Y2, X1:X2].copy()

    # Display all stored YOLO candidates
    detections = candidates_v4[
        candidates_v4.frame == frame_idx
    ].sort_values("confidence", ascending=False)

    for candidate_idx, det in enumerate(
        detections.itertuples(index=False)
    ):
        px = int(round(det.center_x))
        py = int(round(det.center_y))

        if X1 <= px < X2 and Y1 <= py < Y2:
            cv2.circle(
                crop,
                (px - X1, py - Y1),
                11,
                (0, 0, 255),
                2
            )

            cv2.putText(
                crop,
                f"{candidate_idx}:{det.confidence:.2f}",
                (px - X1 + 13, max(20, py - Y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (0, 0, 255),
                1
            )

    # Display existing V4 track estimate
    track = ball_v4[
        ball_v4.frame == frame_idx
    ]

    status = "unknown"

    if not track.empty:
        row = track.iloc[0]
        status = row.status

        if np.isfinite(row.x) and np.isfinite(row.y):
            tx = int(round(row.x))
            ty = int(round(row.y))

            if X1 <= tx < X2 and Y1 <= ty < Y2:
                color = (
                    (0, 255, 0)
                    if row.status == "observed"
                    else (0, 165, 255)
                )

                cv2.circle(
                    crop,
                    (tx - X1, ty - Y1),
                    16,
                    color,
                    2
                )

    # Mark the independently inspected frame-4208 anchor
    if frame_idx == 4208:
        ax_x = int(round(3063.61)) - X1
        ax_y = int(round(588.75)) - Y1

        cv2.drawMarker(
            crop,
            (ax_x, ax_y),
            (255, 0, 255),
            cv2.MARKER_CROSS,
            22,
            2
        )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"Frame {frame_idx} | V4: {status} | "
        f"{len(detections)} candidates",
        fontsize=10
    )

cap.release()

plt.suptitle(
    "Segment C - Active Ball Recovery Diagnostic",
    fontsize=16
)

plt.tight_layout()
plt.show()


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RECOVERY_FRAMES = list(range(4196, 4209))

# Search near the players and near touchline
X1, Y1 = 2700, 350
X2, Y2 = 3550, 750

TILE = 320
STRIDE = 240

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

recovery_records = []

for frame_idx in RECOVERY_FRAMES:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        continue

    region = frame[Y1:Y2, X1:X2]

    rh, rw = region.shape[:2]

    xs = list(range(0, max(1, rw - TILE + 1), STRIDE))
    ys = list(range(0, max(1, rh - TILE + 1), STRIDE))

    xs.append(max(0, rw - TILE))
    ys.append(max(0, rh - TILE))

    for ty in sorted(set(ys)):
        for tx in sorted(set(xs)):
            crop = region[ty:ty + TILE, tx:tx + TILE]

            result = ball_model.predict(
                source=crop,
                imgsz=1280,
                conf=0.005,
                device=DEVICE,
                verbose=False
            )[0]

            for det in result.boxes:
                b = det.xyxy[0].cpu().numpy()

                bw = float(b[2] - b[0])
                bh = float(b[3] - b[1])

                if not (3 <= bw <= 40 and 3 <= bh <= 40):
                    continue

                recovery_records.append({
                    "frame": frame_idx,
                    "center_x": float((b[0] + b[2])/2 + X1 + tx),
                    "center_y": float((b[1] + b[3])/2 + Y1 + ty),
                    "confidence": float(det.conf[0]),
                    "width": bw,
                    "height": bh
                })

cap.release()

recovery_df = pd.DataFrame(
    recovery_records,
    columns=[
        "frame", "center_x", "center_y",
        "confidence", "width", "height"
    ]
)

# Merge overlapping tile detections of the same object.
from torchvision.ops import nms
import torch

merged = []

for frame_idx, group in recovery_df.groupby("frame"):
    boxes = np.column_stack([
        group.center_x - group.width / 2,
        group.center_y - group.height / 2,
        group.center_x + group.width / 2,
        group.center_y + group.height / 2
    ])

    keep = nms(
        torch.tensor(boxes, dtype=torch.float32),
        torch.tensor(group.confidence.to_numpy(), dtype=torch.float32),
        0.4
    ).numpy()

    merged.append(group.iloc[keep])

recovery_df = (
    pd.concat(merged, ignore_index=True)
    if merged else recovery_df
)

print("Total local candidates:", len(recovery_df))
print("Frames with candidates:", recovery_df.frame.nunique())

display(
    recovery_df.groupby("frame")
    .agg(
        candidates=("confidence", "size"),
        max_confidence=("confidence", "max")
    )
)

In [ ]:
FRAMES = [4196, 4198, 4200, 4202, 4204, 4206, 4208]

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(2, 4, figsize=(18, 10))

for ax in axes.flat:
    ax.axis("off")

for ax, frame_idx in zip(axes.flat, FRAMES):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        continue

    crop = frame[Y1:Y2, X1:X2].copy()

    detections = recovery_df[
        recovery_df.frame == frame_idx
    ].sort_values("confidence", ascending=False)

    for i, det in enumerate(detections.itertuples(index=False)):
        px = int(det.center_x - X1)
        py = int(det.center_y - Y1)

        cv2.circle(crop, (px, py), 10, (0, 0, 255), 2)

        cv2.putText(
            crop,
            f"{i}:{det.confidence:.2f}",
            (px + 12, max(16, py - 7)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (0, 0, 255),
            1
        )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"Frame {frame_idx} | {len(detections)} detections"
    )

cap.release()
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

OUT_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_recovery"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = OUT_DIR / "local_candidates_4196_4208.csv"

recovery_df.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

FRAMES = list(range(4196, 4209))

# Near-touchline action region
X1, Y1 = 2850, 470
X2, Y2 = 3400, 730

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(4, 4, figsize=(20, 13))

for ax in axes.flat:
    ax.axis("off")

for ax, frame_idx in zip(axes.flat, FRAMES):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        continue

    crop = frame[Y1:Y2, X1:X2].copy()

    pool = recovery_df[
        recovery_df.frame == frame_idx
    ].sort_values("confidence", ascending=False)

    for idx, det in enumerate(pool.itertuples(index=False)):
        cx = int(round(det.center_x))
        cy = int(round(det.center_y))

        if X1 <= cx < X2 and Y1 <= cy < Y2:
            cv2.circle(
                crop,
                (cx - X1, cy - Y1),
                11,
                (0, 0, 255),
                2
            )

            cv2.putText(
                crop,
                f"{idx}:{det.confidence:.3f}",
                (cx-X1+12, max(16, cy-Y1-8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (0, 0, 255),
                1
            )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {frame_idx}", fontsize=10)

cap.release()

plt.tight_layout()
plt.show()

In [ ]:
near_touchline = recovery_df[
    recovery_df.center_x.between(2850, 3400) &
    recovery_df.center_y.between(470, 730)
].copy()

near_touchline = near_touchline.sort_values(
    ["frame", "confidence"],
    ascending=[True, False]
)

display(
    near_touchline[
        [
            "frame",
            "center_x",
            "center_y",
            "confidence",
            "width",
            "height"
        ]
    ].round(3)
)

print("Candidate counts by frame:")
print(near_touchline.groupby("frame").size())

In [ ]:
import numpy as np
import pandas as pd

verified_points = pd.DataFrame([
    {"frame": 4206, "x": 3128.204, "y": 598.737},
    {"frame": 4207, "x": 3095.529, "y": 593.374},
    {"frame": 4208, "x": 3063.122, "y": 589.130}
])

vx = (verified_points.iloc[-1].x - verified_points.iloc[0].x) / 2
vy = (verified_points.iloc[-1].y - verified_points.iloc[0].y) / 2

search_points = []

for f in range(4200, 4206):
    dt = f - 4206

    search_points.append({
        "frame": f,
        "expected_x": 3128.204 + vx * dt,
        "expected_y": 598.737 + vy * dt
    })

search_df = pd.DataFrame(search_points)

print("Estimated motion:", round(vx, 2), round(vy, 2))
display(search_df.round(2))

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

recovered = []

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, row in zip(axes.flat, search_df.itertuples(index=False)):
    frame_idx = int(row.frame)

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, image = cap.read()

    ax.axis("off")

    if not ok:
        continue

    H, W = image.shape[:2]

    cx = int(row.expected_x)
    cy = int(row.expected_y)

    radius = 160

    x1 = max(0, cx - radius)
    x2 = min(W, cx + radius)
    y1 = max(0, cy - radius)
    y2 = min(H, cy + radius)

    crop = image[y1:y2, x1:x2].copy()

    result = ball_model.predict(
        source=crop,
        imgsz=1280,
        conf=0.005,
        device=DEVICE,
        verbose=False
    )[0]

    for i, det in enumerate(result.boxes):
        box = det.xyxy[0].cpu().numpy()

        bx = float((box[0] + box[2]) / 2 + x1)
        by = float((box[1] + box[3]) / 2 + y1)

        conf = float(det.conf[0])
        bw = float(box[2] - box[0])
        bh = float(box[3] - box[1])

        distance = float(np.hypot(
            bx - row.expected_x,
            by - row.expected_y
        ))

        recovered.append({
            "frame": frame_idx,
            "candidate": i,
            "center_x": bx,
            "center_y": by,
            "confidence": conf,
            "width": bw,
            "height": bh,
            "distance_from_expected": distance
        })

        cv2.circle(
            crop,
            (int(bx-x1), int(by-y1)),
            11, (0, 0, 255), 2
        )

        cv2.putText(
            crop,
            f"{i}: {conf:.3f}",
            (int(bx-x1)+12, int(by-y1)-8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45, (0, 0, 255), 1
        )

    cv2.drawMarker(
        crop,
        (cx-x1, cy-y1),
        (255, 0, 0),
        cv2.MARKER_CROSS,
        18, 2
    )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"F{frame_idx} | {len(result.boxes)} candidates"
    )

cap.release()

plt.tight_layout()
plt.show()

recovered_df = pd.DataFrame(
    recovered,
    columns=[
        "frame", "candidate", "center_x", "center_y",
        "confidence", "width", "height",
        "distance_from_expected"
    ]
)

display(recovered_df.round(3))

In [ ]:
from pathlib import Path

OUT = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_recovery"
)
OUT.mkdir(parents=True, exist_ok=True)

recovered_df.to_csv(
    OUT / "motion_guided_candidates_4200_4205.csv",
    index=False
)

print("Recovery diagnostics saved.")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

FRAMES = list(range(4202, 4209))

# Fixed view around the near-touchline players
X1, Y1 = 2900, 420
X2, Y2 = 3450, 720

KNOWN_BALL = {
    4206: (3128.204, 598.737),
    4207: (3095.529, 593.374),
    4208: (3063.122, 589.130)
}

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

fig, axes = plt.subplots(2, 4, figsize=(19, 10))

for ax in axes.flat:
    ax.axis("off")

for ax, frame_idx in zip(axes.flat, FRAMES):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

    ok, frame = cap.read()
    if not ok:
        continue

    crop = frame[Y1:Y2, X1:X2].copy()

    if frame_idx in KNOWN_BALL:
        bx, by = KNOWN_BALL[frame_idx]

        cv2.circle(
            crop,
            (int(bx - X1), int(by - Y1)),
            12,
            (0, 255, 0),
            2
        )

    ax.imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB),
        interpolation="nearest"
    )

    ax.set_title(
        f"Frame {frame_idx} | " +
        ("Ball candidate" if frame_idx in KNOWN_BALL else "Unresolved")
    )

plt.suptitle("Backward Ball Recovery — Fixed Player Region")
plt.tight_layout()
plt.show()

cap.release()

In [ ]:
import pandas as pd
import numpy as np

combined_reviewed = combined.copy()

verified_ball_points = {
    4206: (3128.204, 598.737, 0.059921),
    4207: (3095.529, 593.374, 0.015000),
    4208: (3063.122, 589.130, 0.487661)
}

for frame_idx, (x, y, conf) in verified_ball_points.items():
    mask = combined_reviewed.frame == frame_idx

    assert mask.sum() == 1

    combined_reviewed.loc[mask, "ball_x"] = x
    combined_reviewed.loc[mask, "ball_y"] = y
    combined_reviewed.loc[mask, "confidence"] = conf
    combined_reviewed.loc[mask, "status"] = "observed"
    combined_reviewed.loc[mask, "track_segment_id"] = "recovery_4206_4208"

print("Updated status counts:")
print(combined_reviewed.status.value_counts())

display(
    combined_reviewed[
        combined_reviewed.frame.between(4202, 4209)
    ]
)

In [ ]:
from pathlib import Path

SAVE_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_tracking_consolidated"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    SAVE_DIR /
    "segment_C_ball_tracking_verified_recovery_v1.csv"
)

combined_reviewed.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)

In [ ]:
import pandas as pd
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Football_CV/"
    "ball_tracking_consolidated/"
    "segment_C_ball_tracking_verified_recovery_v1.csv"
)

assert path.exists(), "Consolidated CSV not found"

df = pd.read_csv(path)

assert len(df) == 300
assert df["frame"].is_unique
assert df["frame"].min() == 4000
assert df["frame"].max() == 4299

print("Final tracking statuses:")
print(df["status"].value_counts())

print("\nRecovered frames 4206–4208:")
display(df[df["frame"].between(4206, 4208)])

print("\nFrames still unresolved:")
print(df[df["status"].isin(["lost", "uncertain"])].shape[0])

print("\nCSV validation successful.")

In [ ]:

import cv2
import pandas as pd
import zipfile
from pathlib import Path

BENCHMARK_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_benchmark_v1"
)
IMAGES_DIR = BENCHMARK_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

early_frames = [
    4000, 4012, 4024, 4036,
    4048, 4060, 4072, 4084
]

difficult_frames = list(range(4185, 4209))

reference_frames = [
    4100, 4116, 4132, 4160,
    4176, 4220, 4240, 4276
]

benchmark_frames = sorted(set(
    early_frames + difficult_frames + reference_frames
))

assert len(benchmark_frames) == 40

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Unable to open match video"

records = []

for frame_idx in benchmark_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        raise RuntimeError(f"Could not read frame {frame_idx}")

    filename = f"frame_{frame_idx:05d}.jpg"
    output = IMAGES_DIR / filename

    cv2.imwrite(
        str(output),
        frame,
        [cv2.IMWRITE_JPEG_QUALITY, 92]
    )

    records.append({
        "frame": frame_idx,
        "filename": filename,
        "gt_ball_visible": "",
        "gt_ball_x": "",
        "gt_ball_y": "",
        "annotation_status": "pending"
    })

cap.release()

annotation_template = pd.DataFrame(records)

manifest_path = BENCHMARK_DIR / "annotation_template.csv"
annotation_template.to_csv(manifest_path, index=False)

zip_path = BENCHMARK_DIR / "football_benchmark_40_frames.zip"

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for image_path in sorted(IMAGES_DIR.glob("*.jpg")):
        archive.write(image_path, arcname=image_path.name)

print("Benchmark created successfully.")
print("Images:", len(list(IMAGES_DIR.glob("*.jpg"))))
print("Images ZIP:", zip_path)
print("Annotation template:", manifest_path)


In [ ]:

prediction_reference = df[
    df.frame.isin(benchmark_frames)
].copy()

prediction_path = (
    BENCHMARK_DIR / "benchmark_model_predictions.csv"
)

prediction_reference.to_csv(
    prediction_path, index=False
)

print("Predictions saved separately:", prediction_path)
print("Benchmark frames:", len(prediction_reference))

display(
    prediction_reference[
        ["frame", "ball_x", "ball_y", "status"]
    ].head(10)
)


In [ ]:

import base64
import json
import numpy as np
import pandas as pd

from pathlib import Path
from google.colab.output import eval_js

BENCHMARK_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_benchmark_v1"
)

IMAGES_DIR = BENCHMARK_DIR / "images"

TEMPLATE_PATH = BENCHMARK_DIR / "annotation_template.csv"
GROUND_TRUTH_PATH = BENCHMARK_DIR / "ball_ground_truth_v1.csv"

assert TEMPLATE_PATH.exists()
assert IMAGES_DIR.exists()

def annotate_ball(image_path, frame_number):

    image_data = (
        "data:image/jpeg;base64,"
        + base64.b64encode(image_path.read_bytes()).decode("utf-8")
    )

    js = r"""
    new Promise((resolve) => {
        const previous = document.getElementById("football-labeler");
        if (previous) previous.remove();

        const root = document.createElement("div");
        root.id = "football-labeler";

        root.style.cssText = `
            position: fixed;
            inset: 0;
            z-index: 2147483640;
            background: #101827;
            color: white;
            overflow: auto;
            padding: 20px;
            font-family: sans-serif;
        `;

        root.innerHTML = `
            <div style="max-width:1500px;margin:auto;">
                <h2>Ball Ground Truth — Frame __FRAME__</h2>
                <p>Hover to magnify. Click the center of the ACTIVE match ball.</p>

                <img id="football-image"
                     style="width:100%;cursor:crosshair;display:block;">

                <div style="display:flex;gap:20px;align-items:center;
                            flex-wrap:wrap;margin-top:15px;">
                    <canvas id="football-zoom" width="260" height="260"
                        style="border:2px solid white;background:#333;">
                    </canvas>

                    <div>
                        <p id="football-coordinate">
                            Move your cursor over the image
                        </p>
                        <button id="football-absent"
                            style="padding:12px;margin:5px;">
                            Ball not visible
                        </button>
                        <button id="football-uncertain"
                            style="padding:12px;margin:5px;">
                            Cannot determine
                        </button>
                    </div>
                </div>
            </div>
        `;

        document.body.appendChild(root);

        const img = root.querySelector("#football-image");
        const canvas = root.querySelector("#football-zoom");
        const ctx = canvas.getContext("2d");
        const info = root.querySelector("#football-coordinate");

        function finish(result) {
            root.remove();
            resolve(result);
        }

        function getPosition(event) {
            const rect = img.getBoundingClientRect();

            const x = (
                (event.clientX - rect.left)
                * img.naturalWidth / rect.width
            );

            const y = (
                (event.clientY - rect.top)
                * img.naturalHeight / rect.height
            );

            return {
                x: Math.max(0, Math.min(img.naturalWidth - 1, x)),
                y: Math.max(0, Math.min(img.naturalHeight - 1, y))
            };
        }

        img.addEventListener("mousemove", (event) => {
            const p = getPosition(event);

            info.textContent =
                "Original image: X=" + p.x.toFixed(1)
                + " Y=" + p.y.toFixed(1);

            const sourceSize = 90;

            const sx = Math.max(0, Math.min(
                img.naturalWidth - sourceSize,
                p.x - sourceSize / 2
            ));

            const sy = Math.max(0, Math.min(
                img.naturalHeight - sourceSize,
                p.y - sourceSize / 2
            ));

            ctx.clearRect(0, 0, 260, 260);
            ctx.drawImage(
                img, sx, sy, sourceSize, sourceSize,
                0, 0, 260, 260
            );

            const zx = (p.x - sx) * 260 / sourceSize;
            const zy = (p.y - sy) * 260 / sourceSize;

            ctx.strokeStyle = "red";
            ctx.lineWidth = 1;
            ctx.beginPath();
            ctx.moveTo(zx - 12, zy);
            ctx.lineTo(zx + 12, zy);
            ctx.moveTo(zx, zy - 12);
            ctx.lineTo(zx, zy + 12);
            ctx.stroke();
        });

        img.addEventListener("click", (event) => {
            const p = getPosition(event);

            finish({
                kind: "visible",
                x: Math.round(p.x),
                y: Math.round(p.y)
            });
        });

        root.querySelector("#football-absent").onclick = () => {
            finish({kind: "not_visible"});
        };

        root.querySelector("#football-uncertain").onclick = () => {
            finish({kind: "uncertain"});
        };

        img.src = __IMAGE_DATA__;
    })
    """

    js = js.replace("__FRAME__", str(frame_number))
    js = js.replace("__IMAGE_DATA__", json.dumps(image_data))

    return eval_js(js)

print("Interactive annotation tool ready.")


In [ ]:

if GROUND_TRUTH_PATH.exists():
    annotations = pd.read_csv(GROUND_TRUTH_PATH)
else:
    annotations = pd.read_csv(TEMPLATE_PATH)

MAX_NEW = 5
completed_now = 0

for idx, row in annotations.iterrows():

    if row["annotation_status"] != "pending":
        continue

    frame_idx = int(row["frame"])
    image_path = IMAGES_DIR / row["filename"]

    assert image_path.exists(), f"Missing: {image_path}"

    print("Annotating frame:", frame_idx)

    result = annotate_ball(image_path, frame_idx)

    kind = result["kind"]

    if kind == "visible":
        annotations.loc[idx, "gt_ball_visible"] = 1
        annotations.loc[idx, "gt_ball_x"] = float(result["x"])
        annotations.loc[idx, "gt_ball_y"] = float(result["y"])
        annotations.loc[idx, "annotation_status"] = "annotated"

    elif kind == "not_visible":
        annotations.loc[idx, "gt_ball_visible"] = 0
        annotations.loc[idx, ["gt_ball_x", "gt_ball_y"]] = np.nan
        annotations.loc[idx, "annotation_status"] = "not_visible"

    else:
        annotations.loc[idx, "gt_ball_visible"] = np.nan
        annotations.loc[idx, ["gt_ball_x", "gt_ball_y"]] = np.nan
        annotations.loc[idx, "annotation_status"] = "uncertain"

    annotations.to_csv(GROUND_TRUTH_PATH, index=False)

    completed_now += 1
    print("Saved frame", frame_idx, "as", kind)

    if MAX_NEW is not None and completed_now >= MAX_NEW:
        break

print("\nAnnotation progress:")
print(annotations["annotation_status"].value_counts())

print("\nSaved:", GROUND_TRUTH_PATH)


In [ ]:

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import json
import base64
import shutil

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/Football_CV")

BENCHMARK_DIR = ROOT / "ball_benchmark_v1"
IMAGES_DIR = BENCHMARK_DIR / "images"

TEMPLATE_PATH = BENCHMARK_DIR / "annotation_template.csv"
GROUND_TRUTH_PATH = BENCHMARK_DIR / "ball_ground_truth_v1.csv"
PREDICTIONS_PATH = BENCHMARK_DIR / "benchmark_model_predictions.csv"

CONSOLIDATED_PATH = (
    ROOT / "ball_tracking_consolidated"
    / "segment_C_ball_tracking_verified_recovery_v1.csv"
)

CANDIDATES_PATH = (
    ROOT / "ball_tracking_v2"
    / "segment_C_ball_candidates_allframes.csv"
)

DRIVE_VIDEO_PATH = ROOT / "videos" / "117093.mp4"
VIDEO_PATH = Path("/content/football_dataset/mot/clips/117093.mp4")

resources = {
    "Benchmark images": IMAGES_DIR,
    "Annotation template": TEMPLATE_PATH,
    "Saved ground truth": GROUND_TRUTH_PATH,
    "Model predictions": PREDICTIONS_PATH,
    "Consolidated tracking CSV": CONSOLIDATED_PATH,
    "YOLO candidate CSV": CANDIDATES_PATH,
    "Video backup": DRIVE_VIDEO_PATH,
}

print("PROJECT RESOURCE CHECK")
print("=" * 65)

for name, path in resources.items():
    status = "FOUND" if path.exists() else "NOT FOUND"
    print(f"{name:27} : {status}")

assert TEMPLATE_PATH.exists(), "Annotation template missing"
assert IMAGES_DIR.exists(), "Benchmark images directory missing"

print("\nBenchmark images:",
      len(list(IMAGES_DIR.glob("*.jpg"))))

print("\nGoogle Drive restoration complete.")


In [ ]:

# Load previously saved model predictions
predictions_df = pd.read_csv(PREDICTIONS_PATH)

# Load consolidated Segment C trajectory
combined_reviewed = pd.read_csv(CONSOLIDATED_PATH)

# Load all-frame ball candidates
candidates_v4 = pd.read_csv(CANDIDATES_PATH)

# Resume ground truth if it was previously saved
if GROUND_TRUTH_PATH.exists():
    annotations = pd.read_csv(GROUND_TRUTH_PATH)
    print("Resuming previously saved annotations.")
else:
    annotations = pd.read_csv(TEMPLATE_PATH)
    print("Starting from the original annotation template.")

annotations["annotation_status"] = (
    annotations["annotation_status"]
    .fillna("pending")
    .astype(str)
)

completed = (
    annotations["annotation_status"] != "pending"
).sum()

remaining = (
    annotations["annotation_status"] == "pending"
).sum()

print("\nANNOTATION PROGRESS")
print("=" * 45)
print("Total benchmark frames:", len(annotations))
print("Completed:", completed)
print("Remaining:", remaining)

print("\nStatus breakdown:")
print(annotations["annotation_status"].value_counts())

display(annotations.head(10))


In [ ]:

from google.colab.output import eval_js
import base64
import json

def annotate_ball(image_path, frame_number):

    image_data = (
        "data:image/jpeg;base64,"
        + base64.b64encode(
            Path(image_path).read_bytes()
        ).decode("utf-8")
    )

    js = r"""
    new Promise((resolve) => {

        const old = document.getElementById("football-labeler");
        if (old) old.remove();

        const root = document.createElement("div");
        root.id = "football-labeler";

        root.style.cssText = `
            position: fixed;
            inset: 0;
            z-index: 2147483640;
            background: #101827;
            color: white;
            overflow: auto;
            padding: 20px;
            font-family: sans-serif;
        `;

        root.innerHTML = `
            <div style="max-width:1500px;margin:auto;">
                <h2>Ball Annotation - Frame __FRAME__</h2>

                <p>
                    Hover over the image to zoom.
                    Click the center of the ACTIVE match ball.
                </p>

                <img
                    id="football-image"
                    style="width:100%;cursor:crosshair;display:block;"
                >

                <div style="
                    display:flex;
                    align-items:center;
                    gap:20px;
                    flex-wrap:wrap;
                    margin-top:15px;
                ">

                    <canvas
                        id="football-zoom"
                        width="300"
                        height="300"
                        style="border:2px solid white;background:#333;"
                    ></canvas>

                    <div>
                        <p id="football-coordinate">
                            Move your cursor over the image
                        </p>

                        <button id="football-absent"
                            style="padding:12px;margin:5px;">
                            Ball not visible
                        </button>

                        <button id="football-uncertain"
                            style="padding:12px;margin:5px;">
                            Cannot determine
                        </button>
                    </div>
                </div>
            </div>
        `;

        document.body.appendChild(root);

        const img = root.querySelector("#football-image");
        const canvas = root.querySelector("#football-zoom");
        const ctx = canvas.getContext("2d");
        const info = root.querySelector("#football-coordinate");

        function finish(result) {
            root.remove();
            resolve(result);
        }

        function getPosition(event) {
            const rect = img.getBoundingClientRect();

            let x = (event.clientX - rect.left)
                    * img.naturalWidth / rect.width;

            let y = (event.clientY - rect.top)
                    * img.naturalHeight / rect.height;

            x = Math.max(0, Math.min(img.naturalWidth-1, x));
            y = Math.max(0, Math.min(img.naturalHeight-1, y));

            return {x, y};
        }

        img.addEventListener("mousemove", (event) => {

            const p = getPosition(event);

            info.textContent =
                "Original coordinates: X="
                + p.x.toFixed(1)
                + " Y=" + p.y.toFixed(1);

            const sourceSize = 80;

            const sx = Math.max(
                0,
                Math.min(
                    img.naturalWidth-sourceSize,
                    p.x-sourceSize/2
                )
            );

            const sy = Math.max(
                0,
                Math.min(
                    img.naturalHeight-sourceSize,
                    p.y-sourceSize/2
                )
            );

            ctx.clearRect(0,0,300,300);

            ctx.drawImage(
                img,
                sx,sy,sourceSize,sourceSize,
                0,0,300,300
            );

            const zx = (p.x-sx)*300/sourceSize;
            const zy = (p.y-sy)*300/sourceSize;

            ctx.strokeStyle = "red";
            ctx.lineWidth = 2;

            ctx.beginPath();

            ctx.moveTo(zx-15,zy);
            ctx.lineTo(zx+15,zy);

            ctx.moveTo(zx,zy-15);
            ctx.lineTo(zx,zy+15);

            ctx.stroke();
        });

        img.addEventListener("click", (event) => {
            const p = getPosition(event);

            finish({
                kind: "visible",
                x: Math.round(p.x),
                y: Math.round(p.y)
            });
        });

        root.querySelector("#football-absent").onclick = () => {
            finish({kind:"not_visible"});
        };

        root.querySelector("#football-uncertain").onclick = () => {
            finish({kind:"uncertain"});
        };

        img.src = __IMAGE_DATA__;
    })
    """

    js = js.replace("__FRAME__", str(frame_number))
    js = js.replace("__IMAGE_DATA__", json.dumps(image_data))

    return eval_js(js)


print("Ball annotation interface restored successfully.")


In [ ]:

import base64
import json
from pathlib import Path
from google.colab.output import eval_js

def annotate_ball(image_path, frame_number):

    image_data = (
        "data:image/jpeg;base64,"
        + base64.b64encode(
            Path(image_path).read_bytes()
        ).decode("utf-8")
    )

    js = r"""
    new Promise((resolve) => {

        const previous = document.getElementById("football-labeler");
        if (previous) previous.remove();

        const root = document.createElement("div");
        root.id = "football-labeler";

        root.style.cssText = `
            width: 100%;
            max-width: 1100px;
            margin: 0 auto;
            box-sizing: border-box;
            background: #151e2c;
            color: white;
            font-family: Arial, sans-serif;
            border: 1px solid #41506b;
            border-radius: 12px;
            padding: 12px;
        `;

        root.innerHTML = `
            <div style="display:flex;justify-content:space-between;
                        align-items:center;margin-bottom:8px;">
                <strong style="font-size:17px;">
                    Ball Annotation - Frame __FRAME__
                </strong>
                <span style="font-size:12px;color:#b9c4d4;">
                    Click ball center to save
                </span>
            </div>

            <div style="
                background:#050505;
                border-radius:8px;
                overflow:hidden;
                text-align:center;
            ">
                <img id="football-image" style="
                    display:block;
                    width:100%;
                    height:auto;
                    max-height:420px;
                    object-fit:contain;
                    cursor:crosshair;
                ">
            </div>

            <div style="
                display:flex;
                gap:12px;
                flex-wrap:wrap;
                align-items:center;
                margin-top:10px;
            ">
                <canvas id="football-zoom"
                    width="220" height="220"
                    style="
                        width:180px;
                        height:180px;
                        border:1px solid #64748b;
                        border-radius:6px;
                        background:#111;
                    ">
                </canvas>

                <div style="flex:1;min-width:180px;">
                    <div id="football-coordinate"
                         style="font-size:14px;margin-bottom:12px;">
                        Hover over image to magnify
                    </div>

                    <button id="football-absent" style="
                        padding:11px 15px;
                        margin-right:8px;
                        margin-bottom:8px;
                        background:#475569;
                        color:white;
                        border:0;
                        border-radius:7px;
                        cursor:pointer;
                    ">Ball not visible</button>

                    <button id="football-uncertain" style="
                        padding:11px 15px;
                        margin-bottom:8px;
                        background:#854d0e;
                        color:white;
                        border:0;
                        border-radius:7px;
                        cursor:pointer;
                    ">Cannot determine</button>
                </div>
            </div>
        `;

        document.body.appendChild(root);

        const img = root.querySelector("#football-image");
        const canvas = root.querySelector("#football-zoom");
        const ctx = canvas.getContext("2d");
        const info = root.querySelector("#football-coordinate");

        function finish(result) {
            root.remove();
            resolve(result);
        }

        function getPosition(event) {
            const rect = img.getBoundingClientRect();

            // Account for object-fit:contain letterboxing
            const scale = Math.min(
                rect.width / img.naturalWidth,
                rect.height / img.naturalHeight
            );

            const displayW = img.naturalWidth * scale;
            const displayH = img.naturalHeight * scale;

            const offsetX = (rect.width - displayW) / 2;
            const offsetY = (rect.height - displayH) / 2;

            const x = (event.clientX - rect.left - offsetX) / scale;
            const y = (event.clientY - rect.top - offsetY) / scale;

            return {
                x: Math.max(0, Math.min(img.naturalWidth-1, x)),
                y: Math.max(0, Math.min(img.naturalHeight-1, y))
            };
        }

        img.addEventListener("mousemove", (event) => {
            const p = getPosition(event);

            info.textContent =
                "Original coordinates: X=" + p.x.toFixed(1)
                + "  Y=" + p.y.toFixed(1);

            const zoomSize = 75;

            const sx = Math.max(0, Math.min(
                img.naturalWidth-zoomSize,
                p.x-zoomSize/2
            ));

            const sy = Math.max(0, Math.min(
                img.naturalHeight-zoomSize,
                p.y-zoomSize/2
            ));

            ctx.clearRect(0,0,220,220);

            ctx.drawImage(
                img,
                sx,sy,zoomSize,zoomSize,
                0,0,220,220
            );

            const zx = (p.x-sx)*220/zoomSize;
            const zy = (p.y-sy)*220/zoomSize;

            ctx.strokeStyle = "red";
            ctx.lineWidth = 2;

            ctx.beginPath();
            ctx.moveTo(zx-12,zy);
            ctx.lineTo(zx+12,zy);
            ctx.moveTo(zx,zy-12);
            ctx.lineTo(zx,zy+12);
            ctx.stroke();
        });

        img.addEventListener("click", (event) => {
            const p = getPosition(event);

            finish({
                kind: "visible",
                x: Math.round(p.x),
                y: Math.round(p.y)
            });
        });

        root.querySelector("#football-absent").onclick = () => {
            finish({kind:"not_visible"});
        };

        root.querySelector("#football-uncertain").onclick = () => {
            finish({kind:"uncertain"});
        };

        img.src = __IMAGE_DATA__;
    })
    """

    js = js.replace("__FRAME__", str(frame_number))
    js = js.replace("__IMAGE_DATA__", json.dumps(image_data))

    return eval_js(js)

print("Compact annotation interface ready.")


In [ ]:

from google.colab import drive
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import shutil

drive.mount("/content/drive")

BENCHMARK_DIR = Path(
    "/content/drive/MyDrive/Football_CV/ball_benchmark_v1"
)

IMAGES_DIR = BENCHMARK_DIR / "images"
TEMPLATE_PATH = BENCHMARK_DIR / "annotation_template.csv"
GROUND_TRUTH_PATH = BENCHMARK_DIR / "ball_ground_truth_v1.csv"

assert TEMPLATE_PATH.exists(), "Annotation template not found"
assert IMAGES_DIR.exists(), "Benchmark images not found"

# Back up your existing annotations before resetting
if GROUND_TRUTH_PATH.exists():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    backup_path = (
        BENCHMARK_DIR /
        f"ball_ground_truth_backup_{timestamp}.csv"
    )

    shutil.copy2(GROUND_TRUTH_PATH, backup_path)

    print("Previous annotations backed up:")
    print(backup_path)

# Reset all 40 frames
annotations = pd.read_csv(TEMPLATE_PATH).copy()

annotations["gt_ball_visible"] = np.nan
annotations["gt_ball_x"] = np.nan
annotations["gt_ball_y"] = np.nan
annotations["annotation_status"] = "pending"

annotations = annotations.sort_values(
    "frame"
).reset_index(drop=True)

assert len(annotations) == 40

# Overwrite the working annotation CSV
annotations.to_csv(GROUND_TRUTH_PATH, index=False)

print("\nANNOTATIONS RESET SUCCESSFULLY")
print("Total frames:", len(annotations))
print("Completed:", 0)
print("Remaining:", len(annotations))
print("First frame:", annotations.frame.iloc[0])
print("Last frame:", annotations.frame.iloc[-1])

display(annotations.head())


In [ ]:
assert "annotate_ball" in globals(), (
    "Annotation interface not loaded. "
    "Run the compact annotate_ball() function cell first."
)

print("Annotation interface ready.")
print("First frame:", annotations.iloc[0]["frame"])

In [ ]:

import pandas as pd
import numpy as np

# Read the reset file
annotations = pd.read_csv(GROUND_TRUTH_PATH)

annotations["annotation_status"] = (
    annotations["annotation_status"]
    .fillna("pending")
)

# Set None to annotate all 40 images
MAX_NEW = None

completed_now = 0

for idx, row in annotations.iterrows():

    if row["annotation_status"] != "pending":
        continue

    frame_idx = int(row["frame"])
    image_path = IMAGES_DIR / row["filename"]

    if not image_path.exists():
        print("Image missing:", image_path)
        continue

    print(f"\nAnnotating Frame {frame_idx}")

    result = annotate_ball(image_path, frame_idx)
    kind = result["kind"]

    if kind == "visible":

        annotations.loc[idx, "gt_ball_visible"] = 1
        annotations.loc[idx, "gt_ball_x"] = float(result["x"])
        annotations.loc[idx, "gt_ball_y"] = float(result["y"])
        annotations.loc[idx, "annotation_status"] = "annotated"

    elif kind == "not_visible":

        annotations.loc[idx, "gt_ball_visible"] = 0
        annotations.loc[idx, ["gt_ball_x", "gt_ball_y"]] = np.nan
        annotations.loc[idx, "annotation_status"] = "not_visible"

    elif kind == "uncertain":

        annotations.loc[idx, "gt_ball_visible"] = np.nan
        annotations.loc[idx, ["gt_ball_x", "gt_ball_y"]] = np.nan
        annotations.loc[idx, "annotation_status"] = "uncertain"

    else:
        print("Unexpected annotation result:", result)
        break

    # Immediately save each annotation
    annotations.to_csv(GROUND_TRUTH_PATH, index=False)

    completed_now += 1

    remaining = int(
        (annotations["annotation_status"] == "pending").sum()
    )

    print(
        f"Saved frame {frame_idx} | "
        f"{kind} | Remaining: {remaining}"
    )

    if MAX_NEW is not None and completed_now >= MAX_NEW:
        break

print("\nANNOTATION SUMMARY")
print(annotations.annotation_status.value_counts())
print("\nSaved:", GROUND_TRUTH_PATH)


In [ ]:
progress = pd.read_csv(GROUND_TRUTH_PATH)

print("ANNOTATION STATUS")
print(progress.annotation_status.value_counts())

print("\nCompleted:",
      (progress.annotation_status != "pending").sum())

print("Remaining:",
      (progress.annotation_status == "pending").sum())

display(
    progress[
        ["frame", "gt_ball_visible",
         "gt_ball_x", "gt_ball_y", "annotation_status"]
    ].head(15)
)

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/Football_CV")
BENCHMARK_DIR = ROOT / "ball_benchmark_v1"

GT_PATH = BENCHMARK_DIR / "ball_ground_truth_v1.csv"

PRED_PATH = (
    ROOT / "ball_tracking_consolidated"
    / "segment_C_ball_tracking_verified_recovery_v1.csv"
)

assert GT_PATH.exists(), "Ground truth CSV missing"
assert PRED_PATH.exists(), "Tracking predictions CSV missing"

gt = pd.read_csv(GT_PATH)
pred = pd.read_csv(PRED_PATH)

assert len(gt) == 40
assert gt.frame.is_unique
assert pred.frame.is_unique

evaluation = gt.merge(
    pred[
        [
            "frame", "ball_x", "ball_y",
            "confidence", "status", "track_segment_id"
        ]
    ],
    on="frame",
    how="left"
)

evaluation["valid_gt"] = (
    (evaluation.annotation_status == "annotated")
    & (evaluation.gt_ball_visible == 1)
    & evaluation.gt_ball_x.notna()
    & evaluation.gt_ball_y.notna()
)

evaluation["has_prediction"] = (
    evaluation.ball_x.notna()
    & evaluation.ball_y.notna()
    & evaluation.status.isin(
        ["observed", "predicted", "interpolated"]
    )
)

evaluation["error_px"] = np.hypot(
    evaluation.ball_x - evaluation.gt_ball_x,
    evaluation.ball_y - evaluation.gt_ball_y
)

evaluation.loc[
    ~evaluation.valid_gt | ~evaluation.has_prediction,
    "error_px"
] = np.nan

print("Total benchmark frames:", len(evaluation))
print("Frames with valid GT:", evaluation.valid_gt.sum())
print("Uncertain GT frames:", (~evaluation.valid_gt).sum())

display(
    evaluation[
        [
            "frame", "gt_ball_x", "gt_ball_y",
            "ball_x", "ball_y",
            "status", "error_px"
        ]
    ].head(15)
)


In [ ]:

valid = evaluation[evaluation.valid_gt].copy()

matched = valid[valid.has_prediction].copy()

total_gt = len(valid)
total_matched = len(matched)

mean_error = (
    matched.error_px.mean()
    if total_matched else np.nan
)

median_error = (
    matched.error_px.median()
    if total_matched else np.nan
)

rmse = (
    np.sqrt(np.mean(matched.error_px ** 2))
    if total_matched else np.nan
)

summary = {
    "Benchmark frames": len(evaluation),
    "Valid ground-truth frames": total_gt,
    "Frames with tracking positions": total_matched,
    "Position coverage (%)": (
        100 * total_matched / total_gt
        if total_gt else np.nan
    ),
    "Mean center error (px)": mean_error,
    "Median center error (px)": median_error,
    "RMSE (px)": rmse,
    "Within 10px (%)": (
        100 * (valid.error_px <= 10).sum() / total_gt
        if total_gt else np.nan
    ),
    "Within 20px (%)": (
        100 * (valid.error_px <= 20).sum() / total_gt
        if total_gt else np.nan
    ),
    "Within 30px (%)": (
        100 * (valid.error_px <= 30).sum() / total_gt
        if total_gt else np.nan
    )
}

metrics_df = pd.DataFrame(
    list(summary.items()),
    columns=["Metric", "Value"]
)

print("BALL TRACKING BENCHMARK RESULTS")
display(metrics_df.round(3))


In [ ]:

status_metrics = (
    matched.groupby("status")
    .agg(
        frames=("frame", "size"),
        mean_error_px=("error_px", "mean"),
        median_error_px=("error_px", "median"),
        max_error_px=("error_px", "max")
    )
    .reset_index()
)

print("Error by tracking status:")
display(status_metrics.round(2))

print("\nLargest localization errors:")

display(
    matched.sort_values(
        "error_px", ascending=False
    )[
        [
            "frame",
            "status",
            "gt_ball_x", "gt_ball_y",
            "ball_x", "ball_y",
            "error_px"
        ]
    ].head(12).round(2)
)

print("\nFrames with GT but no prediction:")

display(
    valid[~valid.has_prediction][
        ["frame", "annotation_status", "status"]
    ]
)


In [ ]:

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 5))

for status, group in matched.groupby("status"):
    ax.scatter(
        group.frame,
        group.error_px,
        s=55,
        label=status
    )

ax.axhline(
    20,
    linestyle="--",
    color="gray",
    label="20px tolerance"
)

ax.set_xlabel("Video frame")
ax.set_ylabel("Center localization error (pixels)")
ax.set_title("Segment C - Ball Tracking Error Against Ground Truth")
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load files
ROOT = Path("/content/drive/MyDrive/Football_CV")
BENCHMARK_DIR = ROOT / "ball_benchmark_v1"

GT_PATH = BENCHMARK_DIR / "ball_ground_truth_v1.csv"

PRED_PATH = (
    ROOT / "ball_tracking_consolidated"
    / "segment_C_ball_tracking_verified_recovery_v1.csv"
)

assert GT_PATH.exists(), f"Missing ground truth: {GT_PATH}"
assert PRED_PATH.exists(), f"Missing predictions: {PRED_PATH}"

gt = pd.read_csv(GT_PATH)
pred = pd.read_csv(PRED_PATH)

assert len(gt) == 40
assert gt.frame.is_unique
assert pred.frame.is_unique

# 2. Merge annotations with predictions
evaluation = gt.merge(
    pred[
        [
            "frame", "ball_x", "ball_y",
            "confidence", "status", "track_segment_id"
        ]
    ],
    on="frame",
    how="left"
)

evaluation["valid_gt"] = (
    (evaluation["annotation_status"] == "annotated")
    & (evaluation["gt_ball_visible"] == 1)
    & evaluation["gt_ball_x"].notna()
    & evaluation["gt_ball_y"].notna()
)

evaluation["has_prediction"] = (
    evaluation["ball_x"].notna()
    & evaluation["ball_y"].notna()
    & evaluation["status"].isin(
        ["observed", "predicted", "interpolated"]
    )
)

evaluation["error_px"] = np.hypot(
    evaluation["ball_x"] - evaluation["gt_ball_x"],
    evaluation["ball_y"] - evaluation["gt_ball_y"]
)

evaluation.loc[
    ~evaluation["valid_gt"] | ~evaluation["has_prediction"],
    "error_px"
] = np.nan

valid = evaluation[evaluation["valid_gt"]].copy()
matched = valid[valid["has_prediction"]].copy()

n_valid = len(valid)
n_matched = len(matched)

# 3. Overall metrics
def success_rate(tolerance):
    if n_valid == 0:
        return np.nan
    return 100 * (valid["error_px"] <= tolerance).sum() / n_valid

metrics = {
    "Total benchmark frames": len(evaluation),
    "Valid ground truth": n_valid,
    "Uncertain or excluded": len(evaluation) - n_valid,
    "Frames with predictions": n_matched,
    "Position coverage (%)": (
        100 * n_matched / n_valid if n_valid else np.nan
    ),
    "Mean center error (px)": (
        matched["error_px"].mean() if n_matched else np.nan
    ),
    "Median center error (px)": (
        matched["error_px"].median() if n_matched else np.nan
    ),
    "RMSE (px)": (
        np.sqrt(np.mean(matched["error_px"] ** 2))
        if n_matched else np.nan
    ),
    "Within 10 px (%)": success_rate(10),
    "Within 20 px (%)": success_rate(20),
    "Within 30 px (%)": success_rate(30)
}

metrics_df = pd.DataFrame(
    metrics.items(), columns=["Metric", "Value"]
)

print("\nBALL TRACKING BENCHMARK RESULTS")
print("=" * 55)
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# 4. Error by prediction type
status_metrics = (
    matched.groupby("status")
    .agg(
        frames=("frame", "size"),
        mean_error_px=("error_px", "mean"),
        median_error_px=("error_px", "median"),
        max_error_px=("error_px", "max")
    )
    .reset_index()
)

print("\nERROR BY TRACKING STATUS")
display(status_metrics.round(2))

# 5. Largest errors
print("\nLARGEST LOCALIZATION ERRORS")
largest_errors = (
    matched.sort_values("error_px", ascending=False)
    [
        [
            "frame", "status",
            "gt_ball_x", "gt_ball_y",
            "ball_x", "ball_y",
            "error_px"
        ]
    ]
    .head(12)
)

display(largest_errors.round(2))

# 6. Missing predictions
print("\nGROUND-TRUTH FRAMES WITHOUT TRACKING POSITIONS")
missing_predictions = valid[~valid["has_prediction"]]
display(missing_predictions[["frame", "status"]])

# 7. Plot errors
fig, ax = plt.subplots(figsize=(14, 5))

for status, group in matched.groupby("status"):
    ax.scatter(
        group["frame"],
        group["error_px"],
        s=55,
        label=status
    )

ax.axhline(
    20,
    linestyle="--",
    color="gray",
    label="20 px tolerance"
)

ax.set_xlabel("Video frame")
ax.set_ylabel("Center localization error (pixels)")
ax.set_title("Segment C Ball Tracking vs Ground Truth")
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

# 8. Save all results
OUT_DIR = BENCHMARK_DIR / "evaluation_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

evaluation.to_csv(
    OUT_DIR / "frame_level_tracking_evaluation.csv",
    index=False
)

metrics_df.to_csv(
    OUT_DIR / "tracking_metrics.csv",
    index=False
)

status_metrics.to_csv(
    OUT_DIR / "status_error_metrics.csv",
    index=False
)

fig.savefig(
    OUT_DIR / "localization_error_plot.png",
    dpi=180,
    bbox_inches="tight"
)

print("\nEvaluation completed successfully.")
print("Results saved to:", OUT_DIR)

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Football_CV")

GT_PATH = ROOT / "ball_benchmark_v1/ball_ground_truth_v1.csv"
PRED_PATH = ROOT / "ball_tracking_consolidated/segment_C_ball_tracking_verified_recovery_v1.csv"
CAND_PATH = ROOT / "ball_tracking_v2/segment_C_ball_candidates_allframes.csv"

gt = pd.read_csv(GT_PATH)
pred = pd.read_csv(PRED_PATH)
candidates = pd.read_csv(CAND_PATH)

valid_gt = gt[
    (gt.annotation_status == "annotated") &
    (gt.gt_ball_visible == 1)
].copy()

rows = []

for label in valid_gt.itertuples(index=False):
    frame = int(label.frame)
    gx = float(label.gt_ball_x)
    gy = float(label.gt_ball_y)

    pool = candidates[candidates.frame == frame].copy()

    tracker_row = pred.loc[pred.frame == frame].iloc[0]
    tracker_found = (
        pd.notna(tracker_row.ball_x) and
        pd.notna(tracker_row.ball_y) and
        tracker_row.status in ["observed", "predicted", "interpolated"]
    )

    if pool.empty:
        rows.append({
            "frame": frame,
            "tracker_found": tracker_found,
            "candidate_count": 0,
            "nearest_distance_px": np.nan,
            "nearest_confidence": np.nan,
            "nearest_x": np.nan,
            "nearest_y": np.nan
        })
        continue

    pool["distance_px"] = np.hypot(
        pool.center_x - gx,
        pool.center_y - gy
    )

    closest = pool.loc[pool.distance_px.idxmin()]

    rows.append({
        "frame": frame,
        "tracker_found": tracker_found,
        "candidate_count": len(pool),
        "nearest_distance_px": float(closest.distance_px),
        "nearest_confidence": float(closest.confidence),
        "nearest_x": float(closest.center_x),
        "nearest_y": float(closest.center_y)
    })

candidate_audit = pd.DataFrame(rows)

candidate_audit["yolo_within_10px"] = (
    candidate_audit.nearest_distance_px <= 10
)

candidate_audit["yolo_within_20px"] = (
    candidate_audit.nearest_distance_px <= 20
)

candidate_audit["yolo_within_30px"] = (
    candidate_audit.nearest_distance_px <= 30
)

print("RAW YOLO CANDIDATE RECALL")
print("=" * 50)

for tolerance in [10, 20, 30]:
    recovered = (
        candidate_audit.nearest_distance_px <= tolerance
    ).sum()

    print(
        f"Within {tolerance:2d} px: "
        f"{recovered}/{len(candidate_audit)} "
        f"({100*recovered/len(candidate_audit):.2f}%)"
    )

print("\nMISSED BY TRACKER - CANDIDATE DIAGNOSTICS")

missed = candidate_audit[
    ~candidate_audit.tracker_found
].copy()

display(
    missed.sort_values("frame").round(3)
)

print("\nTracker misses with YOLO candidate within 20 px:",
      int(missed.yolo_within_20px.sum()))

print("Tracker misses without YOLO candidate within 20 px:",
      int((~missed.yolo_within_20px).sum()))


In [ ]:

def failure_type(row):
    if row.tracker_found:
        return "Tracker has position"
    if row.candidate_count == 0:
        return "No YOLO candidates"
    if row.nearest_distance_px <= 20:
        return "YOLO candidate available - tracker missed"
    return "YOLO candidates present - ball not localized"

candidate_audit["failure_type"] = candidate_audit.apply(
    failure_type, axis=1
)

print("FAILURE ANALYSIS")
display(
    candidate_audit.failure_type.value_counts()
    .rename_axis("Failure type")
    .reset_index(name="Frames")
)

print("\nEARLY SEGMENT DIAGNOSTICS")
display(
    candidate_audit[
        candidate_audit.frame.between(4000, 4084)
    ].round(3)
)

OUT_DIR = ROOT / "ball_benchmark_v1/evaluation_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_audit.to_csv(
    OUT_DIR / "raw_yolo_candidate_audit.csv",
    index=False
)

print("\nSaved raw candidate audit.")


In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile
import shutil
import torch

drive.mount("/content/drive")

# Install Ultralytics if necessary
try:
    from ultralytics import YOLO
except ImportError:
    %pip install -q ultralytics
    from ultralytics import YOLO

ROOT = Path("/content/drive/MyDrive/Football_CV")

MODEL_DIR = Path("/content/football_models/ball_detector_v1")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL = MODEL_DIR / "best.pt"

# Look for a previously backed-up ball detector on Drive
if not BEST_MODEL.exists():

    model_candidates = [
        p for p in ROOT.rglob("*.pt")
        if (
            p.name.lower() in {
                "ball_detector_v1_best.pt",
                "best.pt"
            }
            and "ball" in str(p).lower()
        )
    ]

    if model_candidates:
        source = model_candidates[0]
        shutil.copy2(source, BEST_MODEL)
        print("Restored from Google Drive:", source)

    else:
        # Check for an existing model ZIP in Colab or Drive
        zip_candidates = (
            list(Path("/content").glob("football_ball_model_v1.zip"))
            + list(ROOT.rglob("football_ball_model_v1.zip"))
        )

        if zip_candidates:
            zip_path = zip_candidates[0]
        else:
            print("Upload football_ball_model_v1.zip")
            uploaded = files.upload()
            zip_path = Path(
                next(name for name in uploaded if name.endswith(".zip"))
            )

        with zipfile.ZipFile(zip_path, "r") as z:
            best_matches = [
                name for name in z.namelist()
                if Path(name).name.lower() in {
                    "best.pt",
                    "ball_detector_v1_best.pt"
                }
            ]

            assert best_matches, "Ball model checkpoint not found in ZIP"

            BEST_MODEL.write_bytes(z.read(best_matches[0]))

        print("Restored from ZIP:", zip_path)

# Load trained model into memory
assert BEST_MODEL.exists(), "Model restoration failed"

ball_model = YOLO(str(BEST_MODEL))

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("\nBALL DETECTOR RESTORED")
print("Model path:", BEST_MODEL)
print("Model size:", round(BEST_MODEL.stat().st_size / 1e6, 2), "MB")
print("Classes:", ball_model.names)
print("Device:", DEVICE)

assert ball_model.names == {0: "ball"}
print("\nModel ready for evaluation!")

In [ ]:
import cv2
import matplotlib.pyplot as plt

TEST_IMAGE = (
    ROOT / "ball_benchmark_v1"
    / "images" / "frame_04072.jpg"
)

assert TEST_IMAGE.exists(), "Benchmark image not found"

image = cv2.imread(str(TEST_IMAGE))

results = ball_model.predict(
    source=image,
    imgsz=1280,
    conf=0.005,
    device=DEVICE,
    verbose=False
)

print("Model loaded successfully.")
print("Detections:", len(results[0].boxes))
print("Model classes:", ball_model.names)

plt.figure(figsize=(15, 5))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("SoccerTrack Frame 4072 — Model restored")
plt.axis("off")
plt.show()

In [ ]:

import cv2
import torch
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Football_CV")
BENCHMARK_DIR = ROOT / "ball_benchmark_v1"
IMAGES_DIR = BENCHMARK_DIR / "images"
OUT_DIR = BENCHMARK_DIR / "evaluation_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert "ball_model" in globals(), "Restore ball_model first."

DEVICE = 0 if torch.cuda.is_available() else "cpu"

gt = pd.read_csv(BENCHMARK_DIR / "ball_ground_truth_v1.csv")
pred = pd.read_csv(
    ROOT / "ball_tracking_consolidated" /
    "segment_C_ball_tracking_verified_recovery_v1.csv"
)

valid = gt[
    (gt.annotation_status == "annotated") &
    (gt.gt_ball_visible == 1)
].merge(
    pred[["frame", "ball_x", "ball_y", "status"]],
    on="frame",
    how="left"
)

missing = valid[
    valid.ball_x.isna() | valid.ball_y.isna()
].copy()

print("Frames to test:", len(missing))

CROP_RADIUS = 160
results = []
all_local_candidates = []

for row in missing.itertuples(index=False):
    frame_idx = int(row.frame)
    image_path = IMAGES_DIR / row.filename

    frame = cv2.imread(str(image_path))
    assert frame is not None, f"Missing image {image_path}"

    H, W = frame.shape[:2]
    gx, gy = float(row.gt_ball_x), float(row.gt_ball_y)

    x1 = max(0, int(gx - CROP_RADIUS))
    x2 = min(W, int(gx + CROP_RADIUS))
    y1 = max(0, int(gy - CROP_RADIUS))
    y2 = min(H, int(gy + CROP_RADIUS))

    crop = frame[y1:y2, x1:x2]

    prediction = ball_model.predict(
        source=crop,
        imgsz=1280,
        conf=0.005,
        device=DEVICE,
        verbose=False
    )[0]

    candidate_rows = []

    for det in prediction.boxes:
        b = det.xyxy[0].cpu().numpy()

        cx = float((b[0] + b[2]) / 2 + x1)
        cy = float((b[1] + b[3]) / 2 + y1)
        confidence = float(det.conf[0])
        distance = float(np.hypot(cx - gx, cy - gy))

        candidate = {
            "frame": frame_idx,
            "center_x": cx,
            "center_y": cy,
            "confidence": confidence,
            "width": float(b[2] - b[0]),
            "height": float(b[3] - b[1]),
            "distance_to_gt": distance
        }

        candidate_rows.append(candidate)
        all_local_candidates.append(candidate)

    nearest = (
        min(candidate_rows, key=lambda c: c["distance_to_gt"])
        if candidate_rows else None
    )

    results.append({
        "frame": frame_idx,
        "gt_x": gx,
        "gt_y": gy,
        "num_candidates": len(candidate_rows),
        "nearest_distance_px": (
            nearest["distance_to_gt"] if nearest else np.nan
        ),
        "nearest_confidence": (
            nearest["confidence"] if nearest else np.nan
        ),
        "recovered_10px": (
            nearest is not None and
            nearest["distance_to_gt"] <= 10
        ),
        "recovered_20px": (
            nearest is not None and
            nearest["distance_to_gt"] <= 20
        )
    })

local_audit = pd.DataFrame(results)
local_candidates = pd.DataFrame(all_local_candidates)

local_audit.to_csv(
    OUT_DIR / "local_oracle_detection_audit.csv",
    index=False
)
local_candidates.to_csv(
    OUT_DIR / "local_oracle_candidates.csv",
    index=False
)

print("\nLOCAL DETECTOR RESULTS")
print("Total missed frames:", len(local_audit))
print("Recovered within 10px:", int(local_audit.recovered_10px.sum()))
print("Recovered within 20px:", int(local_audit.recovered_20px.sum()))

display(local_audit.round(3))


In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Football_CV")
BENCHMARK_DIR = ROOT / "ball_benchmark_v1"
IMAGES_DIR = BENCHMARK_DIR / "images"

# Restore saved diagnostics if notebook variables are missing
if "local_audit" not in globals():
    local_audit = pd.read_csv(
        BENCHMARK_DIR / "evaluation_v2" /
        "local_oracle_detection_audit.csv"
    )

if "local_candidates" not in globals():
    local_candidates = pd.read_csv(
        BENCHMARK_DIR / "evaluation_v2" /
        "local_oracle_candidates.csv"
    )

# Only frames not recovered within 10 pixels
failed = local_audit[
    ~local_audit["recovered_10px"].fillna(False).astype(bool)
].sort_values("frame").copy()

print("Unrecovered frames:", len(failed))
print("Frame numbers:", failed.frame.tolist())

columns = 4
rows = int(np.ceil(len(failed) / columns))

fig, axes = plt.subplots(
    rows, columns,
    figsize=(20, rows * 4.5)
)

axes = np.asarray(axes).reshape(-1)

for ax in axes:
    ax.axis("off")

for ax, record in zip(axes, failed.itertuples(index=False)):

    frame_idx = int(record.frame)
    image_path = IMAGES_DIR / f"frame_{frame_idx:05d}.jpg"

    image = cv2.imread(str(image_path))
    if image is None:
        ax.set_title(f"F{frame_idx}: image missing")
        continue

    H, W = image.shape[:2]

    gx = float(record.gt_x)
    gy = float(record.gt_y)

    radius = 130

    x1 = max(0, int(gx - radius))
    x2 = min(W, int(gx + radius))
    y1 = max(0, int(gy - radius))
    y2 = min(H, int(gy + radius))

    crop = image[y1:y2, x1:x2].copy()

    # Green cross: manual ground-truth coordinate
    cv2.drawMarker(
        crop,
        (int(gx - x1), int(gy - y1)),
        (0, 255, 0),
        cv2.MARKER_CROSS,
        24, 2
    )

    # Red circles: local YOLO candidates inside this crop
    pool = local_candidates[
        local_candidates.frame == frame_idx
    ]

    for det in pool.itertuples(index=False):
        px = int(round(det.center_x - x1))
        py = int(round(det.center_y - y1))

        if 0 <= px < crop.shape[1] and 0 <= py < crop.shape[0]:
            cv2.circle(
                crop, (px, py),
                12, (0, 0, 255), 2
            )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"F{frame_idx} | "
        f"{int(record.num_candidates)} candidates | "
        f"nearest {record.nearest_distance_px:.1f}px",
        fontsize=10
    )

plt.suptitle(
    "Unrecovered Footballs - Ground Truth vs Local YOLO",
    fontsize=16
)

plt.tight_layout()
plt.show()


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

AIRBORNE_FRAMES = [4036, 4048, 4060]

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

for ax, frame_idx in zip(axes, AIRBORNE_FRAMES):

    row = gt[gt.frame == frame_idx].iloc[0]

    image_path = (
        BENCHMARK_DIR / "images"
        / f"frame_{frame_idx:05d}.jpg"
    )

    image = cv2.imread(str(image_path))
    assert image is not None

    gx = int(row.gt_ball_x)
    gy = int(row.gt_ball_y)

    radius = 65

    H, W = image.shape[:2]

    x1 = max(0, gx - radius)
    x2 = min(W, gx + radius)
    y1 = max(0, gy - radius)
    y2 = min(H, gy + radius)

    crop = image[y1:y2, x1:x2].copy()

    cv2.drawMarker(
        crop,
        (gx-x1, gy-y1),
        (0, 255, 0),
        cv2.MARKER_CROSS,
        18,
        1
    )

    ax.imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB),
        interpolation="nearest"
    )

    ax.set_title(f"Frame {frame_idx}")
    ax.axis("off")

plt.suptitle("Airborne Football — Original Pixel Detail")
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
from pathlib import Path
import cv2
import shutil

drive.mount("/content/drive")

VIDEO_PATH = Path(
    "/content/football_dataset/mot/clips/117093.mp4"
)

search_roots = [
    Path("/content"),
    Path("/content/drive/MyDrive/Football_CV"),
    Path("/content/drive/MyDrive")
]

found_videos = []
seen = set()

for root in search_roots:
    if not root.exists():
        continue

    for p in root.rglob("117093.mp4"):
        if str(p) not in seen:
            found_videos.append(p)
            seen.add(str(p))

print("Videos found:", len(found_videos))

for i, p in enumerate(found_videos):
    print(i, p)
    print("Size:", round(p.stat().st_size / 1e6, 2), "MB")

if found_videos:
    source = found_videos[0]

    VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)

    if source.resolve() != VIDEO_PATH.resolve():
        shutil.copy2(source, VIDEO_PATH)

    cap = cv2.VideoCapture(str(VIDEO_PATH))

    print("\nVideo opens:", cap.isOpened())

    if cap.isOpened():
        print("Total frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
        print("FPS:", cap.get(cv2.CAP_PROP_FPS))
        print("Resolution:",
              int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
              "x",
              int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

    cap.release()
else:
    print("Video not found locally or in Google Drive.")

In [ ]:
from huggingface_hub import login
login()

In [ ]:
%pip -q install huggingface_hub

from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil

VIDEO_PATH = Path(
    "/content/football_dataset/mot/clips/117093.mp4"
)
VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)

source = hf_hub_download(
    repo_id="atomscott/soccertrack-v2",
    filename="mot/clips/117093.mp4",
    repo_type="dataset"
)

shutil.copy2(source, VIDEO_PATH)

print("Video downloaded:", VIDEO_PATH)
print("Size:", round(VIDEO_PATH.stat().st_size / 1e6, 2), "MB")

In [ ]:
import cv2

cap = cv2.VideoCapture(str(VIDEO_PATH))

assert cap.isOpened(), "Video restoration failed"

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

assert frame_count > 4063, "Incorrect or incomplete video"

cap.set(cv2.CAP_PROP_POS_FRAMES, 4036)
ok, frame = cap.read()

assert ok, "Could not read frame 4036"

print("Video restored successfully!")
print("Frame count:", frame_count)
print("FPS:", fps)
print("Frame shape:", frame.shape)

cap.release()

In [ ]:
from pathlib import Path
import shutil

BACKUP_PATH = Path(
    "/content/drive/MyDrive/Football_CV/videos/117093.mp4"
)

BACKUP_PATH.parent.mkdir(parents=True, exist_ok=True)

if not BACKUP_PATH.exists():
    shutil.copy2(VIDEO_PATH, BACKUP_PATH)
    print("Video backed up:", BACKUP_PATH)
else:
    print("Video backup already exists:", BACKUP_PATH)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

GT = pd.read_csv(
    "/content/drive/MyDrive/Football_CV/ball_benchmark_v1/ball_ground_truth_v1.csv"
)

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Restore the SoccerTrack video first"

ANCHORS = [4036, 4048, 4060]
OFFSETS = [-3, -2, -1, 0, 1, 2, 3]
RADIUS = 85

fig, axes = plt.subplots(
    len(ANCHORS), len(OFFSETS),
    figsize=(20, 9)
)

for row_index, anchor in enumerate(ANCHORS):
    row = GT[GT.frame == anchor].iloc[0]
    gx, gy = int(row.gt_ball_x), int(row.gt_ball_y)

    for col_index, offset in enumerate(OFFSETS):
        frame_idx = anchor + offset

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, image = cap.read()

        ax = axes[row_index, col_index]
        ax.axis("off")

        if not ok:
            continue

        h, w = image.shape[:2]

        x1 = max(0, gx - RADIUS)
        x2 = min(w, gx + RADIUS)
        y1 = max(0, gy - RADIUS)
        y2 = min(h, gy + RADIUS)

        crop = image[y1:y2, x1:x2]

        ax.imshow(
            cv2.cvtColor(crop, cv2.COLOR_BGR2RGB),
            interpolation="nearest"
        )

        # The marker represents the anchor annotation only.
        if offset == 0:
            ax.scatter(
                gx - x1, gy - y1,
                marker="+", color="lime", s=90
            )

        ax.set_title(f"F{frame_idx}", fontsize=10)

    axes[row_index, 0].set_ylabel(
        f"Anchor {anchor}", fontsize=11
    )

cap.release()

plt.suptitle(
    "Airborne Ball Temporal Inspection",
    fontsize=15
)
plt.tight_layout()
plt.show()

In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Football_CV")
GT_PATH = ROOT / "ball_benchmark_v1/ball_ground_truth_v1.csv"

gt = pd.read_csv(GT_PATH)

VIDEO_PATH = Path(
    "/content/football_dataset/mot/clips/117093.mp4"
)

assert VIDEO_PATH.exists(), "Restore 117093.mp4 first"

# Region containing the airborne trajectory.
X1, Y1, X2, Y2 = 1900, 80, 3200, 370

START_FRAME = 4028
END_FRAME = 4065

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened()

# Construct a static background estimate from the sequence.
background_samples = []

for f in range(START_FRAME, END_FRAME + 1, 4):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, frame = cap.read()

    if ok:
        gray = cv2.cvtColor(
            frame[Y1:Y2, X1:X2],
            cv2.COLOR_BGR2GRAY
        )
        background_samples.append(gray)

assert len(background_samples) >= 5

background = np.median(
    np.stack(background_samples),
    axis=0
).astype(np.uint8)

motion_records = []
debug_images = {}

for f in range(START_FRAME, END_FRAME + 1):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, frame = cap.read()

    if not ok:
        continue

    crop = frame[Y1:Y2, X1:X2].copy()
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    difference = cv2.absdiff(gray, background)

    # Initial experimental motion threshold.
    _, mask = cv2.threshold(
        difference, 35, 255, cv2.THRESH_BINARY
    )

    # Remove isolated single-pixel noise.
    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        np.ones((2, 2), np.uint8)
    )

    count, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(mask, 8)
    )

    for component in range(1, count):
        x, y, w, h, area = stats[component]

        # Small moving objects only.
        if not (3 <= area <= 250):
            continue

        if w > 25 or h > 25:
            continue

        cx, cy = centroids[component]

        motion_records.append({
            "frame": f,
            "center_x": float(cx + X1),
            "center_y": float(cy + Y1),
            "width": int(w),
            "height": int(h),
            "area": int(area)
        })

    if f in [4036, 4048, 4060]:
        debug_images[f] = (crop, mask)

cap.release()

motion_df = pd.DataFrame(
    motion_records,
    columns=[
        "frame", "center_x", "center_y",
        "width", "height", "area"
    ]
)

print("Total motion candidates:", len(motion_df))
print("Frames processed:", END_FRAME - START_FRAME + 1)

display(motion_df.groupby("frame").size().describe())


In [ ]:

benchmark = gt[
    gt.frame.between(START_FRAME, END_FRAME)
    & (gt.annotation_status == "annotated")
    & (gt.gt_ball_visible == 1)
].copy()

results = []

for row in benchmark.itertuples(index=False):
    pool = motion_df[motion_df.frame == row.frame]

    if pool.empty:
        distance = np.nan
    else:
        distance = np.hypot(
            pool.center_x.to_numpy() - row.gt_ball_x,
            pool.center_y.to_numpy() - row.gt_ball_y
        ).min()

    results.append({
        "frame": int(row.frame),
        "nearest_motion_distance_px": distance,
        "candidate_count": len(pool),
        "candidate_within_10px": (
            np.isfinite(distance) and distance <= 10
        )
    })

motion_audit = pd.DataFrame(results)

print("AIRBORNE MOTION DETECTION AUDIT")
display(motion_audit.round(2))

print(
    "Frames with a motion candidate within 10px:",
    int(motion_audit.candidate_within_10px.sum()),
    "/", len(motion_audit)
)


In [ ]:

fig, axes = plt.subplots(3, 2, figsize=(16, 9))

for row_idx, f in enumerate([4036, 4048, 4060]):
    crop, mask = debug_images[f]

    axes[row_idx, 0].imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    )
    axes[row_idx, 0].set_title(f"Frame {f} — Original")

    axes[row_idx, 1].imshow(mask, cmap="gray")
    axes[row_idx, 1].set_title(f"Frame {f} — Motion mask")

    for ax in axes[row_idx]:
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import pandas as pd

AIRBORNE_FRAMES = [4036, 4048, 4060]

results = []

for frame_idx in AIRBORNE_FRAMES:
    gt_row = gt.loc[gt.frame == frame_idx].iloc[0]

    gx = float(gt_row.gt_ball_x)
    gy = float(gt_row.gt_ball_y)

    pool = motion_df[motion_df.frame == frame_idx].copy()

    if pool.empty:
        results.append({
            "frame": frame_idx,
            "motion_candidates": 0,
            "nearest_distance_px": np.nan,
            "nearest_x": np.nan,
            "nearest_y": np.nan,
            "within_10px": False
        })
        continue

    pool["distance_px"] = np.hypot(
        pool.center_x - gx,
        pool.center_y - gy
    )

    closest = pool.loc[pool.distance_px.idxmin()]

    results.append({
        "frame": frame_idx,
        "motion_candidates": len(pool),
        "nearest_distance_px": float(closest.distance_px),
        "nearest_x": float(closest.center_x),
        "nearest_y": float(closest.center_y),
        "within_10px": bool(closest.distance_px <= 10)
    })

motion_check = pd.DataFrame(results)

print("AIRBORNE MOTION CANDIDATE EVALUATION")
display(motion_check.round(2))

print(
    "Frames with motion candidates within 10px:",
    int(motion_check.within_10px.sum()),
    "/",
    len(motion_check)
)


In [ ]:

import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Video missing"

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, frame_idx in zip(axes, AIRBORNE_FRAMES):
    row = gt.loc[gt.frame == frame_idx].iloc[0]

    gx = int(row.gt_ball_x)
    gy = int(row.gt_ball_y)

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        ax.axis("off")
        continue

    h, w = frame.shape[:2]
    radius = 90

    x1 = max(0, gx-radius)
    x2 = min(w, gx+radius)
    y1 = max(0, gy-radius)
    y2 = min(h, gy+radius)

    crop = frame[y1:y2, x1:x2].copy()

    pool = motion_df[
        motion_df.frame == frame_idx
    ].copy()

    for det in pool.itertuples(index=False):
        px = int(round(det.center_x - x1))
        py = int(round(det.center_y - y1))

        if 0 <= px < crop.shape[1] and 0 <= py < crop.shape[0]:
            cv2.circle(crop, (px, py), 8, (0, 0, 255), 1)

    cv2.drawMarker(
        crop,
        (gx-x1, gy-y1),
        (0, 255, 0),
        cv2.MARKER_CROSS,
        20, 2
    )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {frame_idx}")
    ax.axis("off")

cap.release()

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd

frames = [4036, 4048, 4060]
results = []

for frame_idx in frames:
    label = gt.loc[gt.frame == frame_idx].iloc[0]
    pool = motion_df[motion_df.frame == frame_idx].copy()

    if not pool.empty:
        pool["distance"] = np.hypot(
            pool.center_x - label.gt_ball_x,
            pool.center_y - label.gt_ball_y
        )
        closest = pool.loc[pool.distance.idxmin()]
        nearest = float(closest.distance)
    else:
        nearest = np.nan

    results.append({
        "frame": frame_idx,
        "candidate_count": len(pool),
        "nearest_distance_px": nearest,
        "within_5px": bool(nearest <= 5),
        "within_10px": bool(nearest <= 10)
    })

motion_results = pd.DataFrame(results)

display(motion_results.round(2))

print(
    "Airborne motion recall @10px:",
    f"{motion_results.within_10px.sum()}/{len(motion_results)}"
)

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Football_CV")
GT_PATH = ROOT / "ball_benchmark_v1/ball_ground_truth_v1.csv"

gt = pd.read_csv(GT_PATH)

assert "motion_df" in globals(), (
    "motion_df is missing. Rerun the airborne motion "
    "candidate extraction cell first."
)

ANCHOR_FRAMES = [4036, 4048, 4060]

anchors = gt[
    gt.frame.isin(ANCHOR_FRAMES)
].sort_values("frame").copy()

assert len(anchors) == 3
assert anchors.gt_ball_x.notna().all()
assert anchors.gt_ball_y.notna().all()

# Linear interpolation between known anchors.
# Used only to guide this diagnostic experiment.
frames = np.arange(4036, 4061)

expected_x = np.interp(
    frames,
    anchors.frame,
    anchors.gt_ball_x
)

expected_y = np.interp(
    frames,
    anchors.frame,
    anchors.gt_ball_y
)

SEARCH_GATE = 22.0

records = []

for frame_idx, px, py in zip(
    frames, expected_x, expected_y
):
    pool = motion_df[
        motion_df.frame == frame_idx
    ].copy()

    pool["distance_to_corridor"] = np.hypot(
        pool.center_x - px,
        pool.center_y - py
    )

    eligible = pool[
        pool.distance_to_corridor <= SEARCH_GATE
    ].sort_values("distance_to_corridor")

    if not eligible.empty:
        best = eligible.iloc[0]

        records.append({
            "frame": frame_idx,
            "x": float(best.center_x),
            "y": float(best.center_y),
            "expected_x": px,
            "expected_y": py,
            "distance_to_corridor": float(
                best.distance_to_corridor
            ),
            "candidate_count": len(pool),
            "eligible_count": len(eligible),
            "status": "motion_candidate"
        })

    else:
        records.append({
            "frame": frame_idx,
            "x": np.nan,
            "y": np.nan,
            "expected_x": px,
            "expected_y": py,
            "distance_to_corridor": np.nan,
            "candidate_count": len(pool),
            "eligible_count": 0,
            "status": "unresolved"
        })

motion_v8 = pd.DataFrame(records)

print("V8 MOTION ASSOCIATION")
print(motion_v8.status.value_counts())

print("\nFrame-by-frame results:")
display(motion_v8.round(2))


In [ ]:

fig, ax = plt.subplots(figsize=(13, 6))

# All raw motion candidates in the interval
pool = motion_df[
    motion_df.frame.between(4036, 4060)
]

ax.scatter(
    pool.center_x,
    pool.center_y,
    color="lightgray",
    s=7,
    alpha=0.25,
    label="All motion candidates"
)

# Ground-truth anchor positions
ax.scatter(
    anchors.gt_ball_x,
    anchors.gt_ball_y,
    color="blue",
    marker="*",
    s=180,
    label="Manual GT anchors"
)

# Expected corridor
ax.plot(
    motion_v8.expected_x,
    motion_v8.expected_y,
    "--",
    color="orange",
    label="GT-guided search corridor"
)

# Selected motion candidates
selected = motion_v8[
    motion_v8.status == "motion_candidate"
]

ax.plot(
    selected.x,
    selected.y,
    "o-",
    color="red",
    markersize=5,
    label="Selected motion candidates"
)

ax.set_title(
    "V8 Diagnostic - Airborne Motion Candidate Association"
)

ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.invert_yaxis()
ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:

OUT_DIR = (
    ROOT / "ball_benchmark_v1"
    / "evaluation_v3"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = (
    OUT_DIR / "airborne_motion_association_v8.csv"
)

motion_v8.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Selected frames:",
      int((motion_v8.status == "motion_candidate").sum()))
print("Unresolved frames:",
      int((motion_v8.status == "unresolved").sum()))


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "motion_df" in globals(), (
    "motion_df is missing. Rerun the background-subtraction "
    "candidate extraction cell first."
)

START, END = 4028, 4065
MAX_STEP = 38.0
MAX_GAP = 2

# Broad airborne search region, not ground-truth coordinates.
candidates = motion_df[
    motion_df.frame.between(START, END)
    & motion_df.center_y.between(80, 230)
    & motion_df.center_x.between(1900, 3200)
].copy()

candidates = candidates.sort_values("frame")

tracks = []
next_id = 0

for frame_idx in range(START, END + 1):

    pool = candidates[candidates.frame == frame_idx]
    detections = pool.to_dict("records")

    used = set()

    # Extend established tracks first.
    active = [
        t for t in tracks
        if frame_idx - t["last_frame"] <= MAX_GAP
    ]

    active.sort(key=lambda t: -len(t["points"]))

    for track in active:

        history = track["points"]
        last = history[-1]

        px, py = last["x"], last["y"]

        if len(history) >= 2:
            previous = history[-2]

            dt = max(1, last["frame"] - previous["frame"])

            vx = (last["x"] - previous["x"]) / dt
            vy = (last["y"] - previous["y"]) / dt

            gap = frame_idx - last["frame"]

            px += vx * gap
            py += vy * gap

        best_idx = None
        best_distance = float("inf")

        for i, det in enumerate(detections):
            if i in used:
                continue

            distance = float(np.hypot(
                det["center_x"] - px,
                det["center_y"] - py
            ))

            if distance < best_distance and distance <= MAX_STEP:
                best_idx = i
                best_distance = distance

        if best_idx is not None:
            det = detections[best_idx]

            track["points"].append({
                "frame": frame_idx,
                "x": float(det["center_x"]),
                "y": float(det["center_y"])
            })

            track["last_frame"] = frame_idx
            used.add(best_idx)

    # Unmatched candidates can initiate independent tracks.
    for i, det in enumerate(detections):

        if i in used:
            continue

        tracks.append({
            "track_id": next_id,
            "last_frame": frame_idx,
            "points": [{
                "frame": frame_idx,
                "x": float(det["center_x"]),
                "y": float(det["center_y"])
            }]
        })

        next_id += 1

# Rank candidate trajectories.
summaries = []

for track in tracks:

    pts = track["points"]

    if len(pts) < 8:
        continue

    x = np.array([p["x"] for p in pts])
    y = np.array([p["y"] for p in pts])
    f = np.array([p["frame"] for p in pts])

    displacement = float(np.hypot(
        x[-1] - x[0],
        y[-1] - y[0]
    ))

    path_length = float(
        np.hypot(np.diff(x), np.diff(y)).sum()
    )

    duration = max(1, f[-1] - f[0])
    average_speed = displacement / duration

    if displacement < 80 or average_speed < 4:
        continue

    smoothness = displacement / max(path_length, 1e-6)

    score = (
        len(pts)
        * smoothness
        * np.log1p(displacement)
    )

    summaries.append({
        "track_id": track["track_id"],
        "start_frame": int(f[0]),
        "end_frame": int(f[-1]),
        "observations": len(pts),
        "displacement_px": displacement,
        "average_speed_px_frame": average_speed,
        "smoothness": smoothness,
        "score": score
    })

tracklet_summary = pd.DataFrame(summaries)

if tracklet_summary.empty:
    print("No tracklets passed the initial filters.")
    print("Try reducing the minimum observations or displacement.")
else:
    tracklet_summary = tracklet_summary.sort_values(
        "score", ascending=False
    ).reset_index(drop=True)

    print("Automatic tracklet candidates:")
    display(tracklet_summary.head(15).round(2))


In [ ]:

fig, ax = plt.subplots(figsize=(14, 7))

ax.scatter(
    candidates.center_x,
    candidates.center_y,
    s=6,
    alpha=0.15,
    color="gray",
    label="Motion candidates"
)

if not tracklet_summary.empty:

    top_ids = tracklet_summary.head(5).track_id.tolist()

    for track_id in top_ids:

        track = next(
            t for t in tracks
            if t["track_id"] == track_id
        )

        points = pd.DataFrame(track["points"])

        ax.plot(
            points.x,
            points.y,
            "-o",
            linewidth=2,
            markersize=3,
            label=f"Track {track_id}"
        )

ax.set_title(
    "V9 - Automatic Airborne Motion Tracklets (No GT Guidance)"
)

ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.invert_yaxis()
ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:

from pathlib import Path

OUT = Path(
    "/content/drive/MyDrive/Football_CV/"
    "ball_benchmark_v1/evaluation_v4"
)
OUT.mkdir(parents=True, exist_ok=True)

tracklet_summary.to_csv(
    OUT / "automatic_motion_tracklets_v9.csv",
    index=False
)

point_rows = []

for track in tracks:
    for p in track["points"]:
        point_rows.append({
            "track_id": track["track_id"],
            **p
        })

pd.DataFrame(point_rows).to_csv(
    OUT / "automatic_motion_tracklet_points_v9.csv",
    index=False
)

print("V9 results saved:", OUT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TRACK_ID = 21

selected_track = next(
    t for t in tracks
    if t["track_id"] == TRACK_ID
)

track_points = pd.DataFrame(selected_track["points"])
track_points = track_points.sort_values("frame")

gt_airborne = gt[
    gt.frame.isin([4036, 4048, 4060])
].copy()

comparison = gt_airborne[
    ["frame", "gt_ball_x", "gt_ball_y"]
].merge(
    track_points[["frame", "x", "y"]],
    on="frame",
    how="left"
)

comparison["error_px"] = np.hypot(
    comparison["x"] - comparison["gt_ball_x"],
    comparison["y"] - comparison["gt_ball_y"]
)

print("TRACK 21 SUMMARY")
print("Start:", track_points.frame.min())
print("End:", track_points.frame.max())
print("Observations:", len(track_points))

print("\nComparison against airborne GT:")
display(comparison.round(2))

print("\nMatched GT frames:",
      comparison.error_px.notna().sum(), "/ 3")

print("Mean error on matched anchors:",
      comparison.error_px.mean())

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    track_points.x,
    track_points.y,
    "-o",
    color="blue",
    markersize=4,
    label="Automatic Track 21"
)

ax.scatter(
    gt_airborne.gt_ball_x,
    gt_airborne.gt_ball_y,
    marker="*",
    s=220,
    color="red",
    label="Manual ground truth"
)

for row in gt_airborne.itertuples(index=False):
    ax.annotate(
        str(row.frame),
        (row.gt_ball_x, row.gt_ball_y),
        xytext=(6, -12),
        textcoords="offset points"
    )

ax.invert_yaxis()
ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.set_title("V9 Track 21 vs Ground Truth")
ax.legend()
ax.grid(alpha=0.25)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Football_CV")
GT_PATH = ROOT / "ball_benchmark_v1/ball_ground_truth_v1.csv"
TRACK_PATH = (
    ROOT / "ball_benchmark_v1/evaluation_v4/"
    "automatic_motion_tracklet_points_v9.csv"
)

gt_check = pd.read_csv(GT_PATH)
track_points_all = pd.read_csv(TRACK_PATH)

TRACK_ID = 21
track21 = (
    track_points_all[track_points_all.track_id == TRACK_ID]
    .sort_values("frame")
    .drop_duplicates("frame")
    .copy()
)

anchors = gt_check[
    gt_check.frame.isin([4036, 4048, 4060])
][["frame", "gt_ball_x", "gt_ball_y"]].copy()

comparison = anchors.merge(
    track21[["frame", "x", "y"]],
    on="frame",
    how="left"
)

comparison["error_px"] = np.hypot(
    comparison.x - comparison.gt_ball_x,
    comparison.y - comparison.gt_ball_y
)

print("TRACK 21")
print("First frame:", int(track21.frame.min()))
print("Last frame:", int(track21.frame.max()))
print("Observations:", len(track21))

print("\nEXACT FRAME ALIGNMENT")
display(comparison.round(3))

print("\nNEAREST SPATIAL POINT (FRAME MAY DIFFER)")
spatial_rows = []

for row in anchors.itertuples(index=False):
    distances = np.hypot(
        track21.x.to_numpy() - row.gt_ball_x,
        track21.y.to_numpy() - row.gt_ball_y
    )
    i = int(np.argmin(distances))
    nearest = track21.iloc[i]

    spatial_rows.append({
        "gt_frame": int(row.frame),
        "track_frame": int(nearest.frame),
        "frame_difference": int(nearest.frame - row.frame),
        "spatial_error_px": float(distances[i])
    })

display(pd.DataFrame(spatial_rows).round(3))

print("\nFIRST 12 TRACK POINTS")
display(track21.head(12).round(2))

print("\nLAST 12 TRACK POINTS")
display(track21.tail(12).round(2))

fig, ax = plt.subplots(figsize=(13, 6))

sc = ax.scatter(
    track21.x,
    track21.y,
    c=track21.frame,
    cmap="viridis",
    s=45
)

ax.plot(track21.x, track21.y, alpha=0.5)

ax.scatter(
    anchors.gt_ball_x,
    anchors.gt_ball_y,
    marker="*",
    s=200,
    color="red",
    label="Ground truth"
)

for row in anchors.itertuples(index=False):
    ax.annotate(
        f"GT {row.frame}",
        (row.gt_ball_x, row.gt_ball_y),
        xytext=(8, 7),
        textcoords="offset points"
    )

ax.invert_yaxis()
ax.set_xlabel("Image X")
ax.set_ylabel("Image Y")
ax.set_title("V9 Track 21 - Temporal Alignment")
ax.legend()
ax.grid(alpha=0.2)

plt.colorbar(sc, ax=ax, label="Video frame")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Football_CV")
GT_PATH = ROOT / "ball_benchmark_v1/ball_ground_truth_v1.csv"
TRACK_PATH = (
    ROOT / "ball_benchmark_v1/evaluation_v4/"
    "automatic_motion_tracklet_points_v9.csv"
)

gt = pd.read_csv(GT_PATH)
tracklets = pd.read_csv(TRACK_PATH)

TRACK_ID = 21

track21 = (
    tracklets[tracklets.track_id == TRACK_ID]
    [["frame", "x", "y"]]
    .sort_values("frame")
    .drop_duplicates("frame")
    .reset_index(drop=True)
)

gt_flight = gt[
    gt.frame.between(4028, 4065)
    & (gt.annotation_status == "annotated")
    & (gt.gt_ball_visible == 1)
].copy()

comparison = gt_flight[
    ["frame", "gt_ball_x", "gt_ball_y"]
].merge(track21, on="frame", how="left")

comparison["error_px"] = np.hypot(
    comparison.x - comparison.gt_ball_x,
    comparison.y - comparison.gt_ball_y
)

print("TRACK 21")
print("Start frame:", int(track21.frame.min()))
print("End frame:", int(track21.frame.max()))
print("Total observations:", len(track21))

print("\nEXACT TEMPORAL EVALUATION")
display(comparison.round(3))

matched = comparison.dropna(subset=["error_px"])

print("\nMatched GT frames:", len(matched), "/", len(gt_flight))

if len(matched):
    print("Mean error:", round(matched.error_px.mean(), 3), "px")
    print("Maximum error:", round(matched.error_px.max(), 3), "px")
    print(
        "Recall @10px:",
        round(100 * (comparison.error_px <= 10).sum()
              / len(gt_flight), 2),
        "%"
    )